# Customer Request & Ground Truth Generation

## Objective

Generate synthetic customer requests in natural language and
their corresponding structured reference requirements.

The dataset will be used to train and evaluate an NLP model
that extracts:

- Characteristic
- Value
- Operator (EQUAL, MIN, MAX)
- Requirement type (MUST, PREFERRED, INDIFFERENT)

## Data Sources

- CHARACTERISTIC_DICTIONARY
- CHARACTERISTIC_VALUES
- CONFIGURATION_RULES
- PRODUCT_MASTER

## Approach

1. Generate structured customer requirements.
2. Convert requirements into natural-language requests.
3. Validate consistency between requests and ground truth.
4. Create training, validation, and test datasets.

In [1]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
MASTER_PATH = PROJECT_ROOT / "data" / "raw" / "oven_configurator_master_v1.0.xlsx"

print("Master file exists:", MASTER_PATH.exists())

# Load master data
dictionary = pd.read_excel(MASTER_PATH, sheet_name="CHARACTERISTIC_DICTIONARY")
values = pd.read_excel(MASTER_PATH, sheet_name="CHARACTERISTIC_VALUES")
rules = pd.read_excel(MASTER_PATH, sheet_name="CONFIGURATION_RULES")

# Inspect dataset structure
for name, df in [
    ("CHARACTERISTIC_DICTIONARY", dictionary),
    ("CHARACTERISTIC_VALUES", values),
    ("CONFIGURATION_RULES", rules),
]:
    print(f"\n{name}: {df.shape}")
    print("Columns:", df.columns.tolist())
    display(df.head(3))

Master file exists: True

CHARACTERISTIC_DICTIONARY: (25, 6)
Columns: ['-', 'NUMERIC / CATEGORICAL / BOOLEAN / ORDINAL', '-.1', 'EXACT / DISTANCE / ORDINAL', 'cm, L, adet vb.', '-.2']


,-,NUMERIC / CATEGORICAL / BOOLEAN / ORDINAL,-.1,EXACT / DISTANCE / ORDINAL,"cm, L, adet vb.",-.2
0,CHARACTERISTIC,DATA_TYPE,ALLOWED_VALUES / RANGE,MATCH_METHOD,UNIT,DESCRIPTION
1,WIDTH_CM,NUMERIC,"45, 60, 70, 90",EXACT,cm,Product installation width
2,PACKAGING_TYPE,CATEGORICAL,"SHRINK, BOX",EXACT,-,Design of product packaging



CHARACTERISTIC_VALUES: (90, 3)
Columns: ['CHARACTERISTIC', 'VALUE', 'DEFINITION']


,CHARACTERISTIC,VALUE,DEFINITION
0,WIDTH_CM,V001,45
1,WIDTH_CM,V002,60
2,WIDTH_CM,V003,70



CONFIGURATION_RULES: (170, 6)
Columns: ['RuleId', 'Type', 'Operator', 'CHARACTERISTIC', 'VALUE', 'DEFINITION']


,RuleId,Type,Operator,CHARACTERISTIC,VALUE,DEFINITION
0,1,Auto Select,Action,TURNSPIT,V001,YES
1,1,AND,Rule,OVEN_FUNCTIONS,V004,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
2,1,AND,Rule,OVEN_FUNCTIONS,V007,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...


In [2]:
# Read characteristic dictionary with the correct header row
dictionary = pd.read_excel(
    MASTER_PATH,
    sheet_name="CHARACTERISTIC_DICTIONARY",
    header=1
)

print("Dictionary shape:", dictionary.shape)
print("Columns:", dictionary.columns.tolist())

display(dictionary)

Dictionary shape: (24, 6)
Columns: ['CHARACTERISTIC', 'DATA_TYPE', 'ALLOWED_VALUES / RANGE', 'MATCH_METHOD', 'UNIT', 'DESCRIPTION']


,CHARACTERISTIC,DATA_TYPE,ALLOWED_VALUES / RANGE,MATCH_METHOD,UNIT,DESCRIPTION
0,WIDTH_CM,NUMERIC,"45, 60, 70, 90",EXACT,cm,Product installation width
1,PACKAGING_TYPE,CATEGORICAL,"SHRINK, BOX",EXACT,-,Design of product packaging
2,PANEL_TYPE,CATEGORICAL,"GLASS, METAL",EXACT,-,Control panel material
3,PANEL_COLOR,CATEGORICAL,"BLACK, WHITE, SILVER, STAINLESS STEEL, DARK SI...",EXACT,-,Control panel color
4,POWER_CABLE,CATEGORICAL,"W/O CABLE, W/O PLUG, EU TYPE PLUG, US TYPE PLU...",EXACT,-,Power cable availability of product based on p...
5,WIFI,BOOLEAN,"YES, NO",EXACT,-,Wireless connectivity
6,TURNSPIT,BOOLEAN,"YES, NO",EXACT,-,A motorized accessory used for slow-roasting l...
7,OVEN_TYPE,CATEGORICAL,"CONVECTION, STEAM, COMBINATION",EXACT,-,Categorization of built in oven by how it gene...
8,NUMBER_OF_FUNCTIONS,NUMERIC,"3, 5, 8, 10, 11",DISTANCE,quantity,Number of cooking modes
9,OVEN_FUNCTIONS,CATEGORICAL,01-12 VALUES ARE AVAILABLE,EXACT,-,Preset cooking function set ID


In [3]:
display(
    dictionary[
        ["CHARACTERISTIC", "DATA_TYPE", "MATCH_METHOD"]
    ]
)

,CHARACTERISTIC,DATA_TYPE,MATCH_METHOD
0,WIDTH_CM,NUMERIC,EXACT
1,PACKAGING_TYPE,CATEGORICAL,EXACT
2,PANEL_TYPE,CATEGORICAL,EXACT
3,PANEL_COLOR,CATEGORICAL,EXACT
4,POWER_CABLE,CATEGORICAL,EXACT
5,WIFI,BOOLEAN,EXACT
6,TURNSPIT,BOOLEAN,EXACT
7,OVEN_TYPE,CATEGORICAL,EXACT
8,NUMBER_OF_FUNCTIONS,NUMERIC,DISTANCE
9,OVEN_FUNCTIONS,CATEGORICAL,EXACT


In [4]:
# Step 5 - First customer request and reference requirements

request_id = "R0001"

customer_request = pd.DataFrame([
    {
        "REQUEST_ID": request_id,
        "REQUEST_TEXT": (
            "60 cm genişliğinde, siyah panelli bir fırın istiyorum. "
            "Wi-Fi özelliği olursa iyi olur."
        )
    }
])

requirements = [
    ("WIDTH_CM", "60", "EQUAL", "MUST"),
    ("PANEL_COLOR", "BLACK", "EQUAL", "MUST"),
    ("WIFI", "YES", "EQUAL", "PREFERRED")
]

ground_truth = pd.DataFrame(
    [
        {
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": value,
            "OPERATOR": operator,
            "REQUIREMENT_TYPE": requirement_type
        }
        for characteristic, value, operator, requirement_type
        in requirements
    ]
)

display(customer_request)
display(ground_truth)

,REQUEST_ID,REQUEST_TEXT
0,R0001,"60 cm genişliğinde, siyah panelli bir fırın is..."


,REQUEST_ID,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,R0001,WIDTH_CM,60,EQUAL,MUST
1,R0001,PANEL_COLOR,BLACK,EQUAL,MUST
2,R0001,WIFI,YES,EQUAL,PREFERRED


In [5]:
import random

# Reproducible random selection
rng = random.Random(42)

# Characteristics for our first pilot
pilot_characteristics = [
    "WIDTH_CM",
    "PANEL_COLOR",
    "CLEANING",
    "WIFI"
]

# Requirement type probabilities
requirement_types = ["MUST", "PREFERRED"]
requirement_weights = [0.75, 0.25]

generated_requirements = []

for characteristic in pilot_characteristics:

    # Find allowed values for this characteristic
    available_values = values[
        values["CHARACTERISTIC"] == characteristic
    ]

    # Randomly select one valid value
    selected = available_values.sample(
        n=1,
        random_state=rng.randint(0, 100000)
    ).iloc[0]

    requirement_type = rng.choices(
        requirement_types,
        weights=requirement_weights,
        k=1
    )[0]

    generated_requirements.append({
        "REQUEST_ID": "R0002",
        "CHARACTERISTIC": characteristic,
        "VALUE": selected["VALUE"],
        "VALUE_DEFINITION": selected["DEFINITION"],
        "OPERATOR": "EQUAL",
        "REQUIREMENT_TYPE": requirement_type
    })

generated_ground_truth = pd.DataFrame(generated_requirements)

display(generated_ground_truth)

,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0002,WIDTH_CM,V001,45,EQUAL,MUST
1,R0002,PANEL_COLOR,V001,BLACK,EQUAL,MUST
2,R0002,CLEANING,V002,CATALYTIC,EQUAL,MUST
3,R0002,WIFI,V001,YES,EQUAL,MUST


In [6]:
# Step 7 - Convert structured requirements into English customer language

def requirement_to_text(row):
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"])
    requirement_type = row["REQUIREMENT_TYPE"]

    if characteristic == "WIDTH_CM":
        phrase = f"a {value} cm wide oven"

    elif characteristic == "PANEL_COLOR":
        phrase = f"a {value.lower()} control panel"

    elif characteristic == "CLEANING":
        phrase = f"{value.lower()} cleaning"

    elif characteristic == "WIFI":
        phrase = (
            "Wi-Fi connectivity"
            if value.upper() == "YES"
            else "no Wi-Fi connectivity"
        )

    else:
        raise ValueError(f"Unsupported characteristic: {characteristic}")

    if requirement_type == "MUST":
        return f"I need {phrase}."

    elif requirement_type == "PREFERRED":
        return f"I would prefer {phrase}."

    else:
        raise ValueError(f"Unsupported requirement type: {requirement_type}")


sentences = [
    requirement_to_text(row)
    for _, row in generated_ground_truth.iterrows()
]

request_text = " ".join(sentences)

generated_customer_request = pd.DataFrame([
    {
        "REQUEST_ID": "R0002",
        "REQUEST_TEXT": request_text
    }
])

print(request_text)
display(generated_customer_request)

I need a 45 cm wide oven. I need a black control panel. I need catalytic cleaning. I need Wi-Fi connectivity.


,REQUEST_ID,REQUEST_TEXT
0,R0002,I need a 45 cm wide oven. I need a black contr...


In [7]:
# Step 8 - Test mixed requirement types

mixed_ground_truth = generated_ground_truth.copy()

mixed_ground_truth.loc[
    mixed_ground_truth["CHARACTERISTIC"] == "WIFI",
    "REQUIREMENT_TYPE"
] = "PREFERRED"

mixed_sentences = [
    requirement_to_text(row)
    for _, row in mixed_ground_truth.iterrows()
]

mixed_request_text = " ".join(mixed_sentences)

print(mixed_request_text)

display(
    mixed_ground_truth[
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ]
)

I need a 45 cm wide oven. I need a black control panel. I need catalytic cleaning. I would prefer Wi-Fi connectivity.


,CHARACTERISTIC,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,WIDTH_CM,45,EQUAL,MUST
1,PANEL_COLOR,BLACK,EQUAL,MUST
2,CLEANING,CATALYTIC,EQUAL,MUST
3,WIFI,YES,EQUAL,PREFERRED


In [8]:
# Step 9 - Compose a more natural English customer request

must_phrases = []
preferred_phrases = []

for _, row in mixed_ground_truth.iterrows():
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"])

    if characteristic == "WIDTH_CM":
        phrase = f"{value} cm wide"
    elif characteristic == "PANEL_COLOR":
        phrase = f"a {value.lower()} control panel"
    elif characteristic == "CLEANING":
        phrase = f"{value.lower()} cleaning"
    elif characteristic == "WIFI":
        phrase = (
            "Wi-Fi connectivity"
            if value.upper() == "YES"
            else "no Wi-Fi connectivity"
        )
    else:
        raise ValueError(f"Unsupported characteristic: {characteristic}")

    if row["REQUIREMENT_TYPE"] == "MUST":
        must_phrases.append(phrase)
    elif row["REQUIREMENT_TYPE"] == "PREFERRED":
        preferred_phrases.append(phrase)

request_parts = []

if must_phrases:
    request_parts.append(
        "I'm looking for an oven that must be "
        + ", ".join(must_phrases)
        + "."
    )

if preferred_phrases:
    request_parts.append(
        "Ideally, it should also have "
        + ", ".join(preferred_phrases)
        + "."
    )

natural_request_text = " ".join(request_parts)

print(natural_request_text)

I'm looking for an oven that must be 45 cm wide, a black control panel, catalytic cleaning. Ideally, it should also have Wi-Fi connectivity.


In [9]:
# Step 10 - Generate grammatically correct requirement sentences

def compose_request(ground_truth):
    must_sentences = []
    preferred_sentences = []

    for _, row in ground_truth.iterrows():
        characteristic = row["CHARACTERISTIC"]
        value = str(row["VALUE_DEFINITION"])
        requirement_type = row["REQUIREMENT_TYPE"]

        if characteristic == "WIDTH_CM":
            phrase = f"be {value} cm wide"
        elif characteristic == "PANEL_COLOR":
            phrase = f"have a {value.lower()} control panel"
        elif characteristic == "CLEANING":
            phrase = f"have {value.lower()} cleaning"
        elif characteristic == "WIFI":
            phrase = (
                "have Wi-Fi connectivity"
                if value.upper() == "YES"
                else "have no Wi-Fi connectivity"
            )
        else:
            raise ValueError(f"Unsupported characteristic: {characteristic}")

        if requirement_type == "MUST":
            must_sentences.append(f"It must {phrase}.")
        elif requirement_type == "PREFERRED":
            preferred_sentences.append(f"I would prefer it to {phrase}.")
        else:
            raise ValueError(f"Unsupported requirement type: {requirement_type}")

    return "I'm looking for an oven. " + " ".join(
        must_sentences + preferred_sentences
    )


natural_request_v2 = compose_request(mixed_ground_truth)

print(natural_request_v2)

I'm looking for an oven. It must be 45 cm wide. It must have a black control panel. It must have catalytic cleaning. I would prefer it to have Wi-Fi connectivity.


In [10]:
# Step 11 - Prepare characteristic generation profile

characteristic_profile = dictionary[
    ["CHARACTERISTIC", "DATA_TYPE", "MATCH_METHOD"]
].copy()

characteristic_profile["REQUEST_WEIGHT"] = 1.0
characteristic_profile["ENABLED"] = True

display(characteristic_profile)


,CHARACTERISTIC,DATA_TYPE,MATCH_METHOD,REQUEST_WEIGHT,ENABLED
0,WIDTH_CM,NUMERIC,EXACT,1.0,True
1,PACKAGING_TYPE,CATEGORICAL,EXACT,1.0,True
2,PANEL_TYPE,CATEGORICAL,EXACT,1.0,True
3,PANEL_COLOR,CATEGORICAL,EXACT,1.0,True
4,POWER_CABLE,CATEGORICAL,EXACT,1.0,True
5,WIFI,BOOLEAN,EXACT,1.0,True
6,TURNSPIT,BOOLEAN,EXACT,1.0,True
7,OVEN_TYPE,CATEGORICAL,EXACT,1.0,True
8,NUMBER_OF_FUNCTIONS,NUMERIC,DISTANCE,1.0,True
9,OVEN_FUNCTIONS,CATEGORICAL,EXACT,1.0,True


In [11]:
# Step 12 - Define customer request generation weights

high_priority = [
    "WIDTH_CM",
    "PANEL_COLOR",
    "CLEANING",
    "OVEN_FUNCTIONS",
    "NUMBER_OF_FUNCTIONS",
    "OVEN_CONTROL",
    "OVEN_TYPE"
]

low_priority = [
    "PACKAGING_TYPE",
    "POWER_CABLE"
]

characteristic_profile["REQUEST_WEIGHT"] = 1.0

characteristic_profile.loc[
    characteristic_profile["CHARACTERISTIC"].isin(high_priority),
    "REQUEST_WEIGHT"
] = 3.0

characteristic_profile.loc[
    characteristic_profile["CHARACTERISTIC"].isin(low_priority),
    "REQUEST_WEIGHT"
] = 0.3

display(
    characteristic_profile[
        ["CHARACTERISTIC", "REQUEST_WEIGHT", "ENABLED"]
    ]
)

,CHARACTERISTIC,REQUEST_WEIGHT,ENABLED
0,WIDTH_CM,3.0,True
1,PACKAGING_TYPE,0.3,True
2,PANEL_TYPE,1.0,True
3,PANEL_COLOR,3.0,True
4,POWER_CABLE,0.3,True
5,WIFI,1.0,True
6,TURNSPIT,1.0,True
7,OVEN_TYPE,3.0,True
8,NUMBER_OF_FUNCTIONS,3.0,True
9,OVEN_FUNCTIONS,3.0,True


In [12]:
# Step 13 - Select characteristics for five customer requests

import random

rng = random.Random(42)

active_profile = characteristic_profile[
    characteristic_profile["ENABLED"]
].copy()

characteristics = active_profile["CHARACTERISTIC"].tolist()
weights = active_profile["REQUEST_WEIGHT"].tolist()


def select_characteristics(rng, min_count=2, max_count=6):
    """Select unique characteristics using weighted sampling."""

    count = rng.randint(min_count, max_count)

    available = list(zip(characteristics, weights))
    selected = []

    for _ in range(min(count, len(available))):
        chosen = rng.choices(
            available,
            weights=[item[1] for item in available],
            k=1
        )[0]

        selected.append(chosen[0])
        available.remove(chosen)

    return selected


request_profiles = []

for i in range(1, 6):
    request_id = f"R{i:04d}"

    selected = select_characteristics(rng)

    request_profiles.append({
        "REQUEST_ID": request_id,
        "N_CHARACTERISTICS": len(selected),
        "SELECTED_CHARACTERISTICS": selected
    })

request_profiles_df = pd.DataFrame(request_profiles)

display(request_profiles_df)

,REQUEST_ID,N_CHARACTERISTICS,SELECTED_CHARACTERISTICS
0,R0001,2,"[WIDTH_CM, OVEN_TYPE]"
1,R0002,3,"[PANEL_COLOR, PANEL_TYPE, ENERGY_CLASS]"
2,R0003,6,"[PACKAGING_TYPE, OVEN_FUNCTIONS, WIDTH_CM, OVE..."
3,R0004,3,"[HANDLE_TYPE, KNOB_COLOR, NUMBER_OF_FUNCTIONS]"
4,R0005,5,"[ROASTING_TRAY, TELESCOPIC_RAILS, WIDTH_CM, OV..."


In [13]:
# Step 14 - Generate ground truth from valid product configurations (Product-Anchored Generation)

product_master = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "product_master.csv",
    dtype=str
)

value_lookup = values.copy()

# Normalize lookup keys
value_lookup["CHARACTERISTIC"] = (
    value_lookup["CHARACTERISTIC"].astype(str).str.strip()
)

value_lookup["VALUE"] = (
    value_lookup["VALUE"].astype(str).str.strip()
)

rng = random.Random(123)
ground_truth_rows = []

for _, request in request_profiles_df.iterrows():

    request_id = request["REQUEST_ID"]

    # Select a valid reference product
    product = product_master.sample(
        n=1,
        random_state=rng.randint(0, 100000)
    ).iloc[0]

    for characteristic in request["SELECTED_CHARACTERISTICS"]:

        # Product master contains VALUE codes (e.g. V001)
        product_value_code = str(product[characteristic]).strip()

        # Match by characteristic AND value code
        matching_values = value_lookup[
            (value_lookup["CHARACTERISTIC"] == characteristic)
            & (value_lookup["VALUE"] == product_value_code)
        ]

        if len(matching_values) != 1:
            raise ValueError(
                f"Expected one matching value: "
                f"{characteristic} = {product_value_code}, "
                f"found {len(matching_values)}"
            )

        selected_value = matching_values.iloc[0]

        requirement_type = rng.choices(
            ["MUST", "PREFERRED"],
            weights=[0.75, 0.25],
            k=1
        )[0]

        ground_truth_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": product_value_code,
            "VALUE_DEFINITION": selected_value["DEFINITION"],
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": requirement_type
        })

pilot_ground_truth = pd.DataFrame(ground_truth_rows)

display(pilot_ground_truth)

,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0001,WIDTH_CM,V001,45,EQUAL,MUST
1,R0001,OVEN_TYPE,V001,CONVECTION,EQUAL,PREFERRED
2,R0002,PANEL_COLOR,V001,BLACK,EQUAL,MUST
3,R0002,PANEL_TYPE,V001,GLASS,EQUAL,PREFERRED
4,R0002,ENERGY_CLASS,V001,A,EQUAL,MUST
5,R0003,PACKAGING_TYPE,V002,BOX,EQUAL,MUST
6,R0003,OVEN_FUNCTIONS,V002,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...,EQUAL,MUST
7,R0003,WIDTH_CM,V002,60,EQUAL,MUST
8,R0003,OVEN_TYPE,V001,CONVECTION,EQUAL,MUST
9,R0003,KNOB_STRUCTURE,V001,ROTARY,EQUAL,MUST


In [14]:
# Step 15 - Generate customer requirements independently of products

import random

rng = random.Random(456) #456 means that the random selection will be reproducible. You can change this number to get different random selections.

independent_rows = []

for i, request in request_profiles_df.iterrows():

    # Use new IDs to distinguish this generation method
    request_id = f"R{101 + i:04d}"

    for characteristic in request["SELECTED_CHARACTERISTICS"]:

        # Get valid values for the selected characteristic
        available_values = values[
            values["CHARACTERISTIC"] == characteristic
        ]

        if available_values.empty:
            raise ValueError(
                f"No values found for {characteristic}"
            )

        # Select a value without referring to PRODUCT_MASTER
        selected = available_values.sample(
            n=1,
            random_state=rng.randint(0, 100000)
        ).iloc[0]

        requirement_type = rng.choices(
            ["MUST", "PREFERRED"],
            weights=[0.65, 0.35],
            k=1
        )[0]

        independent_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": selected["VALUE"],
            "VALUE_DEFINITION": selected["DEFINITION"],
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": requirement_type
        })

independent_ground_truth = pd.DataFrame(independent_rows)

display(independent_ground_truth)

,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0101,WIDTH_CM,V003,70,EQUAL,MUST
1,R0101,OVEN_TYPE,V002,STEAM,EQUAL,MUST
2,R0102,PANEL_COLOR,V001,BLACK,EQUAL,MUST
3,R0102,PANEL_TYPE,V002,METAL,EQUAL,PREFERRED
4,R0102,ENERGY_CLASS,V001,A,EQUAL,PREFERRED
5,R0103,PACKAGING_TYPE,V002,BOX,EQUAL,MUST
6,R0103,OVEN_FUNCTIONS,V001,01-Top+Bottom | Bottom | Top,EQUAL,MUST
7,R0103,WIDTH_CM,V003,70,EQUAL,PREFERRED
8,R0103,OVEN_TYPE,V002,STEAM,EQUAL,MUST
9,R0103,KNOB_STRUCTURE,V001,ROTARY,EQUAL,MUST


In [15]:
# Step 16 - Combine ground truth from both generation methods

product_based = pilot_ground_truth.copy()
product_based["GENERATION_METHOD"] = "PRODUCT_ANCHORED"

independent = independent_ground_truth.copy()
independent["GENERATION_METHOD"] = "REQUIREMENT_FIRST"

request_ground_truth = pd.concat(
    [product_based, independent],
    ignore_index=True
)

# Check for duplicate requirement definitions
duplicates = request_ground_truth.duplicated(
    subset=["REQUEST_ID", "CHARACTERISTIC"]
).sum()

summary = (
    request_ground_truth
    .groupby("GENERATION_METHOD")
    .agg(
        N_REQUESTS=("REQUEST_ID", "nunique"),
        N_REQUIREMENTS=("CHARACTERISTIC", "count")
    )
    .reset_index()
)

display(summary)

print("Total requests:", request_ground_truth["REQUEST_ID"].nunique())
print("Total requirements:", len(request_ground_truth))
print("Duplicate characteristic assignments:", duplicates)

,GENERATION_METHOD,N_REQUESTS,N_REQUIREMENTS
0,PRODUCT_ANCHORED,5,19
1,REQUIREMENT_FIRST,5,19


Total requests: 10
Total requirements: 38
Duplicate characteristic assignments: 0


In [16]:
# Step 17 - Prepare customer language mapping

language_mapping = request_ground_truth[
    ["CHARACTERISTIC", "VALUE_DEFINITION"]
].drop_duplicates().copy()

language_mapping = language_mapping.sort_values(
    ["CHARACTERISTIC", "VALUE_DEFINITION"]
).reset_index(drop=True)

display(language_mapping)

,CHARACTERISTIC,VALUE_DEFINITION
0,ENERGY_CLASS,A
1,HANDLE_TYPE,H3
2,HANDLE_TYPE,H4
3,KNOB_COLOR,BLACK
4,KNOB_COLOR,SILVER
5,KNOB_COLOR,STAINLESS STEEL
6,KNOB_STRUCTURE,ROTARY
7,NUMBER_OF_FUNCTIONS,11
8,OVEN_FUNCTIONS,01-Top+Bottom | Bottom | Top
9,OVEN_FUNCTIONS,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...


In [17]:
# Step 18 - Build complete language mapping from master data

language_master = values[
    ["CHARACTERISTIC", "VALUE", "DEFINITION"]
].copy()

language_master = language_master.rename(
    columns={"DEFINITION": "VALUE_DEFINITION"}
)

language_master["CUSTOMER_LANGUAGE"] = ""
language_master["REVIEW_STATUS"] = "PENDING"

language_master = language_master.sort_values(
    ["CHARACTERISTIC", "VALUE"]
).reset_index(drop=True)

print("Total characteristic-value pairs:", len(language_master))
print(
    "Characteristics covered:",
    language_master["CHARACTERISTIC"].nunique()
)

display(language_master.head(15))

Total characteristic-value pairs: 90
Characteristics covered: 24


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,CUSTOMER_LANGUAGE,REVIEW_STATUS
0,BAKING_TRAY,V001,0,,PENDING
1,BAKING_TRAY,V002,1,,PENDING
2,BAKING_TRAY,V003,2,,PENDING
3,CLEANING,V001,ETC,,PENDING
4,CLEANING,V002,CATALYTIC,,PENDING
5,CLEANING,V003,PYROLYSIS,,PENDING
6,CLEANING,V004,HYDROCLEAN,,PENDING
7,ENERGY_CLASS,V001,A,,PENDING
8,ENERGY_CLASS,V002,A+,,PENDING
9,ENERGY_CLASS,V003,A++,,PENDING


In [18]:
# Step 19 - Generate initial English customer language

def create_customer_language(row):
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"]).strip()

    if characteristic == "WIDTH_CM":
        return f"{value} cm wide", "AUTO"

    elif characteristic == "OVEN_VOLUME_L":
        return f"{value}-liter oven capacity", "AUTO"

    elif characteristic == "NUMBER_OF_FUNCTIONS":
        return f"{value} cooking functions", "AUTO"

    elif characteristic == "PANEL_COLOR":
        return f"{value.lower()} control panel", "AUTO"

    elif characteristic == "PANEL_TYPE":
        return f"{value.lower()} control panel", "AUTO"

    elif characteristic == "OVEN_TYPE":
        return f"{value.lower()} oven", "AUTO"

    elif characteristic == "ENERGY_CLASS":
        return f"energy class {value}", "AUTO"

    elif characteristic == "CLEANING":
        cleaning_terms = {
            "ETC": "easy-to-clean enamel",
            "CATALYTIC": "catalytic cleaning",
            "PYROLYSIS": "pyrolytic cleaning",
            "HYDROCLEAN": "hydroclean cleaning"
        }
        if value in cleaning_terms:
            return cleaning_terms[value], "AUTO"
        return "", "REVIEW"

    elif characteristic == "WIFI":
        return (
            "Wi-Fi connectivity" if value == "YES"
            else "no Wi-Fi connectivity"
        ), "AUTO"

    elif characteristic in [
        "BAKING_TRAY",
        "ROASTING_TRAY",
        "WIRE_SHELF",
        "TELESCOPIC_RAILS"
    ]:
        names = {
            "BAKING_TRAY": "baking trays",
            "ROASTING_TRAY": "roasting trays",
            "WIRE_SHELF": "wire shelves",
            "TELESCOPIC_RAILS": "telescopic rail sets"
        }
        return f"{value} {names[characteristic]}", "AUTO"

    # Technical or complex values require manual review
    return "", "REVIEW"


language_master[
    ["CUSTOMER_LANGUAGE", "REVIEW_STATUS"]
] = language_master.apply(
    lambda row: pd.Series(create_customer_language(row)),
    axis=1
)

display(
    language_master[
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "CUSTOMER_LANGUAGE",
            "REVIEW_STATUS"
        ]
    ]
)

print("\nReview status:")
print(language_master["REVIEW_STATUS"].value_counts())

,CHARACTERISTIC,VALUE_DEFINITION,CUSTOMER_LANGUAGE,REVIEW_STATUS
0,BAKING_TRAY,0,0 baking trays,AUTO
1,BAKING_TRAY,1,1 baking trays,AUTO
2,BAKING_TRAY,2,2 baking trays,AUTO
3,CLEANING,ETC,easy-to-clean enamel,AUTO
4,CLEANING,CATALYTIC,catalytic cleaning,AUTO
...,...,...,...,...
85,WIFI,YES,Wi-Fi connectivity,AUTO
86,WIFI,NO,no Wi-Fi connectivity,AUTO
87,WIRE_SHELF,0,0 wire shelves,AUTO
88,WIRE_SHELF,1,1 wire shelves,AUTO



Review status:
REVIEW_STATUS
REVIEW    46
AUTO      44
Name: count, dtype: int64


In [19]:
# Step 20 - Review characteristics requiring language definitions

review_summary = (
    language_master[
        language_master["REVIEW_STATUS"] == "REVIEW"
    ]
    .groupby("CHARACTERISTIC")
    .agg(
        N_VALUES=("VALUE", "count"),
        EXAMPLE_VALUES=(
            "VALUE_DEFINITION",
            lambda x: ", ".join(x.astype(str).head(3))
        )
    )
    .reset_index()
    .sort_values("N_VALUES", ascending=False)
)

display(review_summary)


,CHARACTERISTIC,N_VALUES,EXAMPLE_VALUES
6,OVEN_FUNCTIONS,12,"01-Top+Bottom | Bottom | Top, 02-Defrost | Top..."
1,HANDLE_TYPE,5,"H1, H2, H3"
4,KNOB_TYPE,5,"K1, K2, K3"
9,POWER_CABLE,5,"W/O CABLE, W/O PLUG, EU TYPE PLUG"
0,HANDLE_COLOR,4,"BLACK, WHITE, SILVER"
2,KNOB_COLOR,4,"BLACK, WHITE, SILVER"
5,OVEN_CONTROL,3,"W/O TIMER , DIGITAL TIMER, ELECTRONIC TOUCH TIMER"
3,KNOB_STRUCTURE,2,"ROTARY, RETRACTABLE"
7,PACKAGING_TYPE,2,"SHRINK, BOX"
8,PIZZA_TRAY,2,"YES, NO"


In [20]:
# Step 21 - Extend customer language mapping

def extend_customer_language(row):
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"]).strip()
    status = row["REVIEW_STATUS"]

    if status != "REVIEW":
        return row["CUSTOMER_LANGUAGE"], status

    if characteristic == "HANDLE_COLOR":
        return f"{value.lower()} oven handle", "AUTO"

    elif characteristic == "KNOB_COLOR":
        return f"{value.lower()} control knobs", "AUTO"

    elif characteristic == "KNOB_STRUCTURE":
        descriptions = {
            "ROTARY": "rotary control knobs",
            "RETRACTABLE": "retractable control knobs"
        }
        return descriptions.get(value, ""), (
            "AUTO" if value in descriptions else "REVIEW"
        )

    elif characteristic == "OVEN_CONTROL":
        descriptions = {
            "W/O TIMER": "no timer",
            "DIGITAL TIMER": "digital timer",
            "ELECTRONIC TOUCH TIMER": "electronic touch timer"
        }
        return descriptions.get(value, ""), (
            "AUTO" if value in descriptions else "REVIEW"
        )

    elif characteristic == "PACKAGING_TYPE":
        return f"{value.lower()} packaging", "AUTO"

    elif characteristic == "PIZZA_TRAY":
        return (
            "pizza tray included" if value == "YES"
            else "no pizza tray included"
        ), "AUTO"

    elif characteristic == "TURNSPIT":
        return (
            "rotisserie turnspit included" if value == "YES"
            else "no rotisserie turnspit"
        ), "AUTO"

    elif characteristic == "POWER_CABLE":
        return value.lower(), "REVIEW"

    return row["CUSTOMER_LANGUAGE"], status


language_master[
    ["CUSTOMER_LANGUAGE", "REVIEW_STATUS"]
] = language_master.apply(
    lambda row: pd.Series(extend_customer_language(row)),
    axis=1
)

print(language_master["REVIEW_STATUS"].value_counts())

display(
    language_master[
        language_master["REVIEW_STATUS"] == "REVIEW"
    ][
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "CUSTOMER_LANGUAGE"
        ]
    ]
)

REVIEW_STATUS
AUTO      63
REVIEW    27
Name: count, dtype: int64


,CHARACTERISTIC,VALUE_DEFINITION,CUSTOMER_LANGUAGE
14,HANDLE_TYPE,H1,
15,HANDLE_TYPE,H2,
16,HANDLE_TYPE,H3,
17,HANDLE_TYPE,H4,
18,HANDLE_TYPE,H5,
25,KNOB_TYPE,K1,
26,KNOB_TYPE,K2,
27,KNOB_TYPE,K3,
28,KNOB_TYPE,K4,
29,KNOB_TYPE,K5,


In [21]:
# Step 22 - Add provisional language for technical design codes

def map_design_code(row):
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"]).strip()

    if characteristic == "HANDLE_TYPE":
        return f"{value} handle design", "PROVISIONAL"

    if characteristic == "KNOB_TYPE":
        return f"{value} control knob design", "PROVISIONAL"

    return row["CUSTOMER_LANGUAGE"], row["REVIEW_STATUS"]


mask = language_master["CHARACTERISTIC"].isin(
    ["HANDLE_TYPE", "KNOB_TYPE"]
)

language_master.loc[
    mask, ["CUSTOMER_LANGUAGE", "REVIEW_STATUS"]
] = language_master.loc[mask].apply(
    lambda row: pd.Series(map_design_code(row)),
    axis=1
).to_numpy()

display(
    language_master.loc[
        mask,
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "CUSTOMER_LANGUAGE",
            "REVIEW_STATUS"
        ]
    ]
)

print("\nReview status:")
print(language_master["REVIEW_STATUS"].value_counts())


,CHARACTERISTIC,VALUE_DEFINITION,CUSTOMER_LANGUAGE,REVIEW_STATUS
14,HANDLE_TYPE,H1,H1 handle design,PROVISIONAL
15,HANDLE_TYPE,H2,H2 handle design,PROVISIONAL
16,HANDLE_TYPE,H3,H3 handle design,PROVISIONAL
17,HANDLE_TYPE,H4,H4 handle design,PROVISIONAL
18,HANDLE_TYPE,H5,H5 handle design,PROVISIONAL
25,KNOB_TYPE,K1,K1 control knob design,PROVISIONAL
26,KNOB_TYPE,K2,K2 control knob design,PROVISIONAL
27,KNOB_TYPE,K3,K3 control knob design,PROVISIONAL
28,KNOB_TYPE,K4,K4 control knob design,PROVISIONAL
29,KNOB_TYPE,K5,K5 control knob design,PROVISIONAL



Review status:
REVIEW_STATUS
AUTO           63
REVIEW         17
PROVISIONAL    10
Name: count, dtype: int64


In [22]:
# Step 23 - Inspect remaining technical values

remaining_review = language_master.loc[
    language_master["REVIEW_STATUS"] == "REVIEW",
    [
        "CHARACTERISTIC",
        "VALUE",
        "VALUE_DEFINITION",
        "CUSTOMER_LANGUAGE"
    ]
].copy()

pd.set_option("display.max_colwidth", 200)

display(remaining_review.to_string(index=False))

'CHARACTERISTIC VALUE                                                                                                                                         VALUE_DEFINITION CUSTOMER_LANGUAGE\nOVEN_FUNCTIONS  V001                                                                                                                             01-Top+Bottom | Bottom | Top                  \nOVEN_FUNCTIONS  V002                                                                                            02-Defrost | Top+Bottom | Top+Bottom+Fan | Grill |  Grill+Fan                  \nOVEN_FUNCTIONS  V003                                                                                               03-Bottom | Top | Top+Bottom | Top+Bottom+Fan | Bottom+Fan                  \nOVEN_FUNCTIONS  V004                   04-OvenLight | Defrost | Top+Bottom | Top+Bottom+Fan | Bottom+Fan | Grill+Turnspit | Double Grill+Turnspit | Double Grill+Fan+Turnspit                  \nOVEN_FUNCTIONS  V005              

In [23]:
# Step 24 - Define the scope of the first NLP dataset

# Characteristics excluded from the first NLP dataset
excluded_characteristics = ["OVEN_FUNCTIONS"]

# Update the main characteristic profile
characteristic_profile.loc[
    characteristic_profile["CHARACTERISTIC"].isin(excluded_characteristics),
    "ENABLED"
] = False

# Create the generation profile from the updated main profile
generation_profile = characteristic_profile.copy()

# Complete provisional language for power cable values
power_mask = language_master["CHARACTERISTIC"] == "POWER_CABLE"

language_master.loc[
    power_mask, "CUSTOMER_LANGUAGE"
] = (
    language_master.loc[power_mask, "VALUE_DEFINITION"]
    .astype(str)
    .str.strip()
)

language_master.loc[
    power_mask, "REVIEW_STATUS"
] = "PROVISIONAL"

# Validate the generation scope
print(
    "Enabled characteristics:",
    generation_profile["ENABLED"].sum()
)

print("\nLanguage mapping status:")
print(language_master["REVIEW_STATUS"].value_counts())

print("\nExcluded from request generation:")
display(
    generation_profile.loc[
        ~generation_profile["ENABLED"],
        ["CHARACTERISTIC", "ENABLED"]
    ]
)

# Verify that both profiles have the same enabled status
assert characteristic_profile["ENABLED"].equals(
    generation_profile["ENABLED"]
)

assert not generation_profile.loc[
    generation_profile["CHARACTERISTIC"] == "OVEN_FUNCTIONS",
    "ENABLED"
].any()

print("\nGeneration scope validation: PASSED")

Enabled characteristics: 23

Language mapping status:
REVIEW_STATUS
AUTO           63
PROVISIONAL    15
REVIEW         12
Name: count, dtype: int64

Excluded from request generation:


,CHARACTERISTIC,ENABLED
9,OVEN_FUNCTIONS,False



Generation scope validation: PASSED


In [24]:
# Step 25 - Select characteristics for the first NLP dataset

import random

rng = random.Random(789)

def select_request_characteristics(profile, rng, min_count=2, max_count=6):

    eligible = profile.loc[
        profile["ENABLED"] == True
    ].copy()

    n = rng.randint(min_count, min(max_count, len(eligible)))

    selected = rng.choices(
        eligible["CHARACTERISTIC"].tolist(),
        weights=eligible["REQUEST_WEIGHT"].tolist(),
        k=n
    )

    # Remove duplicates while preserving order
    selected = list(dict.fromkeys(selected))

    return selected


pilot_profiles = []

for i in range(1, 11):

    selected = select_request_characteristics(
        generation_profile,
        rng
    )

    pilot_profiles.append({
        "REQUEST_ID": f"R{i:04d}",
        "N_CHARACTERISTICS": len(selected),
        "SELECTED_CHARACTERISTICS": selected
    })

pilot_profiles_df = pd.DataFrame(pilot_profiles)

display(pilot_profiles_df)

print(
    "Contains OVEN_FUNCTIONS:",
    pilot_profiles_df["SELECTED_CHARACTERISTICS"].apply(
        lambda x: "OVEN_FUNCTIONS" in x
    ).any()
)

,REQUEST_ID,N_CHARACTERISTICS,SELECTED_CHARACTERISTICS
0,R0001,4,"[HANDLE_COLOR, WIDTH_CM, PANEL_COLOR, ROASTING_TRAY]"
1,R0002,4,"[PIZZA_TRAY, KNOB_STRUCTURE, HANDLE_COLOR, OVEN_TYPE]"
2,R0003,3,"[TURNSPIT, TELESCOPIC_RAILS, NUMBER_OF_FUNCTIONS]"
3,R0004,5,"[PANEL_COLOR, PIZZA_TRAY, KNOB_COLOR, HANDLE_COLOR, WIDTH_CM]"
4,R0005,5,"[CLEANING, PIZZA_TRAY, WIDTH_CM, NUMBER_OF_FUNCTIONS, WIFI]"
5,R0006,3,"[PANEL_COLOR, OVEN_TYPE, KNOB_STRUCTURE]"
6,R0007,5,"[CLEANING, PIZZA_TRAY, KNOB_TYPE, ENERGY_CLASS, WIDTH_CM]"
7,R0008,3,"[WIDTH_CM, OVEN_VOLUME_L, OVEN_TYPE]"
8,R0009,4,"[ROASTING_TRAY, PANEL_COLOR, CLEANING, PANEL_TYPE]"
9,R0010,4,"[CLEANING, PANEL_COLOR, KNOB_COLOR, ROASTING_TRAY]"


Contains OVEN_FUNCTIONS: False


In [25]:
# Step 26 - Generate structured ground truth for pilot requests

import random

rng = random.Random(890)

ground_truth_rows = []

for _, request in pilot_profiles_df.iterrows():

    request_id = request["REQUEST_ID"]

    for characteristic in request["SELECTED_CHARACTERISTICS"]:

        available_values = language_master.loc[
            (language_master["CHARACTERISTIC"] == characteristic)
            & (language_master["REVIEW_STATUS"].isin(["AUTO", "PROVISIONAL"]))
        ]

        if available_values.empty:
            raise ValueError(
                f"No usable language mapping for {characteristic}"
            )

        selected = available_values.iloc[
            rng.randrange(len(available_values))
        ]

        requirement_type = rng.choices(
            ["MUST", "PREFERRED"],
            weights=[0.70, 0.30],
            k=1
        )[0]

        ground_truth_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": selected["VALUE"],
            "VALUE_DEFINITION": selected["VALUE_DEFINITION"],
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": requirement_type,
            "CUSTOMER_LANGUAGE": selected["CUSTOMER_LANGUAGE"]
        })

nlp_pilot_ground_truth = pd.DataFrame(ground_truth_rows)

display(nlp_pilot_ground_truth.head(15))

print("Requests:", nlp_pilot_ground_truth["REQUEST_ID"].nunique())
print("Requirements:", len(nlp_pilot_ground_truth))
print(
    "Missing customer language:",
    nlp_pilot_ground_truth["CUSTOMER_LANGUAGE"].isna().sum()
    + nlp_pilot_ground_truth["CUSTOMER_LANGUAGE"].eq("").sum()
)


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE,CUSTOMER_LANGUAGE
0,R0001,HANDLE_COLOR,V001,BLACK,EQUAL,PREFERRED,black oven handle
1,R0001,WIDTH_CM,V003,70,EQUAL,PREFERRED,70 cm wide
2,R0001,PANEL_COLOR,V005,DARK SILVER,EQUAL,PREFERRED,dark silver control panel
3,R0001,ROASTING_TRAY,V002,1,EQUAL,MUST,1 roasting trays
4,R0002,PIZZA_TRAY,V002,NO,EQUAL,MUST,no pizza tray included
5,R0002,KNOB_STRUCTURE,V002,RETRACTABLE,EQUAL,MUST,retractable control knobs
6,R0002,HANDLE_COLOR,V002,WHITE,EQUAL,PREFERRED,white oven handle
7,R0002,OVEN_TYPE,V003,COMBINATION,EQUAL,MUST,combination oven
8,R0003,TURNSPIT,V002,NO,EQUAL,MUST,no rotisserie turnspit
9,R0003,TELESCOPIC_RAILS,V002,1,EQUAL,MUST,1 telescopic rail sets


Requests: 10
Requirements: 40
Missing customer language: 0


In [26]:
# Step 27 - Generate customer request texts from ground truth

def compose_customer_request(requirements):

    must_sentences = []
    preferred_sentences = []

    for _, row in requirements.iterrows():

        phrase = str(row["CUSTOMER_LANGUAGE"]).strip()
        requirement_type = row["REQUIREMENT_TYPE"]

        if requirement_type == "MUST":
            must_sentences.append(
                f"It must have {phrase}."
            )

        elif requirement_type == "PREFERRED":
            preferred_sentences.append(
                f"I would prefer {phrase}."
            )

        else:
            raise ValueError(
                f"Unsupported requirement type: {requirement_type}"
            )

    return " ".join(
        ["I'm looking for an oven."]
        + must_sentences
        + preferred_sentences
    )


customer_request_rows = []

for request_id, group in nlp_pilot_ground_truth.groupby(
    "REQUEST_ID", sort=True
):

    request_text = compose_customer_request(group)

    customer_request_rows.append({
        "REQUEST_ID": request_id,
        "REQUEST_TEXT": request_text
    })


nlp_pilot_customer_requests = pd.DataFrame(
    customer_request_rows
)

pd.set_option("display.max_colwidth", 500)

display(nlp_pilot_customer_requests)

print(
    "Generated customer requests:",
    len(nlp_pilot_customer_requests)
)

,REQUEST_ID,REQUEST_TEXT
0,R0001,I'm looking for an oven. It must have 1 roasting trays. I would prefer black oven handle. I would prefer 70 cm wide. I would prefer dark silver control panel.
1,R0002,I'm looking for an oven. It must have no pizza tray included. It must have retractable control knobs. It must have combination oven. I would prefer white oven handle.
2,R0003,I'm looking for an oven. It must have no rotisserie turnspit. It must have 1 telescopic rail sets. I would prefer 8 cooking functions.
3,R0004,I'm looking for an oven. It must have stainless steel control panel. It must have no pizza tray included. It must have black control knobs. It must have stainless steel oven handle. It must have 60 cm wide.
4,R0005,I'm looking for an oven. It must have hydroclean cleaning. It must have 10 cooking functions. It must have Wi-Fi connectivity. I would prefer pizza tray included. I would prefer 70 cm wide.
5,R0006,I'm looking for an oven. It must have silver control panel. It must have steam oven. It must have rotary control knobs.
6,R0007,I'm looking for an oven. It must have hydroclean cleaning. It must have no pizza tray included. It must have K5 control knob design. It must have 70 cm wide. I would prefer energy class A+.
7,R0008,I'm looking for an oven. It must have 45 cm wide. It must have 65-liter oven capacity. I would prefer convection oven.
8,R0009,I'm looking for an oven. It must have 2 roasting trays. It must have black control panel. It must have glass control panel. I would prefer hydroclean cleaning.
9,R0010,I'm looking for an oven. It must have dark silver control panel. It must have black control knobs. It must have 1 roasting trays. I would prefer hydroclean cleaning.


Generated customer requests: 10


In [27]:
# Step 28 - Validate customer requests and ground truth

request_ids = set(
    nlp_pilot_customer_requests["REQUEST_ID"]
)

ground_truth_ids = set(
    nlp_pilot_ground_truth["REQUEST_ID"]
)

missing_ground_truth = request_ids - ground_truth_ids
missing_requests = ground_truth_ids - request_ids

duplicate_requests = (
    nlp_pilot_customer_requests["REQUEST_ID"]
    .duplicated()
    .sum()
)

missing_texts = (
    nlp_pilot_customer_requests["REQUEST_TEXT"]
    .isna()
    .sum()
)

print("Customer requests:", len(request_ids))
print("Ground truth requests:", len(ground_truth_ids))
print("Ground truth rows:", len(nlp_pilot_ground_truth))
print("Missing ground truth:", missing_ground_truth)
print("Missing customer requests:", missing_requests)
print("Duplicate request IDs:", duplicate_requests)
print("Missing request texts:", missing_texts)

Customer requests: 10
Ground truth requests: 10
Ground truth rows: 40
Missing ground truth: set()
Missing customer requests: set()
Duplicate request IDs: 0
Missing request texts: 0


In [28]:
# Step 29 - Inspect customer request diversity

requests = nlp_pilot_customer_requests.copy()   #copy of the generated customer requests for analysis

requests["WORD_COUNT"] = ( 
    requests["REQUEST_TEXT"]
    .str.split()
    .str.len()
)

requests["CHAR_COUNT"] = (
    requests["REQUEST_TEXT"]
    .str.len()
)

print("Number of requests:", len(requests))
print("Unique request texts:", requests["REQUEST_TEXT"].nunique())
print("Average word count:", round(requests["WORD_COUNT"].mean(), 1))
print("Minimum word count:", requests["WORD_COUNT"].min())
print("Maximum word count:", requests["WORD_COUNT"].max())

display(
    requests[
        ["REQUEST_ID", "WORD_COUNT", "CHAR_COUNT"]
    ]
)

Number of requests: 10
Unique request texts: 10
Average word count: 29.1
Minimum word count: 22
Maximum word count: 38


,REQUEST_ID,WORD_COUNT,CHAR_COUNT
0,R0001,30,158
1,R0002,29,166
2,R0003,24,134
3,R0004,38,206
4,R0005,33,189
5,R0006,22,119
6,R0007,36,189
7,R0008,22,118
8,R0009,28,159
9,R0010,29,165


In [29]:
# Step 30 - Define alternative customer request expressions

import random

must_templates = [
    "I need an oven that can {phrase}.",
    "The oven must {phrase}.",
    "It is essential for the oven to {phrase}.",
    "I'm looking for a model that can {phrase}."
]

preferred_templates = [
    "I would prefer a model that can {phrase}.",
    "Ideally, the oven should {phrase}.",
    "It would be nice if the oven could {phrase}.",
    "I would also like it to {phrase}."
]

print("MUST templates:", len(must_templates))
print("PREFERRED templates:", len(preferred_templates))

MUST templates: 4
PREFERRED templates: 4


In [30]:
# Step 31 - Generate alternative sentences for the same requirement

import random

rng_language = random.Random(42)

def generate_sentence(phrase, requirement_type, rng):
    if requirement_type == "MUST":
        templates = must_templates
    elif requirement_type == "PREFERRED":
        templates = preferred_templates
    else:
        raise ValueError(
            f"Unsupported requirement type: {requirement_type}"
        )

    template = rng.choice(templates)
    return template.format(phrase=phrase)


# Example: Same technical requirement, different expressions
phrase = "have Wi-Fi connectivity"

for i in range(5):
    sentence = generate_sentence(
        phrase,
        "MUST",
        rng_language
    )
    print(f"{i + 1}. {sentence}")

1. I need an oven that can have Wi-Fi connectivity.
2. I need an oven that can have Wi-Fi connectivity.
3. It is essential for the oven to have Wi-Fi connectivity.
4. The oven must have Wi-Fi connectivity.
5. The oven must have Wi-Fi connectivity.


In [31]:
# Step 32 - Explore short-form customer requests

request_examples = [
    {
        "REQUEST_TEXT": "60 cm, black panel, Wi-Fi required",
        "STYLE": "SHORT_LIST",
        "REQUIREMENT_TYPES": "MUST, MUST, MUST"
    },
    {
        "REQUEST_TEXT": "60 cm, black panel, preferably Wi-Fi",
        "STYLE": "SHORT_LIST",
        "REQUIREMENT_TYPES": "MUST, MUST, PREFERRED"
    },
    {
        "REQUEST_TEXT": "60 cm, black panel, Wi-Fi",
        "STYLE": "AMBIGUOUS_LIST",
        "REQUIREMENT_TYPES": "UNSPECIFIED"
    },
    {
        "REQUEST_TEXT": (
            "I need a 60 cm oven with a black panel. "
            "Wi-Fi would be nice."
        ),
        "STYLE": "FULL_SENTENCE",
        "REQUIREMENT_TYPES": "MUST, MUST, PREFERRED"
    }
]

display(pd.DataFrame(request_examples))

,REQUEST_TEXT,STYLE,REQUIREMENT_TYPES
0,"60 cm, black panel, Wi-Fi required",SHORT_LIST,"MUST, MUST, MUST"
1,"60 cm, black panel, preferably Wi-Fi",SHORT_LIST,"MUST, MUST, PREFERRED"
2,"60 cm, black panel, Wi-Fi",AMBIGUOUS_LIST,UNSPECIFIED
3,I need a 60 cm oven with a black panel. Wi-Fi would be nice.,FULL_SENTENCE,"MUST, MUST, PREFERRED"


In [32]:
# Step 33 - Generate a short-form request from ground truth

sample_id = "R0001"

sample_requirements = nlp_pilot_ground_truth[
    nlp_pilot_ground_truth["REQUEST_ID"] == sample_id
].copy()

must_items = []
preferred_items = []

for _, row in sample_requirements.iterrows():

    phrase = str(row["CUSTOMER_LANGUAGE"])

    if row["REQUIREMENT_TYPE"] == "MUST":
        must_items.append(phrase)

    elif row["REQUIREMENT_TYPE"] == "PREFERRED":
        preferred_items.append(phrase)

request_parts = []

if must_items:
    request_parts.append(
        "Required: " + ", ".join(must_items)
    )

if preferred_items:
    request_parts.append(
        "Preferred: " + ", ".join(preferred_items)
    )

short_request_text = ". ".join(request_parts) + "."

print("REQUEST_ID:", sample_id)
print("CUSTOMER REQUEST:")
print(short_request_text)

display(
    sample_requirements[
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "REQUIREMENT_TYPE",
            "CUSTOMER_LANGUAGE"
        ]
    ]
)

REQUEST_ID: R0001
CUSTOMER REQUEST:
Required: 1 roasting trays. Preferred: black oven handle, 70 cm wide, dark silver control panel.


,CHARACTERISTIC,VALUE_DEFINITION,REQUIREMENT_TYPE,CUSTOMER_LANGUAGE
0,HANDLE_COLOR,BLACK,PREFERRED,black oven handle
1,WIDTH_CM,70,PREFERRED,70 cm wide
2,PANEL_COLOR,DARK SILVER,PREFERRED,dark silver control panel
3,ROASTING_TRAY,1,MUST,1 roasting trays


In [33]:
# Step 34 - Generate customer requests in different styles

import random

def generate_request_text(requirements, style="SHORT_LIST"):

    must_items = []
    preferred_items = []

    for _, row in requirements.iterrows():

        phrase = str(row["CUSTOMER_LANGUAGE"])

        if row["REQUIREMENT_TYPE"] == "MUST":
            must_items.append(phrase)

        elif row["REQUIREMENT_TYPE"] == "PREFERRED":
            preferred_items.append(phrase)

    if style == "SHORT_LIST":

        parts = []

        if must_items:
            parts.append("Required: " + ", ".join(must_items))

        if preferred_items:
            parts.append("Preferred: " + ", ".join(preferred_items))

        return ". ".join(parts) + "."

    elif style == "FULL_SENTENCE":

        parts = ["I'm looking for an oven."]

        for phrase in must_items:
            parts.append(f"It must {phrase}.")

        for phrase in preferred_items:
            parts.append(f"I would prefer it to {phrase}.")

        return " ".join(parts)

    else:
        raise ValueError(f"Unsupported style: {style}")


# Test both styles with the same ground truth

sample_requirements = nlp_pilot_ground_truth[
    nlp_pilot_ground_truth["REQUEST_ID"] == "R0001"
]

print("SHORT_LIST:")
print(generate_request_text(sample_requirements, "SHORT_LIST"))

print("\nFULL_SENTENCE:")
print(generate_request_text(sample_requirements, "FULL_SENTENCE"))

SHORT_LIST:
Required: 1 roasting trays. Preferred: black oven handle, 70 cm wide, dark silver control panel.

FULL_SENTENCE:
I'm looking for an oven. It must 1 roasting trays. I would prefer it to black oven handle. I would prefer it to 70 cm wide. I would prefer it to dark silver control panel.


In [34]:
# Step 35 - Improve natural customer request composition

def generate_request_text(requirements, style="SHORT_LIST"):

    must_items = []
    preferred_items = []

    for _, row in requirements.iterrows():
        phrase = str(row["CUSTOMER_LANGUAGE"]).strip()

        if row["REQUIREMENT_TYPE"] == "MUST":
            must_items.append(phrase)

        elif row["REQUIREMENT_TYPE"] == "PREFERRED":
            preferred_items.append(phrase)

    if style == "SHORT_LIST":
        parts = []

        if must_items:
            parts.append("Required: " + ", ".join(must_items))

        if preferred_items:
            parts.append("Preferred: " + ", ".join(preferred_items))

        return ". ".join(parts) + "."

    elif style == "FULL_SENTENCE":
        parts = []

        if must_items:
            parts.append(
                "I'm looking for an oven with the following requirements: "
                + ", ".join(must_items)
                + "."
            )

        if preferred_items:
            parts.append(
                "Ideally, I would also like "
                + ", ".join(preferred_items)
                + "."
            )

        return " ".join(parts)

    else:
        raise ValueError(f"Unsupported style: {style}")


# Test with the same ground truth

sample_requirements = nlp_pilot_ground_truth[
    nlp_pilot_ground_truth["REQUEST_ID"] == "R0001"
]

print("SHORT_LIST:")
print(generate_request_text(sample_requirements, "SHORT_LIST"))

print("\nFULL_SENTENCE:")
print(generate_request_text(sample_requirements, "FULL_SENTENCE"))

SHORT_LIST:
Required: 1 roasting trays. Preferred: black oven handle, 70 cm wide, dark silver control panel.

FULL_SENTENCE:
I'm looking for an oven with the following requirements: 1 roasting trays. Ideally, I would also like black oven handle, 70 cm wide, dark silver control panel.


In [35]:
# Step 36 - Correct singular and plural expressions

def quantity_phrase(characteristic, value):
    quantity = int(value)

    nouns = {
        "BAKING_TRAY": ("baking tray", "baking trays"),
        "ROASTING_TRAY": ("roasting tray", "roasting trays"),
        "WIRE_SHELF": ("wire shelf", "wire shelves"),
        "TELESCOPIC_RAILS": (
            "telescopic rail set",
            "telescopic rail sets"
        ),
    }

    singular, plural = nouns[characteristic]

    noun = singular if quantity == 1 else plural

    return f"{quantity} {noun}"


quantity_characteristics = [
    "BAKING_TRAY",
    "ROASTING_TRAY",
    "WIRE_SHELF",
    "TELESCOPIC_RAILS",
]

mask = language_master["CHARACTERISTIC"].isin(
    quantity_characteristics
)

language_master.loc[mask, "CUSTOMER_LANGUAGE"] = (
    language_master.loc[mask].apply(
        lambda row: quantity_phrase(
            row["CHARACTERISTIC"],
            row["VALUE_DEFINITION"]
        ),
        axis=1
    )
)

display(
    language_master.loc[
        mask,
        ["CHARACTERISTIC", "VALUE_DEFINITION", "CUSTOMER_LANGUAGE"]
    ]
)

,CHARACTERISTIC,VALUE_DEFINITION,CUSTOMER_LANGUAGE
0,BAKING_TRAY,0,0 baking trays
1,BAKING_TRAY,1,1 baking tray
2,BAKING_TRAY,2,2 baking trays
73,ROASTING_TRAY,0,0 roasting trays
74,ROASTING_TRAY,1,1 roasting tray
75,ROASTING_TRAY,2,2 roasting trays
76,TELESCOPIC_RAILS,0,0 telescopic rail sets
77,TELESCOPIC_RAILS,1,1 telescopic rail set
78,TELESCOPIC_RAILS,2,2 telescopic rail sets
87,WIRE_SHELF,0,0 wire shelves


In [36]:
# Refresh customer language in pilot ground truth

language_lookup = language_master[
    ["CHARACTERISTIC", "VALUE", "CUSTOMER_LANGUAGE"]
].rename(
    columns={"CUSTOMER_LANGUAGE": "UPDATED_LANGUAGE"}
)

nlp_pilot_ground_truth = (
    nlp_pilot_ground_truth
    .drop(columns=["CUSTOMER_LANGUAGE"])
    .merge(
        language_lookup,
        on=["CHARACTERISTIC", "VALUE"],
        how="left",
        validate="many_to_one"
    )
    .rename(
        columns={"UPDATED_LANGUAGE": "CUSTOMER_LANGUAGE"}
    )
)

assert nlp_pilot_ground_truth["CUSTOMER_LANGUAGE"].notna().all()

sample_requirements = nlp_pilot_ground_truth[
    nlp_pilot_ground_truth["REQUEST_ID"] == "R0001"
]

print(generate_request_text(sample_requirements, "FULL_SENTENCE"))

I'm looking for an oven with the following requirements: 1 roasting tray. Ideally, I would also like black oven handle, 70 cm wide, dark silver control panel.


## Step 37 - Generate customer requests in batch

Generate a reproducible pilot dataset using the existing
requirement profiles, language mapping, and text generation functions.

Each customer request must have corresponding ground truth records.

In [37]:
# Step 37 - Generate customer requests in batch

import random
import pandas as pd

N_REQUESTS = 100
BATCH_SEED = 42 # reproducible random selection for batch generation

rng_batch = random.Random(BATCH_SEED)

batch_profiles = []

for i in range(1, N_REQUESTS + 1):

    request_id = f"R{i:04d}"

    selected = select_characteristics(rng_batch)

    batch_profiles.append({
        "REQUEST_ID": request_id,
        "N_CHARACTERISTICS": len(selected),
        "SELECTED_CHARACTERISTICS": selected
    })

batch_profiles_df = pd.DataFrame(batch_profiles)

print("Generated request profiles:", len(batch_profiles_df))

display(batch_profiles_df.head(10))

Generated request profiles: 100


,REQUEST_ID,N_CHARACTERISTICS,SELECTED_CHARACTERISTICS
0,R0001,2,"[WIDTH_CM, OVEN_TYPE]"
1,R0002,3,"[PANEL_COLOR, PANEL_TYPE, ENERGY_CLASS]"
2,R0003,6,"[PACKAGING_TYPE, OVEN_FUNCTIONS, WIDTH_CM, OVEN_TYPE, KNOB_STRUCTURE, PANEL_TYPE]"
3,R0004,3,"[HANDLE_TYPE, KNOB_COLOR, NUMBER_OF_FUNCTIONS]"
4,R0005,5,"[ROASTING_TRAY, TELESCOPIC_RAILS, WIDTH_CM, OVEN_VOLUME_L, KNOB_COLOR]"
5,R0006,4,"[OVEN_TYPE, PANEL_COLOR, OVEN_VOLUME_L, PACKAGING_TYPE]"
6,R0007,5,"[PANEL_TYPE, CLEANING, ROASTING_TRAY, HANDLE_COLOR, KNOB_TYPE]"
7,R0008,6,"[PANEL_COLOR, OVEN_CONTROL, WIDTH_CM, NUMBER_OF_FUNCTIONS, KNOB_TYPE, CLEANING]"
8,R0009,4,"[ROASTING_TRAY, HANDLE_TYPE, WIDTH_CM, OVEN_TYPE]"
9,R0010,4,"[OVEN_CONTROL, OVEN_VOLUME_L, ENERGY_CLASS, OVEN_TYPE]"


In [38]:
# Step 38 - Validate and correct characteristic selection

# Exclude OVEN_FUNCTIONS from the first NLP dataset

characteristic_profile.loc[
    characteristic_profile["CHARACTERISTIC"] == "OVEN_FUNCTIONS",
    "ENABLED"
] = False

print("OVEN_FUNCTIONS enabled:",
      characteristic_profile.loc[
          characteristic_profile["CHARACTERISTIC"] == "OVEN_FUNCTIONS",
          "ENABLED"
      ].iloc[0])

print("Profiles containing OVEN_FUNCTIONS:",
      batch_profiles_df["SELECTED_CHARACTERISTICS"]
      .apply(lambda x: "OVEN_FUNCTIONS" in x)
      .sum())

# Use only enabled characteristics
enabled_profile = characteristic_profile[
    characteristic_profile["ENABLED"] == True
].copy()

def select_batch_characteristics(rng):
    n = rng.randint(2, 6)

    # Weighted sampling without replacement
    selected = []

    available = enabled_profile.copy()

    for _ in range(min(n, len(available))):
        chosen = rng.choices(
            available["CHARACTERISTIC"].tolist(),
            weights=available["REQUEST_WEIGHT"].tolist(),
            k=1
        )[0]

        selected.append(chosen)

        available = available[
            available["CHARACTERISTIC"] != chosen
        ]

    return selected

# Regenerate the batch using the corrected function
rng_batch = random.Random(BATCH_SEED)

for profile in batch_profiles:
    selected = select_batch_characteristics(rng_batch)

    profile["SELECTED_CHARACTERISTICS"] = selected
    profile["N_CHARACTERISTICS"] = len(selected)

batch_profiles_df = pd.DataFrame(batch_profiles)

print("\nTotal profiles:", len(batch_profiles_df))
print(
    "Profiles containing OVEN_FUNCTIONS:",
    batch_profiles_df["SELECTED_CHARACTERISTICS"]
    .apply(lambda x: "OVEN_FUNCTIONS" in x)
    .sum()
)

display(batch_profiles_df.head(10))

OVEN_FUNCTIONS enabled: False
Profiles containing OVEN_FUNCTIONS: 31

Total profiles: 100
Profiles containing OVEN_FUNCTIONS: 0


,REQUEST_ID,N_CHARACTERISTICS,SELECTED_CHARACTERISTICS
0,R0001,2,"[WIDTH_CM, OVEN_TYPE]"
1,R0002,3,"[PANEL_COLOR, PACKAGING_TYPE, OVEN_VOLUME_L]"
2,R0003,6,"[WIDTH_CM, BAKING_TRAY, PANEL_TYPE, OVEN_TYPE, KNOB_TYPE, PANEL_COLOR]"
3,R0004,3,"[HANDLE_COLOR, HANDLE_TYPE, NUMBER_OF_FUNCTIONS]"
4,R0005,5,"[KNOB_STRUCTURE, TELESCOPIC_RAILS, WIDTH_CM, CLEANING, KNOB_COLOR]"
5,R0006,4,"[TURNSPIT, PANEL_COLOR, OVEN_VOLUME_L, WIDTH_CM]"
6,R0007,5,"[PACKAGING_TYPE, CLEANING, ROASTING_TRAY, ENERGY_CLASS, KNOB_COLOR]"
7,R0008,6,"[PANEL_TYPE, OVEN_CONTROL, WIDTH_CM, OVEN_TYPE, KNOB_COLOR, CLEANING]"
8,R0009,4,"[WIRE_SHELF, HANDLE_COLOR, WIDTH_CM, TURNSPIT]"
9,R0010,4,"[OVEN_CONTROL, OVEN_VOLUME_L, TELESCOPIC_RAILS, OVEN_TYPE]"


In [39]:
# Step 39 - Generate structured ground truth for 100 customer requests

rng_values = random.Random(42)

ground_truth_rows = []

# Keep only values from enabled characteristics
enabled_characteristics = set(
    generation_profile.loc[
        generation_profile["ENABLED"],
        "CHARACTERISTIC"
    ]
)

available_values = value_lookup[
    value_lookup["CHARACTERISTIC"].isin(enabled_characteristics)
].copy()

for _, profile in batch_profiles_df.iterrows():

    request_id = profile["REQUEST_ID"]

    for characteristic in profile["SELECTED_CHARACTERISTICS"]:

        # Find valid values for this characteristic
        candidates = available_values[
            available_values["CHARACTERISTIC"] == characteristic
        ]

        if candidates.empty:
            raise ValueError(
                f"No values found for characteristic: {characteristic}"
            )

        # Randomly select one characteristic value
        selected = candidates.iloc[
            rng_values.randrange(len(candidates))
        ]

        # Assign requirement priority
        requirement_type = rng_values.choices(
            ["MUST", "PREFERRED"],
            weights=[0.75, 0.25],
            k=1
        )[0]

        ground_truth_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": selected["VALUE"],
            "VALUE_DEFINITION": selected["DEFINITION"],
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": requirement_type
        })

batch_ground_truth = pd.DataFrame(ground_truth_rows)

print("Total requests:", batch_ground_truth["REQUEST_ID"].nunique())
print("Total requirements:", len(batch_ground_truth))

print("\nRequirement types:")
print(batch_ground_truth["REQUIREMENT_TYPE"].value_counts())

display(batch_ground_truth.head(15))

Total requests: 100
Total requirements: 402

Requirement types:
REQUIREMENT_TYPE
MUST         308
PREFERRED     94
Name: count, dtype: int64


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0001,WIDTH_CM,V001,45,EQUAL,MUST
1,R0001,OVEN_TYPE,V002,STEAM,EQUAL,MUST
2,R0002,PANEL_COLOR,V002,WHITE,EQUAL,MUST
3,R0002,PACKAGING_TYPE,V001,SHRINK,EQUAL,MUST
4,R0002,OVEN_VOLUME_L,V001,50,EQUAL,MUST
5,R0003,WIDTH_CM,V002,60,EQUAL,MUST
6,R0003,BAKING_TRAY,V003,2,EQUAL,MUST
7,R0003,PANEL_TYPE,V001,GLASS,EQUAL,MUST
8,R0003,OVEN_TYPE,V003,COMBINATION,EQUAL,MUST
9,R0003,KNOB_TYPE,V002,K2,EQUAL,MUST


In [40]:
# Step 40 - Add customer language to batch ground truth

language_columns = [
    "CHARACTERISTIC",
    "VALUE",
    "CUSTOMER_LANGUAGE"
]

batch_ground_truth_with_language = batch_ground_truth.merge(
    language_master[language_columns],
    on=["CHARACTERISTIC", "VALUE"],
    how="left",
    validate="many_to_one"
)

# Check for missing language definitions
missing_language = (
    batch_ground_truth_with_language["CUSTOMER_LANGUAGE"]
    .isna()
    .sum()
)

print("Total requirements:", len(batch_ground_truth_with_language))
print("Missing customer language:", missing_language)

assert missing_language == 0, (
    "Some characteristic values have no customer language mapping."
)

display(
    batch_ground_truth_with_language[
        [
            "REQUEST_ID",
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "REQUIREMENT_TYPE",
            "CUSTOMER_LANGUAGE"
        ]
    ].head(15)
)

Total requirements:

 402
Missing customer language: 0


,REQUEST_ID,CHARACTERISTIC,VALUE_DEFINITION,REQUIREMENT_TYPE,CUSTOMER_LANGUAGE
0,R0001,WIDTH_CM,45,MUST,45 cm wide
1,R0001,OVEN_TYPE,STEAM,MUST,steam oven
2,R0002,PANEL_COLOR,WHITE,MUST,white control panel
3,R0002,PACKAGING_TYPE,SHRINK,MUST,shrink packaging
4,R0002,OVEN_VOLUME_L,50,MUST,50-liter oven capacity
5,R0003,WIDTH_CM,60,MUST,60 cm wide
6,R0003,BAKING_TRAY,2,MUST,2 baking trays
7,R0003,PANEL_TYPE,GLASS,MUST,glass control panel
8,R0003,OVEN_TYPE,COMBINATION,MUST,combination oven
9,R0003,KNOB_TYPE,K2,MUST,K2 control knob design


In [41]:
# Step 41 - Generate customer request texts

rng_text = random.Random(42)

customer_request_rows = []

for request_id, requirements in (
    batch_ground_truth_with_language.groupby("REQUEST_ID", sort=True)
):

    # Randomly select one of the two supported writing styles
    style = rng_text.choice([
        "SHORT_LIST",
        "FULL_SENTENCE"
    ])

    # Generate text using the existing function
    request_text = generate_request_text(
        requirements,
        style
    )

    customer_request_rows.append({
        "REQUEST_ID": request_id,
        "REQUEST_TEXT": request_text,
        "STYLE": style
    })

batch_customer_requests = pd.DataFrame(customer_request_rows)

print("Generated customer requests:", len(batch_customer_requests))

print("\nWriting styles:")
print(batch_customer_requests["STYLE"].value_counts())

display(
    batch_customer_requests[
        ["REQUEST_ID", "STYLE", "REQUEST_TEXT"]
    ].head(10)
)

Generated customer requests: 100

Writing styles:
STYLE
SHORT_LIST       56
FULL_SENTENCE    44
Name: count, dtype: int64


,REQUEST_ID,STYLE,REQUEST_TEXT
0,R0001,SHORT_LIST,"Required: 45 cm wide, steam oven."
1,R0002,SHORT_LIST,"Required: white control panel, shrink packaging, 50-liter oven capacity."
2,R0003,FULL_SENTENCE,"I'm looking for an oven with the following requirements: 60 cm wide, 2 baking trays, glass control panel, combination oven, K2 control knob design. Ideally, I would also like silver control panel."
3,R0004,SHORT_LIST,"Required: H2 handle design, 8 cooking functions. Preferred: black oven handle."
4,R0005,SHORT_LIST,"Required: 1 telescopic rail set, 90 cm wide, pyrolytic cleaning, black control knobs. Preferred: rotary control knobs."
5,R0006,SHORT_LIST,"Required: stainless steel control panel, 70 cm wide. Preferred: rotisserie turnspit included, 75-liter oven capacity."
6,R0007,SHORT_LIST,"Required: shrink packaging, energy class A, stainless steel control knobs. Preferred: catalytic cleaning, 0 roasting trays."
7,R0008,SHORT_LIST,"Required: metal control panel, digital timer, 70 cm wide, combination oven, white control knobs, catalytic cleaning."
8,R0009,FULL_SENTENCE,"I'm looking for an oven with the following requirements: 1 wire shelf, white oven handle, 45 cm wide. Ideally, I would also like rotisserie turnspit included."
9,R0010,SHORT_LIST,"Required: digital timer, steam oven. Preferred: 65-liter oven capacity, 2 telescopic rail sets."


## Step 42 - Save and validate the pilot NLP dataset

Save customer request texts and their structured ground truth
as separate CSV files.

Validate request IDs, missing values, duplicate requirements,
and consistency between the two datasets.

In [42]:
# Step 42 - Save and validate the pilot NLP dataset

from pathlib import Path

# Define output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Prepare customer requests
customer_requests = batch_customer_requests[
    ["REQUEST_ID", "STYLE", "REQUEST_TEXT"]
].copy()

# Prepare structured ground truth
request_ground_truth = batch_ground_truth.copy()

# Validate request IDs
request_ids = set(customer_requests["REQUEST_ID"])
ground_truth_ids = set(request_ground_truth["REQUEST_ID"])

assert request_ids == ground_truth_ids, (
    "Customer requests and ground truth IDs do not match."
)

# Validate duplicate request IDs
assert not customer_requests["REQUEST_ID"].duplicated().any(), (
    "Duplicate customer request IDs found."
)

# Validate duplicate characteristic assignments
assert not request_ground_truth.duplicated(
    subset=["REQUEST_ID", "CHARACTERISTIC"]
).any(), (
    "Duplicate characteristic assignments found."
)

# Validate missing values in essential fields
assert customer_requests[
    ["REQUEST_ID", "REQUEST_TEXT"]
].notna().all().all(), (
    "Missing customer request data found."
)

assert request_ground_truth[
    [
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "OPERATOR",
        "REQUIREMENT_TYPE"
    ]
].notna().all().all(), (
    "Missing ground truth data found."
)

# Validate requirement types
assert request_ground_truth["REQUIREMENT_TYPE"].isin(
    ["MUST", "PREFERRED"]
).all(), (
    "Unsupported requirement type found."
)

# Save CSV files
customer_requests_path = output_dir / "customer_requests.csv"
ground_truth_path = output_dir / "request_ground_truth.csv"

customer_requests.to_csv(
    customer_requests_path,
    index=False,
    encoding="utf-8-sig"
)

request_ground_truth.to_csv(
    ground_truth_path,
    index=False,
    encoding="utf-8-sig"
)

# Display summary
print("Pilot NLP dataset saved successfully.")
print()
print("Customer requests:", len(customer_requests))
print("Ground truth rows:", len(request_ground_truth))
print("Unique request IDs:", len(request_ids))
print()
print("Saved files:")
print(customer_requests_path)
print(ground_truth_path)

Pilot NLP dataset saved successfully.

Customer requests: 100
Ground truth rows: 402
Unique request IDs: 100

Saved files:
../data/processed/customer_requests.csv
../data/processed/request_ground_truth.csv


## Step 43 - Validate text and ground truth consistency

Check whether each generated customer request contains the
customer-language expressions associated with its ground truth.

This is a lexical consistency check, not a complete semantic validation.

In [43]:
# Step 43 - Validate text and ground truth consistency

# Prepare language mapping
language_lookup = language_master[
    ["CHARACTERISTIC", "VALUE", "CUSTOMER_LANGUAGE"]
].drop_duplicates(
    subset=["CHARACTERISTIC", "VALUE"]
)

# Add customer language to ground truth
validation_data = request_ground_truth.merge(
    language_lookup,
    on=["CHARACTERISTIC", "VALUE"],
    how="left",
    validate="many_to_one"
)

# Add generated customer request texts
validation_data = validation_data.merge(
    customer_requests[
        ["REQUEST_ID", "REQUEST_TEXT"]
    ],
    on="REQUEST_ID",
    how="left",
    validate="many_to_one"
)

# Check missing language mappings
missing_language = validation_data[
    "CUSTOMER_LANGUAGE"
].isna().sum()

# Check whether each phrase appears in the request text
validation_data["PHRASE_FOUND"] = validation_data.apply(
    lambda row: (
        isinstance(row["CUSTOMER_LANGUAGE"], str)
        and row["CUSTOMER_LANGUAGE"].lower()
        in str(row["REQUEST_TEXT"]).lower()
    ),
    axis=1  #axis=1 to apply the function row-wise
)

# Summary
total_requirements = len(validation_data)
matched_requirements = int(
    validation_data["PHRASE_FOUND"].sum()
)
missing_phrases = total_requirements - matched_requirements

print("Total requirements:", total_requirements)
print("Matching phrases:", matched_requirements)
print("Missing phrases:", missing_phrases)
print("Missing language mappings:", missing_language)

print(
    "Phrase coverage:",
    f"{matched_requirements / total_requirements:.1%}"
)

print("\nRequirements requiring review:")

display(
    validation_data.loc[
        ~validation_data["PHRASE_FOUND"],
        [
            "REQUEST_ID",
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "REQUIREMENT_TYPE",
            "CUSTOMER_LANGUAGE",
            "REQUEST_TEXT"
        ]
    ].head(20)
)

Total requirements: 402
Matching phrases: 402
Missing phrases: 0
Missing language mappings: 0
Phrase coverage: 100.0%

Requirements requiring review:


,REQUEST_ID,CHARACTERISTIC,VALUE_DEFINITION,REQUIREMENT_TYPE,CUSTOMER_LANGUAGE,REQUEST_TEXT


## Step 44 - Validate requirement types in customer texts

Verify that MUST and PREFERRED requirements appear in the
correct sections of the generated customer request.

This validation is specific to the current text templates.

In [44]:
# Step 44 - Validate requirement types in customer texts

def extract_requirement_sections(request_text, style):
    text = str(request_text)

    if style == "SHORT_LIST":
        if "Preferred:" in text:
            must_section, preferred_section = text.split(
                "Preferred:", 1
            )
        else:
            must_section = text
            preferred_section = ""

    elif style == "FULL_SENTENCE":
        if "Ideally," in text:
            must_section, preferred_section = text.split(
                "Ideally,", 1
            )
        else:
            must_section = text
            preferred_section = ""

    else:
        raise ValueError(f"Unsupported style: {style}")

    return must_section.lower(), preferred_section.lower()


# Add writing style to validation data
type_validation = validation_data.merge(
    customer_requests[["REQUEST_ID", "STYLE"]],
    on="REQUEST_ID",
    how="left",
    validate="many_to_one"
)


def check_requirement_type(row):
    must_section, preferred_section = extract_requirement_sections(
        row["REQUEST_TEXT"],
        row["STYLE"]
    )

    phrase = str(row["CUSTOMER_LANGUAGE"]).lower()

    if row["REQUIREMENT_TYPE"] == "MUST":
        return phrase in must_section

    if row["REQUIREMENT_TYPE"] == "PREFERRED":
        return phrase in preferred_section

    return False


type_validation["TYPE_MATCH"] = type_validation.apply(
    check_requirement_type,
    axis=1
)

total = len(type_validation)
correct = int(type_validation["TYPE_MATCH"].sum())

print("Total requirements:", total)
print("Correct requirement types:", correct)
print("Incorrect requirement types:", total - correct)
print(f"Type consistency: {correct / total:.1%}")

print("\nRequirements requiring review:")

display(
    type_validation.loc[
        ~type_validation["TYPE_MATCH"],
        [
            "REQUEST_ID",
            "STYLE",
            "CHARACTERISTIC",
            "REQUIREMENT_TYPE",
            "CUSTOMER_LANGUAGE",
            "REQUEST_TEXT"
        ]
    ].head(20)
)

Total requirements: 402
Correct requirement types: 402
Incorrect requirement types: 0
Type consistency: 100.0%

Requirements requiring review:


,REQUEST_ID,STYLE,CHARACTERISTIC,REQUIREMENT_TYPE,CUSTOMER_LANGUAGE,REQUEST_TEXT


## Step 45 - Audit dataset diversity and label distribution

Analyze characteristic coverage, requirement type distribution,
and customer request writing styles before scaling the dataset.

In [45]:
# Step 45 - Audit dataset diversity and label distribution

print("DATASET SUMMARY")
print("-" * 40)

print("Customer requests:", len(customer_requests))
print("Ground truth requirements:", len(request_ground_truth))

print(
    "Characteristics represented:",
    request_ground_truth["CHARACTERISTIC"].nunique()
)

print("\nREQUIREMENT TYPE DISTRIBUTION")
display(
    request_ground_truth["REQUIREMENT_TYPE"]
    .value_counts()
    .rename_axis("REQUIREMENT_TYPE")
    .reset_index(name="COUNT")
)

print("\nWRITING STYLE DISTRIBUTION")
display(
    customer_requests["STYLE"]
    .value_counts()
    .rename_axis("STYLE")
    .reset_index(name="COUNT")
)

print("\nCHARACTERISTIC COVERAGE")
characteristic_coverage = (
    request_ground_truth
    .groupby("CHARACTERISTIC")
    .agg(
        N_REQUESTS=("REQUEST_ID", "nunique"),
        N_VALUES=("VALUE", "nunique")
    )
    .reset_index()
    .sort_values("N_REQUESTS", ascending=False)
)

display(characteristic_coverage)

print("\nREQUIREMENTS PER REQUEST")
requirements_per_request = (
    request_ground_truth.groupby("REQUEST_ID")
    .size()
)

display(requirements_per_request.describe())

DATASET SUMMARY
----------------------------------------
Customer requests: 100
Ground truth requirements: 402
Characteristics represented: 23

REQUIREMENT TYPE DISTRIBUTION


,REQUIREMENT_TYPE,COUNT
0,MUST,308
1,PREFERRED,94



WRITING STYLE DISTRIBUTION


,STYLE,COUNT
0,SHORT_LIST,56
1,FULL_SENTENCE,44



CHARACTERISTIC COVERAGE


,CHARACTERISTIC,N_REQUESTS,N_VALUES
20,WIDTH_CM,37,4
1,CLEANING,37,4
13,PANEL_COLOR,36,5
10,OVEN_TYPE,33,3
9,OVEN_CONTROL,30,3
8,NUMBER_OF_FUNCTIONS,26,5
19,TURNSPIT,18,2
0,BAKING_TRAY,17,3
6,KNOB_STRUCTURE,16,2
4,HANDLE_TYPE,16,5



REQUIREMENTS PER REQUEST


count    100.000000
mean       4.020000
std        1.421196
min        2.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        6.000000
dtype: float64

## Step 46 - Measure characteristic-value coverage

Compare the generated ground truth with the master
characteristic-value dictionary to identify values
that are missing from the pilot NLP dataset.

In [46]:
# Step 46 - Measure characteristic-value coverage

# Include only characteristics enabled for NLP generation
enabled_characteristics = set(
    generation_profile.loc[
        generation_profile["ENABLED"],
        "CHARACTERISTIC"
    ]
)

# All eligible characteristic-value pairs
eligible_values = language_master.loc[
    language_master["CHARACTERISTIC"].isin(
        enabled_characteristics
    ),
    ["CHARACTERISTIC", "VALUE", "VALUE_DEFINITION"]
].drop_duplicates(
    subset=["CHARACTERISTIC", "VALUE"]
)

# Characteristic-value pairs represented in ground truth
used_values = (
    request_ground_truth
    .groupby(["CHARACTERISTIC", "VALUE"])
    .size()
    .reset_index(name="N_OCCURRENCES")
)

# Compare master values with generated examples
value_coverage = eligible_values.merge(
    used_values,
    on=["CHARACTERISTIC", "VALUE"],
    how="left",
    validate="one_to_one"
)

value_coverage["N_OCCURRENCES"] = (
    value_coverage["N_OCCURRENCES"]
    .fillna(0)
    .astype(int)
)

value_coverage["COVERED"] = (
    value_coverage["N_OCCURRENCES"] > 0
)

total_values = len(value_coverage)
covered_values = int(value_coverage["COVERED"].sum())

print("Eligible characteristic-value pairs:", total_values)
print("Covered pairs:", covered_values)
print("Missing pairs:", total_values - covered_values)
print(f"Value coverage: {covered_values / total_values:.1%}")

print("\nMissing characteristic-value pairs:")

display(
    value_coverage.loc[
        ~value_coverage["COVERED"],
        [
            "CHARACTERISTIC",
            "VALUE",
            "VALUE_DEFINITION"
        ]
    ]
)

Eligible characteristic-value pairs: 78
Covered pairs: 76
Missing pairs: 2
Value coverage: 97.4%

Missing characteristic-value pairs:


,CHARACTERISTIC,VALUE,VALUE_DEFINITION
13,HANDLE_COLOR,V004,STAINLESS STEEL
58,POWER_CABLE,V003,EU TYPE PLUG


## Step 47 - Analyze rare characteristic-value pairs

Identify underrepresented characteristic-value pairs
before scaling the synthetic customer request dataset.

In [47]:
# Step 47 - Analyze rare characteristic-value pairs

minimum_examples = 5

frequency_summary = (
    value_coverage["N_OCCURRENCES"]
    .describe()
)

print("VALUE FREQUENCY SUMMARY")
display(frequency_summary)

rare_values = (
    value_coverage.loc[
        value_coverage["N_OCCURRENCES"] < minimum_examples,
        [
            "CHARACTERISTIC",
            "VALUE",
            "VALUE_DEFINITION",
            "N_OCCURRENCES"
        ]
    ]
    .sort_values(
        ["N_OCCURRENCES", "CHARACTERISTIC"]
    )
    .reset_index(drop=True)
)

print(
    f"\nValues with fewer than {minimum_examples} examples:",
    len(rare_values)
)

display(rare_values)

print("\nCOVERAGE BY FREQUENCY")

frequency_bands = pd.cut(
    value_coverage["N_OCCURRENCES"],
    bins=[-1, 0, 2, 4, 9, float("inf")],
    labels=["0", "1-2", "3-4", "5-9", "10+"]
)

display(
    frequency_bands
    .value_counts()
    .reindex(["0", "1-2", "3-4", "5-9", "10+"])
    .rename_axis("EXAMPLES_PER_VALUE")
    .reset_index(name="N_VALUES")
)

VALUE FREQUENCY SUMMARY


count    78.000000
mean      5.153846
std       3.438049
min       0.000000
25%       2.000000
50%       5.000000
75%       7.000000
max      19.000000
Name: N_OCCURRENCES, dtype: float64


Values with fewer than 5 examples: 38


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,N_OCCURRENCES
0,HANDLE_COLOR,V004,STAINLESS STEEL,0
1,POWER_CABLE,V003,EU TYPE PLUG,0
2,HANDLE_TYPE,V003,H3,1
3,KNOB_COLOR,V003,SILVER,1
4,OVEN_VOLUME_L,V003,75,1
5,POWER_CABLE,V002,W/O PLUG,1
6,POWER_CABLE,V004,US TYPE PLUG,1
7,POWER_CABLE,V005,UK TYPE PLUG,1
8,ENERGY_CLASS,V001,A,2
9,HANDLE_COLOR,V003,SILVER,2



COVERAGE BY FREQUENCY


,EXAMPLES_PER_VALUE,N_VALUES
0,0,2
1,1-2,19
2,3-4,17
3,5-9,32
4,10+,8


## Step 48 - Define dataset expansion targets

Set initial targets for expanding the synthetic NLP dataset.
These are data-generation targets, not model performance thresholds.

In [48]:
# Step 48 - Define dataset expansion targets

expansion_targets = {
    "N_REQUESTS": 1000,
    "MIN_EXAMPLES_PER_VALUE": 10,
    "MIN_REQUIREMENTS_PER_REQUEST": 2,
    "MAX_REQUIREMENTS_PER_REQUEST": 6,
    "RANDOM_SEED": 42,
    "WRITING_STYLES": [
        "SHORT_LIST",
        "FULL_SENTENCE"
    ]
}

current_requests = customer_requests["REQUEST_ID"].nunique()

current_requirements = len(request_ground_truth)

current_value_counts = value_coverage["N_OCCURRENCES"]

print("DATASET EXPANSION PLAN")
print("-" * 40)

print("Current requests:", current_requests)
print("Target requests:", expansion_targets["N_REQUESTS"])

print(
    "Additional requests:",
    expansion_targets["N_REQUESTS"] - current_requests
)

print("Current requirements:", current_requirements)

print(
    "Values below target frequency:",
    (current_value_counts < expansion_targets["MIN_EXAMPLES_PER_VALUE"]).sum()
)

print(
    "Minimum examples per value:",
    expansion_targets["MIN_EXAMPLES_PER_VALUE"]
)

print(
    "Writing styles:",
    expansion_targets["WRITING_STYLES"]
)

DATASET EXPANSION PLAN
----------------------------------------
Current requests: 100
Target requests: 1000
Additional requests: 900
Current requirements: 402
Values below target frequency: 70
Minimum examples per value: 10
Writing styles: ['SHORT_LIST', 'FULL_SENTENCE']


## Step 49 - Identify underrepresented characteristic-value pairs

Calculate the additional examples required to reach the
minimum coverage target for each eligible characteristic-value pair.

In [49]:
# Step 49 - Identify underrepresented characteristic-value pairs

min_examples = expansion_targets["MIN_EXAMPLES_PER_VALUE"]

coverage_targets = value_coverage.copy()

coverage_targets["ADDITIONAL_NEEDED"] = (
    min_examples - coverage_targets["N_OCCURRENCES"]
).clip(lower=0).astype(int)

coverage_targets = coverage_targets.sort_values(
    ["ADDITIONAL_NEEDED", "CHARACTERISTIC", "VALUE"],
    ascending=[False, True, True]
).reset_index(drop=True)

print("COVERAGE EXPANSION REQUIREMENTS")
print("-" * 40)

print("Eligible characteristic-value pairs:", len(coverage_targets))

print(
    "Pairs requiring additional examples:",
    (coverage_targets["ADDITIONAL_NEEDED"] > 0).sum()
)

print(
    "Total additional requirement occurrences needed:",
    coverage_targets["ADDITIONAL_NEEDED"].sum()
)

print("\nHighest priority values:")

display(
    coverage_targets.loc[
        coverage_targets["ADDITIONAL_NEEDED"] > 0,
        [
            "CHARACTERISTIC",
            "VALUE",
            "VALUE_DEFINITION",
            "N_OCCURRENCES",
            "ADDITIONAL_NEEDED"
        ]
    ].head(20)
)

COVERAGE EXPANSION REQUIREMENTS
----------------------------------------
Eligible characteristic-value pairs: 78
Pairs requiring additional examples: 70
Total additional requirement occurrences needed: 393

Highest priority values:


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,N_OCCURRENCES,ADDITIONAL_NEEDED
0,HANDLE_COLOR,V004,STAINLESS STEEL,0,10
1,POWER_CABLE,V003,EU TYPE PLUG,0,10
2,HANDLE_TYPE,V003,H3,1,9
3,KNOB_COLOR,V003,SILVER,1,9
4,OVEN_VOLUME_L,V003,75,1,9
5,POWER_CABLE,V002,W/O PLUG,1,9
6,POWER_CABLE,V004,US TYPE PLUG,1,9
7,POWER_CABLE,V005,UK TYPE PLUG,1,9
8,ENERGY_CLASS,V001,A,2,8
9,HANDLE_COLOR,V003,SILVER,2,8


## Step 50 - Prepare coverage-aware sampling weights

Increase the sampling probability of underrepresented
characteristic-value pairs while preserving the existing
characteristic selection profile.

In [50]:
# Step 50 - Prepare coverage-aware sampling weights

sampling_weights = coverage_targets.copy()

min_examples = expansion_targets["MIN_EXAMPLES_PER_VALUE"]

# Higher priority for values with fewer examples
sampling_weights["COVERAGE_WEIGHT"] = (
    1.0
    + sampling_weights["ADDITIONAL_NEEDED"] / min_examples
)

# Normalize weights within each characteristic
sampling_weights["SAMPLING_PROBABILITY"] = (
    sampling_weights["COVERAGE_WEIGHT"]
    / sampling_weights.groupby("CHARACTERISTIC")[
        "COVERAGE_WEIGHT"
    ].transform("sum")
)

# Verify probabilities
probability_check = (
    sampling_weights.groupby("CHARACTERISTIC")[
        "SAMPLING_PROBABILITY"
    ].sum()
)

assert (
    (probability_check - 1.0).abs() < 1e-9
).all()

print("Sampling probabilities prepared successfully.")

display(
    sampling_weights[
        [
            "CHARACTERISTIC",
            "VALUE",
            "VALUE_DEFINITION",
            "N_OCCURRENCES",
            "ADDITIONAL_NEEDED",
            "COVERAGE_WEIGHT",
            "SAMPLING_PROBABILITY"
        ]
    ].head(20)
)

Sampling probabilities prepared successfully.


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,N_OCCURRENCES,ADDITIONAL_NEEDED,COVERAGE_WEIGHT,SAMPLING_PROBABILITY
0,HANDLE_COLOR,V004,STAINLESS STEEL,0,10,2.0,0.294118
1,POWER_CABLE,V003,EU TYPE PLUG,0,10,2.0,0.210526
2,HANDLE_TYPE,V003,H3,1,9,1.9,0.226190
3,KNOB_COLOR,V003,SILVER,1,9,1.9,0.287879
4,OVEN_VOLUME_L,V003,75,1,9,1.9,0.267606
5,POWER_CABLE,V002,W/O PLUG,1,9,1.9,0.200000
6,POWER_CABLE,V004,US TYPE PLUG,1,9,1.9,0.200000
7,POWER_CABLE,V005,UK TYPE PLUG,1,9,1.9,0.200000
8,ENERGY_CLASS,V001,A,2,8,1.8,0.375000
9,HANDLE_COLOR,V003,SILVER,2,8,1.8,0.264706


## Step 51 - Prepare a coverage-aware value selector

Create a reusable function to sample characteristic values
using the coverage-aware probabilities.

The function returns the selected value record without
changing the existing customer request generation logic.

In [51]:
# Step 51 - Prepare a coverage-aware value selector

import numpy as np

def select_value_coverage_aware(characteristic, rng, weights_df):
    candidates = weights_df.loc[
        weights_df["CHARACTERISTIC"] == characteristic
    ]

    if candidates.empty:
        raise ValueError(
            f"No eligible values found for: {characteristic}"
        )

    probabilities = candidates[
        "SAMPLING_PROBABILITY"
    ].to_numpy(dtype=float)

    selected_position = rng.choice(
        len(candidates),
        p=probabilities
    )

    selected_value = candidates.iloc[selected_position]     #iloc to select the row at the given position

    return {
        "CHARACTERISTIC": selected_value["CHARACTERISTIC"],
        "VALUE": selected_value["VALUE"],
        "VALUE_DEFINITION": selected_value["VALUE_DEFINITION"]
    }


# Test the function
rng_test = np.random.default_rng(42)

test_characteristics = [
    "HANDLE_COLOR",
    "WIDTH_CM",
    "POWER_CABLE",
    "CLEANING"
]

test_results = []

for characteristic in test_characteristics:
    selected = select_value_coverage_aware(
        characteristic,
        rng_test,
        sampling_weights
    )

    test_results.append(selected)

display(pd.DataFrame(test_results))

,CHARACTERISTIC,VALUE,VALUE_DEFINITION
0,HANDLE_COLOR,V002,WHITE
1,WIDTH_CM,V004,90
2,POWER_CABLE,V001,W/O CABLE
3,CLEANING,V002,CATALYTIC


## Step 52 - Inspect the existing generation profile

Verify the characteristic selection weights and eligible
characteristics before integrating coverage-aware value sampling.

In [52]:
# Step 52 - Inspect the existing generation profile

print("Generation profile columns:")
print(generation_profile.columns.tolist())

print("\nEnabled characteristic count:")
print(generation_profile["ENABLED"].sum())

print("\nExcluded characteristics:")
display(
    generation_profile.loc[
        ~generation_profile["ENABLED"],
        ["CHARACTERISTIC", "ENABLED"]
    ]
)

print("\nEnabled characteristic profile:")
display(
    generation_profile.loc[
        generation_profile["ENABLED"]
    ].head(25)
)

print("\nSampling weight columns:")
print(sampling_weights.columns.tolist())

Generation profile columns:
['CHARACTERISTIC', 'DATA_TYPE', 'MATCH_METHOD', 'REQUEST_WEIGHT', 'ENABLED']

Enabled characteristic count:
23

Excluded characteristics:


,CHARACTERISTIC,ENABLED
9,OVEN_FUNCTIONS,False



Enabled characteristic profile:


,CHARACTERISTIC,DATA_TYPE,MATCH_METHOD,REQUEST_WEIGHT,ENABLED
0,WIDTH_CM,NUMERIC,EXACT,3.0,True
1,PACKAGING_TYPE,CATEGORICAL,EXACT,0.3,True
2,PANEL_TYPE,CATEGORICAL,EXACT,1.0,True
3,PANEL_COLOR,CATEGORICAL,EXACT,3.0,True
4,POWER_CABLE,CATEGORICAL,EXACT,0.3,True
5,WIFI,BOOLEAN,EXACT,1.0,True
6,TURNSPIT,BOOLEAN,EXACT,1.0,True
7,OVEN_TYPE,CATEGORICAL,EXACT,3.0,True
8,NUMBER_OF_FUNCTIONS,NUMERIC,DISTANCE,3.0,True
10,BAKING_TRAY,NUMERIC,DISTANCE,1.0,True



Sampling weight columns:
['CHARACTERISTIC', 'VALUE', 'VALUE_DEFINITION', 'N_OCCURRENCES', 'COVERED', 'ADDITIONAL_NEEDED', 'COVERAGE_WEIGHT', 'SAMPLING_PROBABILITY']


## Step 53 - Prepare the expanded request generator

Generate customer requirement profiles using the existing
characteristic weights and coverage-aware value selection.

Customer requirements are sampled independently of product
configuration rules.

In [53]:
# Step 53 - Prepare the expanded request generator

def generate_expanded_ground_truth(
    n_requests,
    start_id,
    rng,
    generation_profile,
    sampling_weights,
    min_characteristics=2,
    max_characteristics=6
):
    enabled_profile = generation_profile.loc[
        generation_profile["ENABLED"]
    ].copy()

    characteristics = enabled_profile[
        "CHARACTERISTIC"
    ].to_numpy()

    characteristic_weights = enabled_profile[
        "REQUEST_WEIGHT"
    ].to_numpy(dtype=float)

    characteristic_probabilities = (
        characteristic_weights / characteristic_weights.sum()
    )

    rows = []

    for request_number in range(
        start_id,
        start_id + n_requests
    ):
        request_id = f"R{request_number:04d}"

        n_characteristics = int(
            rng.integers(
                min_characteristics,
                max_characteristics + 1
            )
        )

        selected_characteristics = rng.choice(
            characteristics,
            size=n_characteristics,
            replace=False,
            p=characteristic_probabilities
        )

        for characteristic in selected_characteristics:

            selected_value = select_value_coverage_aware(
                characteristic,
                rng,
                sampling_weights
            )

            requirement_type = rng.choice(
                ["MUST", "PREFERRED"],
                p=[0.75, 0.25]
            )

            rows.append({
                "REQUEST_ID": request_id,
                "CHARACTERISTIC": characteristic,
                "VALUE": selected_value["VALUE"],
                "VALUE_DEFINITION": selected_value["VALUE_DEFINITION"],
                "OPERATOR": "EQUAL",
                "REQUIREMENT_TYPE": requirement_type
            })

    return pd.DataFrame(rows)


# Test with 5 new requests

rng_expansion_test = np.random.default_rng(42)

test_expanded_ground_truth = generate_expanded_ground_truth(
    n_requests=5,
    start_id=101,
    rng=rng_expansion_test,
    generation_profile=generation_profile,
    sampling_weights=sampling_weights,
    min_characteristics=expansion_targets[
        "MIN_REQUIREMENTS_PER_REQUEST"
    ],
    max_characteristics=expansion_targets[
        "MAX_REQUIREMENTS_PER_REQUEST"
    ]
)

print(
    "Generated requests:",
    test_expanded_ground_truth["REQUEST_ID"].nunique()
)

print(
    "Generated requirements:",
    len(test_expanded_ground_truth)
)

print(
    "Contains OVEN_FUNCTIONS:",
    test_expanded_ground_truth[
        "CHARACTERISTIC"
    ].eq("OVEN_FUNCTIONS").any()
)

print(
    "Duplicate characteristic assignments:",
    test_expanded_ground_truth.duplicated(
        ["REQUEST_ID", "CHARACTERISTIC"]
    ).sum()
)

display(test_expanded_ground_truth.head(20))

Generated requests: 5
Generated requirements: 18
Contains OVEN_FUNCTIONS: False
Duplicate characteristic assignments: 0


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0101,NUMBER_OF_FUNCTIONS,V003,8,EQUAL,MUST
1,R0101,CLEANING,V004,HYDROCLEAN,EQUAL,PREFERRED
2,R0102,OVEN_VOLUME_L,V002,65,EQUAL,PREFERRED
3,R0102,PANEL_COLOR,V003,SILVER,EQUAL,MUST
4,R0102,NUMBER_OF_FUNCTIONS,V002,5,EQUAL,MUST
5,R0102,OVEN_TYPE,V003,COMBINATION,EQUAL,MUST
6,R0102,OVEN_CONTROL,V003,ELECTRONIC TOUCH TIMER,EQUAL,MUST
7,R0103,CLEANING,V003,PYROLYSIS,EQUAL,MUST
8,R0103,OVEN_VOLUME_L,V003,75,EQUAL,MUST
9,R0104,HANDLE_TYPE,V003,H3,EQUAL,MUST


## Step 54 - Generate customer texts for expanded requests

Map the structured requirements to customer-friendly language
and generate customer request texts using the existing templates.

The structured ground truth remains the reference for NLP evaluation.

In [54]:
# Step 54 - Generate customer texts for expanded requests

# Attach customer-friendly language
test_ground_truth_with_language = (
    test_expanded_ground_truth
    .merge(
        language_master[
            [
                "CHARACTERISTIC",
                "VALUE",
                "CUSTOMER_LANGUAGE"
            ]
        ],
        on=["CHARACTERISTIC", "VALUE"],
        how="left",
        validate="many_to_one"
    )
)

# Check missing language mappings
missing_language = (
    test_ground_truth_with_language[
        "CUSTOMER_LANGUAGE"
    ].isna().sum()
)

print("Missing language mappings:", missing_language)

assert missing_language == 0, (
    "Some characteristic-value pairs have no customer language."
)

# Generate request texts
rng_style_test = np.random.default_rng(43)

test_customer_requests = []

for request_id, requirements in (
    test_ground_truth_with_language.groupby(
        "REQUEST_ID",
        sort=True
    )
):

    style = rng_style_test.choice(
        ["SHORT_LIST", "FULL_SENTENCE"]
    )

    request_text = generate_request_text(
        requirements,
        style
    )

    test_customer_requests.append({
        "REQUEST_ID": request_id,
        "STYLE": style,
        "REQUEST_TEXT": request_text
    })

test_customer_requests_df = pd.DataFrame(
    test_customer_requests
)

print(
    "Generated customer texts:",
    len(test_customer_requests_df)
)

display(test_customer_requests_df)

Missing language mappings: 0
Generated customer texts: 5


,REQUEST_ID,STYLE,REQUEST_TEXT
0,R0101,FULL_SENTENCE,"I'm looking for an oven with the following requirements: 8 cooking functions. Ideally, I would also like hydroclean cleaning."
1,R0102,FULL_SENTENCE,"I'm looking for an oven with the following requirements: silver control panel, 5 cooking functions, combination oven, electronic touch timer. Ideally, I would also like 65-liter oven capacity."
2,R0103,SHORT_LIST,"Required: pyrolytic cleaning, 75-liter oven capacity."
3,R0104,SHORT_LIST,"Required: H3 handle design, energy class A, steam oven, silver control panel. Preferred: no timer, 2 baking trays."
4,R0105,FULL_SENTENCE,"I'm looking for an oven with the following requirements: black control panel, 45 cm wide, catalytic cleaning."


## Step 55 - Expand structured ground truth to 1,000 requests

Generate 900 additional structured customer requests using
coverage-aware sampling.

Preserve the existing 100 pilot requests and validate the
expanded dataset before generating natural-language texts.

In [55]:
import inspect

print(inspect.signature(generate_expanded_ground_truth))

(n_requests, start_id, rng, generation_profile, sampling_weights, min_characteristics=2, max_characteristics=6)


In [56]:
# Step 55 - Expand structured ground truth to 1,000 requests

TARGET_REQUESTS = 1000

# Existing pilot dataset
existing_request_count = batch_ground_truth["REQUEST_ID"].nunique()

additional_requests = TARGET_REQUESTS - existing_request_count

print("Existing requests:", existing_request_count)
print("Additional requests needed:", additional_requests)

assert additional_requests >= 0

# Reproducible random generator
rng_expansion = np.random.default_rng(2026)

# Generate additional structured requests
if additional_requests > 0:

    new_ground_truth = generate_expanded_ground_truth(
        n_requests=additional_requests,
        start_id=existing_request_count + 1,
        rng=rng_expansion,
        generation_profile=generation_profile,
        sampling_weights=sampling_weights,
        min_characteristics=2,
        max_characteristics=6
    )

    expanded_ground_truth = pd.concat(
        [batch_ground_truth, new_ground_truth],
        ignore_index=True
    )

else:
    expanded_ground_truth = batch_ground_truth.copy()

# Basic validation
total_requests = expanded_ground_truth["REQUEST_ID"].nunique()

duplicate_assignments = expanded_ground_truth.duplicated(
    subset=["REQUEST_ID", "CHARACTERISTIC"]
).sum()

contains_excluded = expanded_ground_truth[
    "CHARACTERISTIC"
].eq("OVEN_FUNCTIONS").any()

print("\nEXPANDED DATASET SUMMARY")
print("-" * 40)
print("Total requests:", total_requests)
print("Total requirements:", len(expanded_ground_truth))
print("Duplicate characteristic assignments:", duplicate_assignments)
print("Contains OVEN_FUNCTIONS:", contains_excluded)

assert total_requests == TARGET_REQUESTS
assert duplicate_assignments == 0
assert not contains_excluded

display(expanded_ground_truth.head(15))

Existing requests: 100
Additional requests needed: 900



EXPANDED DATASET SUMMARY
----------------------------------------
Total requests: 1000
Total requirements: 3988
Duplicate characteristic assignments: 0
Contains OVEN_FUNCTIONS: False


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0001,WIDTH_CM,V001,45,EQUAL,MUST
1,R0001,OVEN_TYPE,V002,STEAM,EQUAL,MUST
2,R0002,PANEL_COLOR,V002,WHITE,EQUAL,MUST
3,R0002,PACKAGING_TYPE,V001,SHRINK,EQUAL,MUST
4,R0002,OVEN_VOLUME_L,V001,50,EQUAL,MUST
5,R0003,WIDTH_CM,V002,60,EQUAL,MUST
6,R0003,BAKING_TRAY,V003,2,EQUAL,MUST
7,R0003,PANEL_TYPE,V001,GLASS,EQUAL,MUST
8,R0003,OVEN_TYPE,V003,COMBINATION,EQUAL,MUST
9,R0003,KNOB_TYPE,V002,K2,EQUAL,MUST


## Step 56 - Validate characteristic-value coverage

Measure the frequency of each eligible characteristic-value pair
in the expanded dataset.

Identify pairs that remain below the minimum coverage target.

In [57]:
# Step 56 - Validate characteristic-value coverage

MIN_EXAMPLES_PER_VALUE = 10

# Count actual occurrences in the expanded dataset
value_counts = (
    expanded_ground_truth
    .groupby(["CHARACTERISTIC", "VALUE"])
    .size()
    .reset_index(name="N_OCCURRENCES")
)

# Include all eligible characteristic-value pairs
expanded_coverage = (
    sampling_weights[
        ["CHARACTERISTIC", "VALUE", "VALUE_DEFINITION"]
    ]
    .merge(
        value_counts,
        on=["CHARACTERISTIC", "VALUE"],
        how="left",
        validate="one_to_one"
    )
)

expanded_coverage["N_OCCURRENCES"] = (
    expanded_coverage["N_OCCURRENCES"]
    .fillna(0)
    .astype(int)
)

expanded_coverage["ADDITIONAL_NEEDED"] = (
    MIN_EXAMPLES_PER_VALUE
    - expanded_coverage["N_OCCURRENCES"]
).clip(lower=0)

expanded_coverage = expanded_coverage.sort_values(
    ["ADDITIONAL_NEEDED", "N_OCCURRENCES"],
    ascending=[False, True]
).reset_index(drop=True)

# Summary
total_pairs = len(expanded_coverage)

covered_pairs = (
    expanded_coverage["N_OCCURRENCES"] > 0
).sum()

pairs_below_target = (
    expanded_coverage["ADDITIONAL_NEEDED"] > 0
).sum()

total_additional_needed = (
    expanded_coverage["ADDITIONAL_NEEDED"].sum()
)

print("EXPANDED DATASET COVERAGE")
print("-" * 40)
print("Eligible characteristic-value pairs:", total_pairs)
print("Covered pairs:", covered_pairs)
print("Coverage:", f"{covered_pairs / total_pairs:.1%}")
print("Pairs below minimum target:", pairs_below_target)
print("Additional occurrences needed:", total_additional_needed)

print("\nPairs requiring additional examples:")

display(
    expanded_coverage.loc[
        expanded_coverage["ADDITIONAL_NEEDED"] > 0,
        [
            "CHARACTERISTIC",
            "VALUE",
            "VALUE_DEFINITION",
            "N_OCCURRENCES",
            "ADDITIONAL_NEEDED"
        ]
    ]
)

EXPANDED DATASET COVERAGE


----------------------------------------
Eligible characteristic-value pairs: 78
Covered pairs: 78
Coverage: 100.0%
Pairs below minimum target: 3
Additional occurrences needed: 10

Pairs requiring additional examples:


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,N_OCCURRENCES,ADDITIONAL_NEEDED
0,POWER_CABLE,V004,US TYPE PLUG,5,5
1,POWER_CABLE,V002,W/O PLUG,7,3
2,POWER_CABLE,V001,W/O CABLE,8,2


## Step 57 - Complete minimum characteristic-value coverage

Generate targeted additional customer requests for
characteristic-value pairs below the minimum frequency.

Preserve the existing dataset and avoid regenerating
the original 1,000 requests.

In [58]:
# Step 57 - Complete minimum characteristic-value coverage

# Identify missing occurrences
coverage_gaps = expanded_coverage.loc[
    expanded_coverage["ADDITIONAL_NEEDED"] > 0
].copy()

# Create one targeted request for each missing occurrence
targeted_rows = []

next_request_number = (
    expanded_ground_truth["REQUEST_ID"]
    .str.extract(r"R(\d+)")[0]
    .astype(int)
    .max() + 1
)

for _, gap in coverage_gaps.iterrows():

    for _ in range(int(gap["ADDITIONAL_NEEDED"])):

        request_id = f"R{next_request_number:04d}"
        next_request_number += 1

        targeted_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": gap["CHARACTERISTIC"],
            "VALUE": gap["VALUE"],
            "VALUE_DEFINITION": gap["VALUE_DEFINITION"],
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": "MUST"
        })

targeted_ground_truth = pd.DataFrame(targeted_rows)

# Add targeted requirements to existing ground truth
final_ground_truth = pd.concat(
    [expanded_ground_truth, targeted_ground_truth],
    ignore_index=True
)

print("Original requests:",
      expanded_ground_truth["REQUEST_ID"].nunique())

print("Targeted requests:",
      targeted_ground_truth["REQUEST_ID"].nunique())

print("Final requests:",
      final_ground_truth["REQUEST_ID"].nunique())

print("Final requirements:",
      len(final_ground_truth))

display(targeted_ground_truth)

Original requests: 1000
Targeted requests: 10
Final requests: 1010
Final requirements: 3998


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R1001,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
1,R1002,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
2,R1003,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
3,R1004,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
4,R1005,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
5,R1006,POWER_CABLE,V002,W/O PLUG,EQUAL,MUST
6,R1007,POWER_CABLE,V002,W/O PLUG,EQUAL,MUST
7,R1008,POWER_CABLE,V002,W/O PLUG,EQUAL,MUST
8,R1009,POWER_CABLE,V001,W/O CABLE,EQUAL,MUST
9,R1010,POWER_CABLE,V001,W/O CABLE,EQUAL,MUST


## Step 58 - Complete targeted customer requests

Add complementary requirements to the 10 targeted requests
while preserving their required power cable values.

Each request will contain 2 to 6 distinct characteristics.

In [59]:
# Step 58 - Complete targeted customer requests

rng_targeted = np.random.default_rng(58)

# Existing requirements, excluding POWER_CABLE
candidate_requirements = expanded_ground_truth.loc[
    expanded_ground_truth["CHARACTERISTIC"] != "POWER_CABLE"
].copy()

# One candidate per characteristic-value combination
candidate_requirements = candidate_requirements.drop_duplicates(
    subset=["CHARACTERISTIC", "VALUE"]
)

additional_rows = []

for request_id in targeted_ground_truth["REQUEST_ID"].unique():

    # Final number of requirements: 2 to 6
    target_count = int(rng_targeted.integers(2, 7))

    # POWER_CABLE is already assigned
    n_additional = target_count - 1

    # Select distinct characteristics
    available_characteristics = (
        candidate_requirements["CHARACTERISTIC"]
        .drop_duplicates()
        .to_numpy()
    )

    selected_characteristics = rng_targeted.choice(
        available_characteristics,
        size=n_additional,
        replace=False
    )

    for characteristic in selected_characteristics:

        options = candidate_requirements.loc[
            candidate_requirements["CHARACTERISTIC"] == characteristic
        ]

        selected = options.iloc[
            int(rng_targeted.integers(len(options)))
        ]

        additional_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": selected["CHARACTERISTIC"],
            "VALUE": selected["VALUE"],
            "VALUE_DEFINITION": selected["VALUE_DEFINITION"],
            "OPERATOR": selected["OPERATOR"],
            "REQUIREMENT_TYPE": selected["REQUIREMENT_TYPE"]
        })

additional_ground_truth = pd.DataFrame(additional_rows)

final_ground_truth = pd.concat(
    [final_ground_truth, additional_ground_truth],
    ignore_index=True
)

# Validate targeted requests
targeted_ids = targeted_ground_truth["REQUEST_ID"].unique()

targeted_completed = final_ground_truth.loc[
    final_ground_truth["REQUEST_ID"].isin(targeted_ids)
]

counts = targeted_completed.groupby("REQUEST_ID").size()

duplicates = targeted_completed.duplicated(
    subset=["REQUEST_ID", "CHARACTERISTIC"]
).sum()

assert counts.between(2, 6).all()
assert duplicates == 0

print("Total requests:", final_ground_truth["REQUEST_ID"].nunique())
print("Total requirements:", len(final_ground_truth))
print("Requirements per targeted request:")
display(counts.to_frame("N_REQUIREMENTS"))

display(targeted_completed.head(20))

Total requests: 1010
Total requirements: 4028
Requirements per targeted request:


,N_REQUIREMENTS
REQUEST_ID,
R1001,4
R1002,6
R1003,6
R1004,3
R1005,5
R1006,2
R1007,5
R1008,2
R1009,2


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
3988,R1001,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
3989,R1002,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
3990,R1003,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
3991,R1004,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
3992,R1005,POWER_CABLE,V004,US TYPE PLUG,EQUAL,MUST
3993,R1006,POWER_CABLE,V002,W/O PLUG,EQUAL,MUST
3994,R1007,POWER_CABLE,V002,W/O PLUG,EQUAL,MUST
3995,R1008,POWER_CABLE,V002,W/O PLUG,EQUAL,MUST
3996,R1009,POWER_CABLE,V001,W/O CABLE,EQUAL,MUST
3997,R1010,POWER_CABLE,V001,W/O CABLE,EQUAL,MUST


## Step 59 - Final ground truth quality control

Validate request counts, requirement counts, duplicate assignments,
excluded characteristics, and characteristic-value coverage.

In [60]:
# Step 59 - Final ground truth quality control

# 1. Request and requirement counts
n_requests = final_ground_truth["REQUEST_ID"].nunique()
n_requirements = len(final_ground_truth)

# 2. Requirements per request
requirements_per_request = (
    final_ground_truth.groupby("REQUEST_ID")
    .size()
)

# 3. Duplicate characteristic assignments
duplicate_assignments = final_ground_truth.duplicated(
    subset=["REQUEST_ID", "CHARACTERISTIC"]
).sum()

# 4. Excluded characteristics
excluded_found = final_ground_truth[
    "CHARACTERISTIC"
].isin(excluded_characteristics).sum()

# 5. Characteristic-value coverage
final_pair_counts = (
    final_ground_truth
    .groupby(["CHARACTERISTIC", "VALUE"])
    .size()
    .reset_index(name="N_OCCURRENCES")
)

final_coverage = (
    sampling_weights[
        ["CHARACTERISTIC", "VALUE", "VALUE_DEFINITION"]
    ]
    .drop_duplicates()
    .merge(
        final_pair_counts,
        on=["CHARACTERISTIC", "VALUE"],
        how="left"
    )
)

final_coverage["N_OCCURRENCES"] = (
    final_coverage["N_OCCURRENCES"]
    .fillna(0)
    .astype(int)
)

min_examples = expansion_targets["MIN_EXAMPLES_PER_VALUE"]

below_target = final_coverage[
    final_coverage["N_OCCURRENCES"] < min_examples
]

print("FINAL GROUND TRUTH QUALITY CONTROL")
print("-" * 45)

print("Total requests:", n_requests)
print("Total requirements:", n_requirements)
print("Minimum requirements per request:", requirements_per_request.min())
print("Maximum requirements per request:", requirements_per_request.max())
print("Duplicate characteristic assignments:", duplicate_assignments)
print("Excluded characteristic occurrences:", excluded_found)

print("\nCHARACTERISTIC-VALUE COVERAGE")
print("Eligible pairs:", len(final_coverage))
print("Covered pairs:", (final_coverage["N_OCCURRENCES"] > 0).sum())
print("Pairs below minimum target:", len(below_target))
print("Minimum examples per value:", final_coverage["N_OCCURRENCES"].min())

assert n_requests == 1010
assert requirements_per_request.between(2, 6).all()
assert duplicate_assignments == 0
assert excluded_found == 0
assert len(below_target) == 0

print("\nAll structural and coverage checks passed.")

FINAL GROUND TRUTH QUALITY CONTROL
---------------------------------------------
Total requests: 1010
Total requirements: 4028
Minimum requirements per request: 2
Maximum requirements per request: 6
Duplicate characteristic assignments: 0
Excluded characteristic occurrences: 0

CHARACTERISTIC-VALUE COVERAGE
Eligible pairs: 78
Covered pairs: 78
Pairs below minimum target: 0
Minimum examples per value: 10

All structural and coverage checks passed.


## Step 60 - Generate customer request texts for the final dataset

Generate initial customer request texts from the validated
structured ground truth using the existing language mappings
and request composition function.

These texts will serve as the baseline for subsequent
LLM-assisted natural language diversification.

In [61]:
# Step 60 - Generate customer request texts for the final dataset

rng_final_text = np.random.default_rng(60)

# Add customer language mappings
final_text_data = final_ground_truth.merge(
    language_master[
        ["CHARACTERISTIC", "VALUE", "CUSTOMER_LANGUAGE"]
    ],
    on=["CHARACTERISTIC", "VALUE"],
    how="left",
    validate="many_to_one"
)

# Verify that every requirement has a language mapping
missing_language = final_text_data["CUSTOMER_LANGUAGE"].isna().sum()

assert missing_language == 0, (
    f"Missing customer language mappings: {missing_language}"
)

# Generate one text for each request
final_customer_requests_list = []

for request_id, requirements in final_text_data.groupby(
    "REQUEST_ID", sort=True
):

    style = rng_final_text.choice(
        ["SHORT_LIST", "FULL_SENTENCE"]
    )

    request_text = generate_request_text(
        requirements,
        style
    )

    final_customer_requests_list.append({
        "REQUEST_ID": request_id,
        "STYLE": style,
        "REQUEST_TEXT": request_text
    })

final_customer_requests = pd.DataFrame(
    final_customer_requests_list
)

# Validate generated texts
assert len(final_customer_requests) == 1010
assert final_customer_requests["REQUEST_ID"].is_unique
assert final_customer_requests["REQUEST_TEXT"].notna().all()
assert final_customer_requests["REQUEST_TEXT"].str.strip().ne("").all()

print("Generated customer requests:", len(final_customer_requests))

print("\nWriting styles:")
print(final_customer_requests["STYLE"].value_counts())

print("\nSample customer requests:")
display(final_customer_requests.head(10))

Generated customer requests: 1010

Writing styles:
STYLE
SHORT_LIST       513
FULL_SENTENCE    497
Name: count, dtype: int64

Sample customer requests:


,REQUEST_ID,STYLE,REQUEST_TEXT
0,R0001,SHORT_LIST,"Required: 45 cm wide, steam oven."
1,R0002,SHORT_LIST,"Required: white control panel, shrink packaging, 50-liter oven capacity."
2,R0003,SHORT_LIST,"Required: 60 cm wide, 2 baking trays, glass control panel, combination oven, K2 control knob design. Preferred: silver control panel."
3,R0004,SHORT_LIST,"Required: H2 handle design, 8 cooking functions. Preferred: black oven handle."
4,R0005,FULL_SENTENCE,"I'm looking for an oven with the following requirements: 1 telescopic rail set, 90 cm wide, pyrolytic cleaning, black control knobs. Ideally, I would also like rotary control knobs."
5,R0006,SHORT_LIST,"Required: stainless steel control panel, 70 cm wide. Preferred: rotisserie turnspit included, 75-liter oven capacity."
6,R0007,FULL_SENTENCE,"I'm looking for an oven with the following requirements: shrink packaging, energy class A, stainless steel control knobs. Ideally, I would also like catalytic cleaning, 0 roasting trays."
7,R0008,FULL_SENTENCE,"I'm looking for an oven with the following requirements: metal control panel, digital timer, 70 cm wide, combination oven, white control knobs, catalytic cleaning."
8,R0009,FULL_SENTENCE,"I'm looking for an oven with the following requirements: 1 wire shelf, white oven handle, 45 cm wide. Ideally, I would also like rotisserie turnspit included."
9,R0010,FULL_SENTENCE,"I'm looking for an oven with the following requirements: digital timer, steam oven. Ideally, I would also like 65-liter oven capacity, 2 telescopic rail sets."


## Step 61 - Prepare LLM rewriting input

Prepare customer requests and their structured requirements
for controlled LLM-assisted paraphrasing.

The LLM must preserve all characteristics, values,
operators, and requirement types.

In [62]:
# Step 61 - Prepare LLM rewriting input

import json

# Combine structured requirements by request
requirements_by_request = {}

for request_id, group in final_ground_truth.groupby("REQUEST_ID"):

    requirements_by_request[request_id] = group[
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ].to_dict(orient="records")

# Prepare a small pilot batch
llm_pilot_requests = []

for row in final_customer_requests.head(5).itertuples(index=False):

    llm_pilot_requests.append({
        "REQUEST_ID": row.REQUEST_ID,
        "ORIGINAL_TEXT": row.REQUEST_TEXT,
        "REQUIREMENTS": requirements_by_request[row.REQUEST_ID]
    })

print("LLM pilot requests:", len(llm_pilot_requests))

print(
    json.dumps(
        llm_pilot_requests[0],
        indent=2,
        ensure_ascii=False,
        default=str
    )
)

LLM pilot requests: 5
{
  "REQUEST_ID": "R0001",
  "ORIGINAL_TEXT": "Required: 45 cm wide, steam oven.",
  "REQUIREMENTS": [
    {
      "CHARACTERISTIC": "WIDTH_CM",
      "VALUE_DEFINITION": 45,
      "OPERATOR": "EQUAL",
      "REQUIREMENT_TYPE": "MUST"
    },
    {
      "CHARACTERISTIC": "OVEN_TYPE",
      "VALUE_DEFINITION": "STEAM",
      "OPERATOR": "EQUAL",
      "REQUIREMENT_TYPE": "MUST"
    }
  ]
}


## Step 62 - Define the LLM paraphrasing prompt

Create a controlled prompt for generating natural customer language
while preserving the structured ground truth.

In [63]:
# Step 62 - Define the LLM paraphrasing prompt

LLM_SYSTEM_PROMPT = """
You generate realistic English customer requests for an oven product
recommendation system.

Your task is to rewrite a synthetic customer request into natural,
varied English while preserving its exact technical meaning.

Rules:
1. Preserve every requirement in the structured REQUIREMENTS list.
2. Do not add, remove, or change any product requirement.
3. Preserve the distinction between MUST and PREFERRED.
4. Preserve numerical values, units, and comparison operators.
5. Do not invent additional product features or constraints.
6. Use natural customer language, not technical database field names.
7. Vary sentence structure, vocabulary, and writing style.
8. Avoid repetitive openings and fixed templates.
9. You may use complete sentences, short messages, or informal lists.
10. Return only the rewritten customer request text.

The structured REQUIREMENTS list is the authoritative source.
"""

def build_llm_prompt(request_data):

    return (
        "Rewrite this customer request into natural English.\n\n"
        + json.dumps(
            request_data,
            indent=2,
            ensure_ascii=False,
            default=str
        )
    )


print("SYSTEM PROMPT:")
print(LLM_SYSTEM_PROMPT)

print("\nUSER PROMPT EXAMPLE:")
print(build_llm_prompt(llm_pilot_requests[0]))

SYSTEM PROMPT:

You generate realistic English customer requests for an oven product
recommendation system.

Your task is to rewrite a synthetic customer request into natural,
varied English while preserving its exact technical meaning.

Rules:
1. Preserve every requirement in the structured REQUIREMENTS list.
2. Do not add, remove, or change any product requirement.
3. Preserve the distinction between MUST and PREFERRED.
4. Preserve numerical values, units, and comparison operators.
5. Do not invent additional product features or constraints.
6. Use natural customer language, not technical database field names.
7. Vary sentence structure, vocabulary, and writing style.
8. Avoid repetitive openings and fixed templates.
9. You may use complete sentences, short messages, or informal lists.
10. Return only the rewritten customer request text.

The structured REQUIREMENTS list is the authoritative source.


USER PROMPT EXAMPLE:
Rewrite this customer request into natural English.

{
  "REQU

## Step 64 - Prepare LLM output and validation structure

Create a separate dataset for LLM-generated customer language.

Keep the original request and structured ground truth unchanged.
Track the generated text, validation status, and review notes.

In [64]:
# Step 64 - Prepare LLM output and validation structure

llm_output_columns = [
    "REQUEST_ID",
    "ORIGINAL_TEXT",
    "LLM_TEXT",
    "VALIDATION_STATUS",
    "REVIEW_NOTES"
]

llm_pilot_output = pd.DataFrame(
    [
        {
            "REQUEST_ID": item["REQUEST_ID"],
            "ORIGINAL_TEXT": item["ORIGINAL_TEXT"],
            "LLM_TEXT": None,
            "VALIDATION_STATUS": "PENDING",
            "REVIEW_NOTES": None
        }
        for item in llm_pilot_requests
    ],
    columns=llm_output_columns
)

assert len(llm_pilot_output) == 5
assert llm_pilot_output["REQUEST_ID"].is_unique

print("LLM pilot records:", len(llm_pilot_output))
print("Pending validation:", (
    llm_pilot_output["VALIDATION_STATUS"] == "PENDING"
).sum())

display(llm_pilot_output)

LLM pilot records: 5
Pending validation: 5


,REQUEST_ID,ORIGINAL_TEXT,LLM_TEXT,VALIDATION_STATUS,REVIEW_NOTES
0,R0001,"Required: 45 cm wide, steam oven.",None,PENDING,None
1,R0002,"Required: white control panel, shrink packaging, 50-liter oven capacity.",None,PENDING,None
2,R0003,"Required: 60 cm wide, 2 baking trays, glass control panel, combination oven, K2 control knob design. Preferred: silver control panel.",None,PENDING,None
3,R0004,"Required: H2 handle design, 8 cooking functions. Preferred: black oven handle.",None,PENDING,None
4,R0005,"I'm looking for an oven with the following requirements: 1 telescopic rail set, 90 cm wide, pyrolytic cleaning, black control knobs. Ideally, I would also like rotary control knobs.",None,PENDING,None


In [65]:
# Step 65 - Export the first five requests for LLM rewriting

pilot_payload = {
    "system_prompt": LLM_SYSTEM_PROMPT.strip(),
    "requests": llm_pilot_requests
}

pilot_json = json.dumps(
    pilot_payload,
    ensure_ascii=False,
    indent=2,
    default=str
)

from pathlib import Path

pilot_path = Path("../data/interim/llm_pilot_input.json")
pilot_path.parent.mkdir(parents=True, exist_ok=True)

pilot_path.write_text(
    pilot_json,
    encoding="utf-8"
)

print("Pilot file saved:", pilot_path)
print("Number of requests:", len(llm_pilot_requests))

Pilot file saved: ../data/interim/llm_pilot_input.json
Number of requests: 5


## Step 66 - Import LLM-generated customer requests

Load the LLM-generated paraphrases for the five pilot requests.
Verify request IDs and preserve the original ground truth.

In [66]:
# Step 66 - Import LLM-generated customer requests

import pandas as pd
from pathlib import Path


llm_output_path = Path("../data/interim/llm_pilot_output.csv")

llm_pilot_output = pd.read_csv(
    llm_output_path,
    encoding="utf-8-sig"
)

expected_ids = {
    item["REQUEST_ID"]
    for item in llm_pilot_requests
}

actual_ids = set(llm_pilot_output["REQUEST_ID"])

assert actual_ids == expected_ids
assert llm_pilot_output["REQUEST_ID"].is_unique
assert llm_pilot_output["LLM_TEXT"].notna().all()

print("Imported LLM requests:", len(llm_pilot_output))
print("Request IDs validated:", actual_ids == expected_ids)

display(llm_pilot_output)

Imported LLM requests: 5
Request IDs validated: True


,REQUEST_ID,ORIGINAL_TEXT,LLM_TEXT,VALIDATION_STATUS,REVIEW_NOTES
0,R0001,"Required: 45 cm wide, steam oven.",I'm after a steam oven that's exactly 45 cm wide. Both of those are essential for me.,PENDING,Pending independent semantic audit
1,R0002,"Required: white control panel, shrink packaging, 50-liter oven capacity.",Could you find me an oven with a 50-litre capacity and a white control panel? It also needs to come in shrink-wrap packaging.,PENDING,Pending independent semantic audit
2,R0003,"Required: 60 cm wide, 2 baking trays, glass control panel, combination oven, K2 control knob design. Preferred: silver control panel.","I need a 60 cm combination oven with a glass control panel, two baking trays and K2-style control knobs. A silver finish on the panel would be nice, but it's not essential.",PENDING,Pending independent semantic audit
3,R0004,"Required: H2 handle design, 8 cooking functions. Preferred: black oven handle.","The oven must have the H2 handle design and exactly eight cooking functions. I'd prefer the handle in black, although that's optional.",PENDING,Pending independent semantic audit
4,R0005,"I'm looking for an oven with the following requirements: 1 telescopic rail set, 90 cm wide, pyrolytic cleaning, black control knobs. Ideally, I would also like rotary control knobs.","I'm looking for a 90 cm oven with pyrolytic cleaning, black control knobs and one telescopic rail set. Rotary knobs would be a bonus, but they're not a deal-breaker.",PENDING,Pending independent semantic audit


## Step 67 - Review semantic consistency of LLM-generated requests

Compare LLM-generated customer texts with the original structured
requirements. Check characteristic values, numerical constraints,
and MUST/PREFERRED distinctions.

Keep the original ground truth unchanged.

In [67]:
# Step 67 - Prepare semantic validation review

semantic_review = llm_pilot_output[
    ["REQUEST_ID", "ORIGINAL_TEXT", "LLM_TEXT"]
].copy()

semantic_review["REQUIREMENTS_PRESERVED"] = pd.NA
semantic_review["MUST_PREFERRED_PRESERVED"] = pd.NA
semantic_review["NO_ADDED_REQUIREMENTS"] = pd.NA
semantic_review["REVIEW_STATUS"] = "PENDING"
semantic_review["REVIEW_NOTES"] = ""

print("Requests awaiting semantic review:", len(semantic_review))

display(semantic_review)

Requests awaiting semantic review: 5


,REQUEST_ID,ORIGINAL_TEXT,LLM_TEXT,REQUIREMENTS_PRESERVED,MUST_PREFERRED_PRESERVED,NO_ADDED_REQUIREMENTS,REVIEW_STATUS,REVIEW_NOTES
0,R0001,"Required: 45 cm wide, steam oven.",I'm after a steam oven that's exactly 45 cm wide. Both of those are essential for me.,<NA>,<NA>,<NA>,PENDING,
1,R0002,"Required: white control panel, shrink packaging, 50-liter oven capacity.",Could you find me an oven with a 50-litre capacity and a white control panel? It also needs to come in shrink-wrap packaging.,<NA>,<NA>,<NA>,PENDING,
2,R0003,"Required: 60 cm wide, 2 baking trays, glass control panel, combination oven, K2 control knob design. Preferred: silver control panel.","I need a 60 cm combination oven with a glass control panel, two baking trays and K2-style control knobs. A silver finish on the panel would be nice, but it's not essential.",<NA>,<NA>,<NA>,PENDING,
3,R0004,"Required: H2 handle design, 8 cooking functions. Preferred: black oven handle.","The oven must have the H2 handle design and exactly eight cooking functions. I'd prefer the handle in black, although that's optional.",<NA>,<NA>,<NA>,PENDING,
4,R0005,"I'm looking for an oven with the following requirements: 1 telescopic rail set, 90 cm wide, pyrolytic cleaning, black control knobs. Ideally, I would also like rotary control knobs.","I'm looking for a 90 cm oven with pyrolytic cleaning, black control knobs and one telescopic rail set. Rotary knobs would be a bonus, but they're not a deal-breaker.",<NA>,<NA>,<NA>,PENDING,


In [68]:
# Step 68 - Record semantic validation results

review_results = {
    "R0001": {
        "STATUS": "PASS",
        "NOTES": "Width and steam oven requirements preserved."
    },
    "R0002": {
        "STATUS": "PASS",
        "NOTES": "Panel color, packaging and capacity preserved."
    },
    "R0003": {
        "STATUS": "PASS",
        "NOTES": (
            "All requirements preserved. "
            "Glass panel and silver finish compatibility "
            "requires separate technical validation."
        )
    },
    "R0004": {
        "STATUS": "PASS",
        "NOTES": "Mandatory handle design and function count preserved."
    },
    "R0005": {
        "STATUS": "PASS",
        "NOTES": "Mandatory requirements and rotary knob preference preserved."
    }
}

for request_id, result in review_results.items():

    mask = semantic_review["REQUEST_ID"] == request_id

    semantic_review.loc[
        mask, "REQUIREMENTS_PRESERVED"
    ] = True

    semantic_review.loc[
        mask, "MUST_PREFERRED_PRESERVED"
    ] = True

    semantic_review.loc[
        mask, "NO_ADDED_REQUIREMENTS"
    ] = True

    semantic_review.loc[
        mask, "REVIEW_STATUS"
    ] = result["STATUS"]

    semantic_review.loc[
        mask, "REVIEW_NOTES"
    ] = result["NOTES"]

print("SEMANTIC VALIDATION SUMMARY")
print("-" * 40)

print(
    semantic_review["REVIEW_STATUS"].value_counts()
)

display(
    semantic_review[
        [
            "REQUEST_ID",
            "REQUIREMENTS_PRESERVED",
            "MUST_PREFERRED_PRESERVED",
            "NO_ADDED_REQUIREMENTS",
            "REVIEW_STATUS",
            "REVIEW_NOTES"
        ]
    ]
)

SEMANTIC VALIDATION SUMMARY
----------------------------------------
REVIEW_STATUS
PASS    5
Name: count, dtype: int64


,REQUEST_ID,REQUIREMENTS_PRESERVED,MUST_PREFERRED_PRESERVED,NO_ADDED_REQUIREMENTS,REVIEW_STATUS,REVIEW_NOTES
0,R0001,True,True,True,PASS,Width and steam oven requirements preserved.
1,R0002,True,True,True,PASS,"Panel color, packaging and capacity preserved."
2,R0003,True,True,True,PASS,All requirements preserved. Glass panel and silver finish compatibility requires separate technical validation.
3,R0004,True,True,True,PASS,Mandatory handle design and function count preserved.
4,R0005,True,True,True,PASS,Mandatory requirements and rotary knob preference preserved.


In [69]:

# Step 69 - Prepare controlled semantic validation test cases

import pandas as pd

semantic_test_ground_truth = [
    {
        "CHARACTERISTIC": "WIDTH_CM",
        "VALUE_DEFINITION": 60,
        "OPERATOR": "EQUAL",
        "REQUIREMENT_TYPE": "MUST"
    },
    {
        "CHARACTERISTIC": "CLEANING",
        "VALUE_DEFINITION": "PYROLYSIS",
        "OPERATOR": "EQUAL",
        "REQUIREMENT_TYPE": "MUST"
    },
    {
        "CHARACTERISTIC": "PANEL_COLOR",
        "VALUE_DEFINITION": "BLACK",
        "OPERATOR": "EQUAL",
        "REQUIREMENT_TYPE": "PREFERRED"
    }
]

semantic_test_cases = [
    {
        "TEST_ID": "S01",
        "SCENARIO": "Correct paraphrase",
        "CUSTOMER_TEXT": (
            "I need a 60 cm oven with pyrolytic cleaning. "
            "A black control panel would be nice, "
            "but it isn't essential."
        ),
        "EXPECTED_RESULT": "PASS"
    },
    {
        "TEST_ID": "S02",
        "SCENARIO": "Changed numerical value",
        "CUSTOMER_TEXT": (
            "I'm looking for a 90 cm oven with pyrolytic "
            "cleaning. A black panel would be a bonus."
        ),
        "EXPECTED_RESULT": "FAIL"
    },
    {
        "TEST_ID": "S03",
        "SCENARIO": "Preference changed to mandatory",
        "CUSTOMER_TEXT": (
            "I need a 60 cm oven with pyrolytic cleaning, "
            "and the control panel must be black."
        ),
        "EXPECTED_RESULT": "FAIL"
    },
    {
        "TEST_ID": "S04",
        "SCENARIO": "Additional requirement",
        "CUSTOMER_TEXT": (
            "I need a 60 cm oven with pyrolytic cleaning "
            "and Wi-Fi connectivity. A black panel "
            "would be nice."
        ),
        "EXPECTED_RESULT": "FAIL"
    }
]

semantic_test_df = pd.DataFrame(semantic_test_cases)

print("Controlled test cases:", len(semantic_test_df))
print("Ground truth requirements:", len(semantic_test_ground_truth))

display(semantic_test_df)


Controlled test cases: 4
Ground truth requirements: 3


,TEST_ID,SCENARIO,CUSTOMER_TEXT,EXPECTED_RESULT
0,S01,Correct paraphrase,"I need a 60 cm oven with pyrolytic cleaning. A black control panel would be nice, but it isn't essential.",PASS
1,S02,Changed numerical value,I'm looking for a 90 cm oven with pyrolytic cleaning. A black panel would be a bonus.,FAIL
2,S03,Preference changed to mandatory,"I need a 60 cm oven with pyrolytic cleaning, and the control panel must be black.",FAIL
3,S04,Additional requirement,I need a 60 cm oven with pyrolytic cleaning and Wi-Fi connectivity. A black panel would be nice.,FAIL


### Step 70 - Prepare an independent LLM requirement extraction prompt

In [70]:

# Step 70 - Prepare an independent LLM extraction prompt

import json

validation_system_prompt = """
You are an independent requirement extraction assistant
for an oven product recommendation system.

Extract only the product requirements explicitly stated
in the customer text.

Rules:
1. Do not use or infer the original ground truth.
2. Do not add requirements that are not expressed.
3. Preserve numerical values and units.
4. Distinguish mandatory requirements (MUST) from
   optional preferences (PREFERRED).
5. Identify the intended product characteristic.
6. Preserve comparison operators such as EQUAL,
   MIN, and MAX when expressed.
7. Do not assess technical feasibility.
8. Return valid JSON only.

Return this structure:
{
  "REQUIREMENTS": [
    {
      "CHARACTERISTIC": "...",
      "VALUE_DEFINITION": "...",
      "OPERATOR": "EQUAL",
      "REQUIREMENT_TYPE": "MUST"
    }
  ]
}
"""

def build_validation_prompt(customer_text):
    return (
        "Extract the requirements from this customer text:\n\n"
        + customer_text
    )

print("SYSTEM PROMPT:")
print(validation_system_prompt)

print("\nEXAMPLE USER PROMPT:")
print(
    build_validation_prompt(
        semantic_test_df.loc[0, "CUSTOMER_TEXT"]
    )
)


SYSTEM PROMPT:

You are an independent requirement extraction assistant
for an oven product recommendation system.

Extract only the product requirements explicitly stated
in the customer text.

Rules:
1. Do not use or infer the original ground truth.
2. Do not add requirements that are not expressed.
3. Preserve numerical values and units.
4. Distinguish mandatory requirements (MUST) from
   optional preferences (PREFERRED).
5. Identify the intended product characteristic.
6. Preserve comparison operators such as EQUAL,
   MIN, and MAX when expressed.
7. Do not assess technical feasibility.
8. Return valid JSON only.

Return this structure:
{
  "REQUIREMENTS": [
    {
      "CHARACTERISTIC": "...",
      "VALUE_DEFINITION": "...",
      "OPERATOR": "EQUAL",
      "REQUIREMENT_TYPE": "MUST"
    }
  ]
}


EXAMPLE USER PROMPT:
Extract the requirements from this customer text:

I need a 60 cm oven with pyrolytic cleaning. A black control panel would be nice, but it isn't essential.


### Step 71 - Export controlled test cases for LLM extraction

In [71]:

# Step 71 - Export controlled test cases for LLM extraction

import json
from pathlib import Path

validation_inputs = []

for _, row in semantic_test_df.iterrows():
    validation_inputs.append({
        "TEST_ID": row["TEST_ID"],
        "CUSTOMER_TEXT": row["CUSTOMER_TEXT"],
        "USER_PROMPT": build_validation_prompt(
            row["CUSTOMER_TEXT"]
        )
    })

validation_path = Path(
    "../data/interim/semantic_validation_inputs.json"
)

validation_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(
        validation_inputs,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Validation cases exported:", len(validation_inputs))
print("Saved to:", validation_path)

for item in validation_inputs:
    print(
        item["TEST_ID"],
        "-",
        item["CUSTOMER_TEXT"]
    )


Validation cases exported: 4
Saved to: ../data/interim/semantic_validation_inputs.json
S01 - I need a 60 cm oven with pyrolytic cleaning. A black control panel would be nice, but it isn't essential.
S02 - I'm looking for a 90 cm oven with pyrolytic cleaning. A black panel would be a bonus.
S03 - I need a 60 cm oven with pyrolytic cleaning, and the control panel must be black.
S04 - I need a 60 cm oven with pyrolytic cleaning and Wi-Fi connectivity. A black panel would be nice.


In [72]:

# Step 72 - Prepare reference extraction outputs
# These are manually prepared test fixtures,
# not independent LLM API responses.

base = [
    {
        "CHARACTERISTIC": "WIDTH_CM",
        "VALUE_DEFINITION": 60,
        "OPERATOR": "EQUAL",
        "REQUIREMENT_TYPE": "MUST"
    },
    {
        "CHARACTERISTIC": "CLEANING",
        "VALUE_DEFINITION": "PYROLYSIS",
        "OPERATOR": "EQUAL",
        "REQUIREMENT_TYPE": "MUST"
    },
    {
        "CHARACTERISTIC": "PANEL_COLOR",
        "VALUE_DEFINITION": "BLACK",
        "OPERATOR": "EQUAL",
        "REQUIREMENT_TYPE": "PREFERRED"
    }
]

import copy

extraction_outputs = {}

for test_id in ["S01", "S02", "S03", "S04"]:
    extraction_outputs[test_id] = {
        "REQUIREMENTS": copy.deepcopy(base)
    }

# S02: Numerical value changed
extraction_outputs["S02"]["REQUIREMENTS"][0][
    "VALUE_DEFINITION"
] = 90

# S03: Preference changed to mandatory
extraction_outputs["S03"]["REQUIREMENTS"][2][
    "REQUIREMENT_TYPE"
] = "MUST"

# S04: Additional Wi-Fi requirement
extraction_outputs["S04"]["REQUIREMENTS"].append({
    "CHARACTERISTIC": "WIFI",
    "VALUE_DEFINITION": "YES",
    "OPERATOR": "EQUAL",
    "REQUIREMENT_TYPE": "MUST"
})

for test_id, result in extraction_outputs.items():
    print(test_id, "-", len(result["REQUIREMENTS"]), "requirements")


S01 - 3 requirements
S02 - 3 requirements
S03 - 3 requirements
S04 - 4 requirements


### Step 73 - Compare extracted requirements with ground truth

In [73]:

# Step 73 - Compare extracted requirements with ground truth

import pandas as pd

comparison_fields = [
    "CHARACTERISTIC",
    "VALUE_DEFINITION",
    "OPERATOR",
    "REQUIREMENT_TYPE"
]

def normalize_value(value):
    return str(value).strip().upper()

def compare_requirements(ground_truth, extracted):
    expected = {
        normalize_value(item["CHARACTERISTIC"]): item
        for item in ground_truth
    }

    actual = {
        normalize_value(item["CHARACTERISTIC"]): item
        for item in extracted
    }

    differences = []

    for characteristic in sorted(set(expected) | set(actual)):
        if characteristic not in actual:
            differences.append({
                "CHARACTERISTIC": characteristic,
                "ERROR_TYPE": "MISSING_REQUIREMENT"
            })
            continue

        if characteristic not in expected:
            differences.append({
                "CHARACTERISTIC": characteristic,
                "ERROR_TYPE": "ADDED_REQUIREMENT"
            })
            continue

        for field in comparison_fields[1:]:
            expected_value = normalize_value(
                expected[characteristic][field]
            )
            actual_value = normalize_value(
                actual[characteristic][field]
            )

            if expected_value != actual_value:
                differences.append({
                    "CHARACTERISTIC": characteristic,
                    "ERROR_TYPE": f"{field}_MISMATCH",
                    "EXPECTED": expected_value,
                    "ACTUAL": actual_value
                })

    return differences


comparison_results = []
comparison_details = []

for test_id, output in extraction_outputs.items():
    differences = compare_requirements(
        semantic_test_ground_truth,
        output["REQUIREMENTS"]
    )

    comparison_results.append({
        "TEST_ID": test_id,
        "PREDICTED_RESULT": "PASS" if not differences else "FAIL",
        "N_ERRORS": len(differences)
    })

    for difference in differences:
        comparison_details.append({
            "TEST_ID": test_id,
            **difference
        })

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.merge(
    semantic_test_df[["TEST_ID", "EXPECTED_RESULT"]],
    on="TEST_ID",
    how="left"
)

comparison_df["CORRECT"] = (
    comparison_df["PREDICTED_RESULT"]
    == comparison_df["EXPECTED_RESULT"]
)

print("SEMANTIC COMPARISON SUMMARY")
display(comparison_df)

print("\nDETECTED DIFFERENCES")
display(pd.DataFrame(comparison_details))

print(
    "\nTest accuracy:",
    f"{comparison_df['CORRECT'].mean():.1%}"
)


SEMANTIC COMPARISON SUMMARY


,TEST_ID,PREDICTED_RESULT,N_ERRORS,EXPECTED_RESULT,CORRECT
0,S01,PASS,0,PASS,True
1,S02,FAIL,1,FAIL,True
2,S03,FAIL,1,FAIL,True
3,S04,FAIL,1,FAIL,True



DETECTED DIFFERENCES


,TEST_ID,CHARACTERISTIC,ERROR_TYPE,EXPECTED,ACTUAL
0,S02,WIDTH_CM,VALUE_DEFINITION_MISMATCH,60,90
1,S03,PANEL_COLOR,REQUIREMENT_TYPE_MISMATCH,PREFERRED,MUST
2,S04,WIFI,ADDED_REQUIREMENT,NaN,NaN



Test accuracy: 100.0%


In [74]:
for name in [
    "expanded_ground_truth",
    "final_ground_truth",
    "final_customer_requests"
]:
    obj = globals().get(name)

    if obj is None:
        print(name, "-> NOT FOUND")
    else:
        print(
            name,
            "-> rows:", len(obj),
            "| unique requests:",
            obj["REQUEST_ID"].nunique()
        )

expanded_ground_truth -> rows: 3988 | unique requests: 1000
final_ground_truth -> rows: 4028 | unique requests: 1010
final_customer_requests -> rows: 1010 | unique requests: 1010


In [75]:

# Step 74 - Prepare batch LLM rewriting inputs

import json
from pathlib import Path

# Use the validated final dataset
ground_truth_df = final_ground_truth.copy()
customer_text_df = final_customer_requests.copy()

# Validate request IDs
assert ground_truth_df["REQUEST_ID"].nunique() == 1010
assert customer_text_df["REQUEST_ID"].nunique() == 1010

assert (
    set(ground_truth_df["REQUEST_ID"])
    == set(customer_text_df["REQUEST_ID"])
)

# Create structured LLM input records
llm_batch_records = []

for request_id, group in ground_truth_df.groupby(
    "REQUEST_ID", sort=True
):
    original_text = customer_text_df.loc[
        customer_text_df["REQUEST_ID"] == request_id,
        "REQUEST_TEXT"
    ].iloc[0]

    requirements = group[
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ].to_dict(orient="records")

    llm_batch_records.append({
        "REQUEST_ID": request_id,
        "ORIGINAL_TEXT": original_text,
        "REQUIREMENTS": requirements
    })

# Split into batches
BATCH_SIZE = 20

llm_batches = [
    llm_batch_records[i:i + BATCH_SIZE]
    for i in range(0, len(llm_batch_records), BATCH_SIZE)
]

# Save batch inputs
output_path = Path(
    "../data/interim/llm_rewriting_batches.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        llm_batches,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("LLM BATCH PREPARATION")
print("-" * 35)
print("Total requests:", len(llm_batch_records))
print("Batch size:", BATCH_SIZE)
print("Number of batches:", len(llm_batches))
print("Saved to:", output_path)

print("\nFirst batch:")
print("Requests:", len(llm_batches[0]))
print(
    "IDs:",
    llm_batches[0][0]["REQUEST_ID"],
    "-",
    llm_batches[0][-1]["REQUEST_ID"]
)


LLM BATCH PREPARATION
-----------------------------------
Total requests: 1010
Batch size: 20
Number of batches: 51
Saved to: ../data/interim/llm_rewriting_batches.json

First batch:
Requests: 20
IDs: R0001 - R0020


In [76]:
import os
import importlib.util

print("OpenAI SDK installed:",
      importlib.util.find_spec("openai") is not None)

print("API key configured:",
      bool(os.getenv("OPENAI_API_KEY")))

OpenAI SDK installed: True
API key configured: True


In [77]:
# Step 75 - Inspect CONTAINS and OVEN_FUNCTIONS implementation

from pathlib import Path

rule_file = Path("../src/rule_engine.py")

if not rule_file.exists():
    print("RuleEngine file not found:", rule_file.resolve())
else:
    source_lines = rule_file.read_text(
        encoding="utf-8"
    ).splitlines()

    keywords = ["CONTAINS", "OVEN_FUNCTIONS"]

    matching_lines = [
        i for i, line in enumerate(source_lines)
        if any(keyword in line.upper() for keyword in keywords)
    ]

    print("Total source lines:", len(source_lines))
    print("Matching lines:", len(matching_lines))

    shown = set()

    for index in matching_lines:
        start = max(0, index - 5)
        end = min(len(source_lines), index + 8)

        print(f"\n--- Source lines {start + 1}-{end} ---")

        for i in range(start, end):
            if i not in shown:
                print(f"{i + 1:4}: {source_lines[i]}")
                shown.add(i)

Total source lines: 203
Matching lines: 0


In [78]:
# Step 76 - Inspect RuleEngine structure

from pathlib import Path
import ast

rule_file = Path("../src/rule_engine.py")
source = rule_file.read_text(encoding="utf-8")

tree = ast.parse(source)

print("=== CLASSES AND FUNCTIONS ===")

for node in tree.body:
    if isinstance(node, ast.ClassDef):
        print(f"\nCLASS: {node.name}")

        for item in node.body:
            if isinstance(
                item,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            ):
                print(
                    f"  METHOD: {item.name} "
                    f"(line {item.lineno})"
                )

    elif isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    ):
        print(
            f"FUNCTION: {node.name} "
            f"(line {node.lineno})"
        )

print("\n=== OPERATOR-RELATED LINES ===")

keywords = [
    "operator",
    "condition",
    "evaluate",
    "constraint",
    "rule",
    "isin",
    "subset"
]

for line_no, line in enumerate(source.splitlines(), 1):
    if any(word in line.lower() for word in keywords):
        print(f"{line_no:4}: {line.strip()}")

=== CLASSES AND FUNCTIONS ===

CLASS: RuleEngine
  METHOD: __init__ (line 9)
  METHOD: conditions_match (line 52)
  METHOD: get_options (line 60)
  METHOD: violates_inc (line 122)
  METHOD: choose_value (line 135)
  METHOD: validate_product (line 177)

=== OPERATOR-RELATED LINES ===
   6: class RuleEngine:
   9: def __init__(self, characteristic_values, configuration_rules):
  19: self.rules = []
  20: self.inc_rules = []
  22: # Compile Excel rules only once
  23: for rule_id, group in configuration_rules.groupby("RuleId"):
  25: conditions = defaultdict(set)
  29: if row.Operator == "Rule":
  30: conditions[row.CHARACTERISTIC].add(row.VALUE)
  32: elif row.Operator == "Action":
  44: self.inc_rules.append((rule_id, combination))
  47: self.rules.append(
  48: (rule_id, dict(conditions), actions)
  52: def conditions_match(conditions, product):
  57: for char, allowed_values in conditions.items()
  61: """Evaluate Auto Select, Enable and Disable rules."""
  72: for rule_id, conditions

In [79]:
# Step 77 - Inspect rule matching and product representation

from pathlib import Path
import pandas as pd

source_lines = Path("../src/rule_engine.py").read_text(
    encoding="utf-8"
).splitlines()

print("=== RULE MATCHING LOGIC (LINES 23-60) ===")

for i in range(22, min(60, len(source_lines))):
    print(f"{i + 1:3}: {source_lines[i]}")

print("\n=== PRODUCT VALIDATION (LINES 177-203) ===")

for i in range(176, min(203, len(source_lines))):
    print(f"{i + 1:3}: {source_lines[i]}")

print("\n=== OVEN_FUNCTIONS IN PRODUCT_MASTER ===")

if "OVEN_FUNCTIONS" in product_master.columns:
    print("Column exists: YES")
    print("Data type:", product_master["OVEN_FUNCTIONS"].dtype)
    print("\nSample values:")
    print(product_master["OVEN_FUNCTIONS"].dropna().head(10).to_string(index=False))
else:
    print("Column exists: NO")
    print("Available columns:")
    print(product_master.columns.tolist())

=== RULE MATCHING LOGIC (LINES 23-60) ===
 23:         for rule_id, group in configuration_rules.groupby("RuleId"):
 24: 
 25:             conditions = defaultdict(set)
 26:             actions = []
 27: 
 28:             for row in group.itertuples(index=False):
 29:                 if row.Operator == "Rule":
 30:                     conditions[row.CHARACTERISTIC].add(row.VALUE)
 31: 
 32:                 elif row.Operator == "Action":
 33:                     actions.append(
 34:                         (row.Type, row.CHARACTERISTIC, row.VALUE)
 35:                     )
 36: 
 37:             inc_rows = group[group["Type"] == "INC"]
 38: 
 39:             if not inc_rows.empty:
 40:                 combination = {
 41:                     row.CHARACTERISTIC: row.VALUE
 42:                     for row in inc_rows.itertuples(index=False)
 43:                 }
 44:                 self.inc_rules.append((rule_id, combination))
 45: 
 46:             if actions:
 47:                 sel

In [80]:
# Step 78 - Inspect OVEN_FUNCTIONS data structure

col = "OVEN_FUNCTIONS"

if col in product_master.columns:
    values = product_master[col].dropna()

    print("Products:", len(product_master))
    print("Non-null values:", len(values))
    print("Unique values:", values.nunique())
    print("Data type:", values.dtype)

    print("\nFirst 15 product values:")
    for value in values.head(15):
        print(repr(value))

    print("\nMost frequent values:")
    print(values.astype(str).value_counts().head(10))
else:
    print("OVEN_FUNCTIONS not found in PRODUCT_MASTER")

Products: 500
Non-null values: 500
Unique values: 12
Data type: object

First 15 product values:
'V009'
'V011'
'V011'
'V006'
'V007'
'V009'
'V005'
'V010'
'V006'
'V003'
'V002'
'V010'
'V002'
'V005'
'V005'

Most frequent values:
OVEN_FUNCTIONS
V005    63
V006    58
V007    48
V009    46
V004    46
V003    41
V010    39
V008    38
V002    33
V012    33
Name: count, dtype: int64


In [81]:
# Step 79 - Load oven function sets

import pandas as pd

master_path = "../data/raw/oven_configurator_master_v1.0.xlsx"

function_df = pd.read_excel(
    master_path,
    sheet_name="OVEN_FUNCTION_SET",
    header=None
)

oven_function_map = {}

for _, row in function_df.iloc[1:].iterrows():
    code = row.iloc[1]

    if pd.isna(code):
        continue

    functions = [
        str(value).strip()
        for value in row.iloc[4:]
        if pd.notna(value) and str(value).strip()
    ]

    oven_function_map[str(code).strip()] = functions

print("Function sets:", len(oven_function_map))
print("V010 functions:", oven_function_map.get("V010"))

Function sets: 12
V010 functions: ['OvenLight', 'Defrost', 'Fan+Ring', 'Top+Bottom', 'Top+Bottom+Fan', 'Pizza', 'Grill', 'DoubleGrill', 'DoubleGrill+Fan', 'HydroClean', 'Steam']


In [82]:
# Step 80 - Test CONTAINS matching

def matches_requirement(product, characteristic, value, operator):

    product_value = product.get(characteristic)

    if operator == "EQUAL":
        return str(product_value) == str(value)

    if operator == "CONTAINS":

        if characteristic == "OVEN_FUNCTIONS":
            functions = oven_function_map.get(
                str(product_value), []
            )
            return str(value) in functions

        raise ValueError(
            f"CONTAINS not implemented for {characteristic}"
        )

    raise ValueError(f"Unknown operator: {operator}")


# Test with a real product
product = product_master[
    product_master["OVEN_FUNCTIONS"] == "V010"
].iloc[0].to_dict()

print("Product function set:", product["OVEN_FUNCTIONS"])

print(
    "CONTAINS Pizza:",
    matches_requirement(
        product, "OVEN_FUNCTIONS", "Pizza", "CONTAINS"
    )
)

print(
    "CONTAINS Grill:",
    matches_requirement(
        product, "OVEN_FUNCTIONS", "Grill", "CONTAINS"
    )
)

print(
    "EQUAL V010:",
    matches_requirement(
        product, "OVEN_FUNCTIONS", "V010", "EQUAL"
    )
)

Product function set: V010
CONTAINS Pizza: True
CONTAINS Grill: True
EQUAL V010: True


In [83]:
# Step 81 - Evaluate multiple customer requirements

def evaluate_product(product, requirements):

    must_results = []
    preferred_results = []

    for req in requirements:
        matched = matches_requirement(
            product,
            req["CHARACTERISTIC"],
            req["VALUE"],
            req["OPERATOR"]
        )

        if req["REQUIREMENT_TYPE"] == "MUST":
            must_results.append(matched)

        elif req["REQUIREMENT_TYPE"] == "PREFERRED":
            preferred_results.append(matched)

        else:
            raise ValueError(
                f"Unknown requirement type: {req['REQUIREMENT_TYPE']}"
            )

    eligible = all(must_results)

    preferred_score = (
        sum(preferred_results) / len(preferred_results)
        if preferred_results else 0.0
    )

    return {
        "ELIGIBLE": eligible,
        "MUST_MATCHES": sum(must_results),
        "MUST_TOTAL": len(must_results),
        "PREFERRED_MATCHES": sum(preferred_results),
        "PREFERRED_TOTAL": len(preferred_results),
        "PREFERRED_SCORE": preferred_score
    }


# Test with the V010 product from Step 80

test_requirements = [
    {
        "CHARACTERISTIC": "OVEN_FUNCTIONS",
        "VALUE": "Pizza",
        "OPERATOR": "CONTAINS",
        "REQUIREMENT_TYPE": "MUST"
    },
    {
        "CHARACTERISTIC": "OVEN_FUNCTIONS",
        "VALUE": "Grill",
        "OPERATOR": "CONTAINS",
        "REQUIREMENT_TYPE": "MUST"
    },
    {
        "CHARACTERISTIC": "OVEN_FUNCTIONS",
        "VALUE": "Defrost",
        "OPERATOR": "CONTAINS",
        "REQUIREMENT_TYPE": "PREFERRED"
    }
]

result = evaluate_product(product, test_requirements)

print(result)

{'ELIGIBLE': True, 'MUST_MATCHES': 2, 'MUST_TOTAL': 2, 'PREFERRED_MATCHES': 1, 'PREFERRED_TOTAL': 1, 'PREFERRED_SCORE': 1.0}


In [84]:
# Step 82 - Analyze oven function coverage

from collections import Counter
import pandas as pd

function_product_counts = Counter()
function_set_counts = Counter()

for function_set, functions in oven_function_map.items():
    unique_functions = set(functions)

    product_count = (
        product_master["OVEN_FUNCTIONS"] == function_set
    ).sum()

    for function in unique_functions:
        function_set_counts[function] += 1
        function_product_counts[function] += product_count

function_coverage = pd.DataFrame([
    {
        "FUNCTION": function,
        "SET_COUNT": function_set_counts[function],
        "PRODUCT_COUNT": function_product_counts[function],
        "PRODUCT_COVERAGE_PCT": round(
            100 * function_product_counts[function]
            / len(product_master), 2
        )
    }
    for function in function_set_counts
]).sort_values(
    ["PRODUCT_COUNT", "FUNCTION"]
).reset_index(drop=True)

print("Unique oven functions:", len(function_coverage))
print("Functions with no matching products:",
      (function_coverage["PRODUCT_COUNT"] == 0).sum())

display(function_coverage)

Unique oven functions: 21
Functions with no matching products: 0


,FUNCTION,SET_COUNT,PRODUCT_COUNT,PRODUCT_COVERAGE_PCT
0,Steam,1,39,7.8
1,Top,2,64,12.8
2,Grill+Fan,2,66,13.2
3,3D,2,79,15.8
4,DoubleGrill+Fan+Turnspit,2,94,18.8
5,DoubleGrill+Turnspit,2,94,18.8
6,Grill+Turnspit,2,94,18.8
7,Pyrolysis,2,106,21.2
8,TURBO,3,111,22.2
9,Bottom+Fan,3,150,30.0


In [85]:
# Step 83 - Build valid function pairs from product portfolio

from itertools import combinations
from collections import Counter
import pandas as pd

pair_counts = Counter()

for _, product_row in product_master.iterrows():

    function_set = str(product_row["OVEN_FUNCTIONS"])

    functions = sorted(set(
        oven_function_map.get(function_set, [])
    ))

    for pair in combinations(functions, 2):
        pair_counts[pair] += 1

function_pairs = pd.DataFrame([
    {
        "FUNCTION_1": pair[0],
        "FUNCTION_2": pair[1],
        "MATCHING_PRODUCTS": count
    }
    for pair, count in pair_counts.items()
]).sort_values(
    ["MATCHING_PRODUCTS", "FUNCTION_1", "FUNCTION_2"]
).reset_index(drop=True)

print("Unique valid function pairs:", len(function_pairs))
print("Pairs with at least 10 matching products:",
      (function_pairs["MATCHING_PRODUCTS"] >= 10).sum())

print("\nRare valid combinations:")
display(function_pairs.head(10))

print("\nCommon valid combinations:")
display(function_pairs.tail(10))

Unique valid function pairs: 137
Pairs with at least 10 matching products: 137

Rare valid combinations:


,FUNCTION_1,FUNCTION_2,MATCHING_PRODUCTS
0,Bottom,OvenLight,32
1,OvenLight,TURBO,32
2,3D,Grill,33
3,3D,Grill+Fan,33
4,Bottom,Grill+Fan,33
5,DoubleGrill,Grill+Fan,33
6,DoubleGrill+Fan,Grill+Fan,33
7,Grill+Fan,Pizza,33
8,Grill+Fan,TURBO,33
9,Defrost,Steam,39



Common valid combinations:


,FUNCTION_1,FUNCTION_2,MATCHING_PRODUCTS
127,Defrost,DoubleGrill,309
128,Defrost,DoubleGrill+Fan,309
129,DoubleGrill,DoubleGrill+Fan,309
130,DoubleGrill,Top+Bottom,309
131,DoubleGrill,Top+Bottom+Fan,309
132,DoubleGrill+Fan,Top+Bottom,309
133,DoubleGrill+Fan,Top+Bottom+Fan,309
134,Defrost,Top+Bottom,436
135,Defrost,Top+Bottom+Fan,436
136,Top+Bottom,Top+Bottom+Fan,477


In [86]:
# Step 84 - Test multi-value CONTAINS ground truth structure

test_request = pd.DataFrame([
    {
        "REQUEST_ID": "TEST_CONTAINS_001",
        "CHARACTERISTIC": "OVEN_FUNCTIONS",
        "VALUE": "Pizza",
        "OPERATOR": "CONTAINS",
        "REQUIREMENT_TYPE": "MUST"
    },
    {
        "REQUEST_ID": "TEST_CONTAINS_001",
        "CHARACTERISTIC": "OVEN_FUNCTIONS",
        "VALUE": "Grill",
        "OPERATOR": "CONTAINS",
        "REQUIREMENT_TYPE": "MUST"
    },
    {
        "REQUEST_ID": "TEST_CONTAINS_001",
        "CHARACTERISTIC": "OVEN_FUNCTIONS",
        "VALUE": "Defrost",
        "OPERATOR": "CONTAINS",
        "REQUIREMENT_TYPE": "PREFERRED"
    }
])

print("Requirements:", len(test_request))
print("Unique characteristics:",
      test_request["CHARACTERISTIC"].nunique())

print("Duplicate exact requirements:",
      test_request.duplicated(
          subset=[
              "REQUEST_ID",
              "CHARACTERISTIC",
              "VALUE",
              "OPERATOR",
              "REQUIREMENT_TYPE"
          ]
      ).sum())

display(test_request)

Requirements: 3
Unique characteristics: 1
Duplicate exact requirements: 0


,REQUEST_ID,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,TEST_CONTAINS_001,OVEN_FUNCTIONS,Pizza,CONTAINS,MUST
1,TEST_CONTAINS_001,OVEN_FUNCTIONS,Grill,CONTAINS,MUST
2,TEST_CONTAINS_001,OVEN_FUNCTIONS,Defrost,CONTAINS,PREFERRED


In [87]:
# Step 85 - Generate pilot requirements with EQUAL and CONTAINS

import random
import pandas as pd

rng = random.Random(42)

pilot_records = []
pilot_size = 100

equal_characteristics = [
    col for col in product_master.columns
    if col != "OVEN_FUNCTIONS"
    and col in set(final_ground_truth["CHARACTERISTIC"])
]

for request_number in range(1011, 1011 + pilot_size):

    request_id = f"R{request_number:04d}"

    product_row = product_master.iloc[
        rng.randrange(len(product_master))
    ]

    function_set = str(product_row["OVEN_FUNCTIONS"])

    available_functions = sorted(set(
        oven_function_map[function_set]
    ))

    selected_functions = rng.sample(
        available_functions,
        k=min(
            rng.randint(1, 3),
            len(available_functions)
        )
    )

    selected_equal = rng.sample(
        equal_characteristics,
        k=min(2, len(equal_characteristics))
    )

    requirements = []

    for function in selected_functions:
        requirements.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": "OVEN_FUNCTIONS",
            "VALUE": function,
            "VALUE_DEFINITION": function,
            "OPERATOR": "CONTAINS",
            "REQUIREMENT_TYPE": rng.choice(
                ["MUST", "MUST", "PREFERRED"]
            )
        })

    for characteristic in selected_equal:
        value = product_row[characteristic]

        requirements.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": value,
            "VALUE_DEFINITION": value,
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": rng.choice(
                ["MUST", "MUST", "PREFERRED"]
            )
        })

    pilot_records.extend(requirements)

contains_pilot_gt = pd.DataFrame(pilot_records)

print("Pilot requests:",
      contains_pilot_gt["REQUEST_ID"].nunique())

print("Ground Truth rows:",
      len(contains_pilot_gt))

print("\nOperator distribution:")
print(contains_pilot_gt["OPERATOR"].value_counts())

print("\nRequirement type distribution:")
print(contains_pilot_gt["REQUIREMENT_TYPE"].value_counts())

display(contains_pilot_gt.head(15))

Pilot requests: 100
Ground Truth rows: 407

Operator distribution:
OPERATOR
CONTAINS    207
EQUAL       200
Name: count, dtype: int64

Requirement type distribution:
REQUIREMENT_TYPE
MUST         271
PREFERRED    136
Name: count, dtype: int64


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R1011,OVEN_FUNCTIONS,Bottom+Fan,Bottom+Fan,CONTAINS,MUST
1,R1011,KNOB_STRUCTURE,V001,V001,EQUAL,MUST
2,R1011,KNOB_COLOR,V003,V003,EQUAL,PREFERRED
3,R1012,OVEN_FUNCTIONS,Defrost,Defrost,CONTAINS,MUST
4,R1012,OVEN_FUNCTIONS,Grill+Turnspit,Grill+Turnspit,CONTAINS,MUST
5,R1012,OVEN_FUNCTIONS,DoubleGrill+Turnspit,DoubleGrill+Turnspit,CONTAINS,MUST
6,R1012,OVEN_VOLUME_L,V003,V003,EQUAL,PREFERRED
7,R1012,WIDTH_CM,V003,V003,EQUAL,PREFERRED
8,R1013,OVEN_FUNCTIONS,DoubleGrill+Fan,DoubleGrill+Fan,CONTAINS,MUST
9,R1013,OVEN_FUNCTIONS,OvenLight,OvenLight,CONTAINS,MUST


In [88]:
# Step 86 - Validate pilot requests against product portfolio

validation_results = []

for request_id, group in contains_pilot_gt.groupby("REQUEST_ID"):

    requirements = group[
        [
            "CHARACTERISTIC",
            "VALUE",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ].to_dict("records")

    eligible_count = 0

    for _, product_row in product_master.iterrows():

        result = evaluate_product(
            product_row.to_dict(),
            requirements
        )

        if result["ELIGIBLE"]:
            eligible_count += 1

    validation_results.append({
        "REQUEST_ID": request_id,
        "MATCHING_PRODUCTS": eligible_count,
        "HAS_MATCH": eligible_count > 0
    })

pilot_validation = pd.DataFrame(validation_results)

print("Total requests:", len(pilot_validation))

print(
    "Requests with matching products:",
    pilot_validation["HAS_MATCH"].sum()
)

print(
    "Requests without matching products:",
    (~pilot_validation["HAS_MATCH"]).sum()
)

print(
    "Minimum matching products:",
    pilot_validation["MATCHING_PRODUCTS"].min()
)

print(
    "Maximum matching products:",
    pilot_validation["MATCHING_PRODUCTS"].max()
)

display(pilot_validation.head(10))

Total requests: 100
Requests with matching products: 100
Requests without matching products: 0
Minimum matching products: 3
Maximum matching products: 477


,REQUEST_ID,MATCHING_PRODUCTS,HAS_MATCH
0,R1011,63,True
1,R1012,94,True
2,R1013,65,True
3,R1014,119,True
4,R1015,17,True
5,R1016,61,True
6,R1017,3,True
7,R1018,78,True
8,R1019,291,True
9,R1020,64,True


In [89]:
# Step 87 - Reuse existing value definitions

import pandas as pd

value_lookup = (
    final_ground_truth[
        ["CHARACTERISTIC", "VALUE", "VALUE_DEFINITION"]
    ]
    .dropna()
    .drop_duplicates()
)

# Check whether each characteristic-value pair
# has exactly one definition

definition_counts = (
    value_lookup
    .groupby(["CHARACTERISTIC", "VALUE"])
    ["VALUE_DEFINITION"]
    .nunique()
)

print(
    "Conflicting definitions:",
    (definition_counts > 1).sum()
)

# Build lookup dictionary

value_definition_map = {
    (row.CHARACTERISTIC, row.VALUE): row.VALUE_DEFINITION
    for row in value_lookup.itertuples(index=False)
}

# Update pilot definitions

contains_pilot_gt["VALUE_DEFINITION"] = (
    contains_pilot_gt.apply(
        lambda row: value_definition_map.get(
            (row["CHARACTERISTIC"], row["VALUE"]),
            row["VALUE_DEFINITION"]
        )
        if row["OPERATOR"] == "EQUAL"
        else row["VALUE_DEFINITION"],
        axis=1
    )
)

# Identify unresolved EQUAL definitions

unresolved = contains_pilot_gt[
    (contains_pilot_gt["OPERATOR"] == "EQUAL")
    & (
        contains_pilot_gt["VALUE_DEFINITION"].astype(str)
        == contains_pilot_gt["VALUE"].astype(str)
    )
]

print("Unresolved EQUAL rows:", len(unresolved))

display(
    contains_pilot_gt[
        contains_pilot_gt["OPERATOR"] == "EQUAL"
    ].head(10)
)

Conflicting definitions: 0
Unresolved EQUAL rows: 0


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
1,R1011,KNOB_STRUCTURE,V001,ROTARY,EQUAL,MUST
2,R1011,KNOB_COLOR,V003,SILVER,EQUAL,PREFERRED
6,R1012,OVEN_VOLUME_L,V003,75,EQUAL,PREFERRED
7,R1012,WIDTH_CM,V003,70,EQUAL,PREFERRED
11,R1013,WIRE_SHELF,V002,1,EQUAL,PREFERRED
12,R1013,BAKING_TRAY,V001,0,EQUAL,MUST
14,R1014,WIRE_SHELF,V002,1,EQUAL,MUST
15,R1014,TURNSPIT,V002,NO,EQUAL,MUST
18,R1015,PIZZA_TRAY,V002,NO,EQUAL,PREFERRED
19,R1015,KNOB_TYPE,V004,K4,EQUAL,MUST


In [90]:
# Step 88 - Generate 1,990 new customer requests

import random
import pandas as pd

rng = random.Random(42)

new_records = []
target_new_requests = 1990

equal_characteristics = sorted(
    set(final_ground_truth["CHARACTERISTIC"])
    & set(product_master.columns)
    - {"OVEN_FUNCTIONS"}
)

# Give rare functions a higher sampling probability
function_weights = {
    row.FUNCTION: 1 / max(row.PRODUCT_COUNT, 1)
    for row in function_coverage.itertuples(index=False)
}

for request_number in range(1011, 1011 + target_new_requests):

    request_id = f"R{request_number:04d}"

    product = product_master.iloc[
        rng.randrange(len(product_master))
    ]

    function_set = str(product["OVEN_FUNCTIONS"])

    available_functions = sorted(
        set(oven_function_map[function_set])
    )

    function_count = min(
        rng.randint(1, 3),
        len(available_functions)
    )

    # Weighted sampling without replacement
    selected_functions = []
    candidates = available_functions.copy()

    for _ in range(function_count):
        weights = [
            function_weights[f] for f in candidates
        ]

        selected = rng.choices(
            candidates,
            weights=weights,
            k=1
        )[0]

        selected_functions.append(selected)
        candidates.remove(selected)

    equal_count = rng.randint(1, 3)

    selected_equal = rng.sample(
        equal_characteristics,
        k=min(equal_count, len(equal_characteristics))
    )

    requirements = []

    for function in selected_functions:
        requirements.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": "OVEN_FUNCTIONS",
            "VALUE": function,
            "VALUE_DEFINITION": function,
            "OPERATOR": "CONTAINS",
            "REQUIREMENT_TYPE": rng.choice(
                ["MUST", "MUST", "PREFERRED"]
            )
        })

    for characteristic in selected_equal:

        value = product[characteristic]

        requirements.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": value,
            "VALUE_DEFINITION": value_definition_map.get(
                (characteristic, value),
                str(value)
            ),
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": rng.choice(
                ["MUST", "MUST", "PREFERRED"]
            )
        })

    # Guarantee at least one MUST requirement
    if not any(
        req["REQUIREMENT_TYPE"] == "MUST"
        for req in requirements
    ):
        requirements[0]["REQUIREMENT_TYPE"] = "MUST"

    new_records.extend(requirements)

new_ground_truth = pd.DataFrame(new_records)

print("New requests:",
      new_ground_truth["REQUEST_ID"].nunique())

print("New Ground Truth rows:",
      len(new_ground_truth))

print("\nOperator distribution:")
print(new_ground_truth["OPERATOR"].value_counts())

print("\nRequirement type distribution:")
print(new_ground_truth["REQUIREMENT_TYPE"].value_counts())

print("\nUnique CONTAINS functions:",
      new_ground_truth.loc[
          new_ground_truth["OPERATOR"] == "CONTAINS",
          "VALUE"
      ].nunique())

display(new_ground_truth.head(10))

New requests: 1990
New Ground Truth rows: 7993

Operator distribution:
OPERATOR
CONTAINS    4023
EQUAL       3970
Name: count, dtype: int64

Requirement type distribution:
REQUIREMENT_TYPE
MUST         5330
PREFERRED    2663
Name: count, dtype: int64

Unique CONTAINS functions: 21


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R1011,OVEN_FUNCTIONS,Bottom+Fan,Bottom+Fan,CONTAINS,PREFERRED
1,R1011,KNOB_TYPE,V002,K2,EQUAL,MUST
2,R1011,HANDLE_TYPE,V002,H2,EQUAL,PREFERRED
3,R1012,OVEN_FUNCTIONS,DoubleGrill+Fan+Turnspit,DoubleGrill+Fan+Turnspit,CONTAINS,PREFERRED
4,R1012,OVEN_FUNCTIONS,Fan+Ring,Fan+Ring,CONTAINS,PREFERRED
5,R1012,OVEN_FUNCTIONS,Defrost,Defrost,CONTAINS,MUST
6,R1012,KNOB_TYPE,V005,K5,EQUAL,PREFERRED
7,R1013,OVEN_FUNCTIONS,Pizza,Pizza,CONTAINS,PREFERRED
8,R1013,OVEN_FUNCTIONS,DoubleGrill+Fan,DoubleGrill+Fan,CONTAINS,MUST
9,R1013,OVEN_FUNCTIONS,3D,3D,CONTAINS,MUST


In [91]:
# Step 89 - Combine and validate 3,000 customer requests

import pandas as pd

all_ground_truth = pd.concat(
    [final_ground_truth, new_ground_truth],
    ignore_index=True
)

# Duplicate requirement check
duplicate_requirements = all_ground_truth.duplicated(
    subset=[
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "OPERATOR",
        "REQUIREMENT_TYPE"
    ]
).sum()

# Missing values
missing_values = all_ground_truth[
    [
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "VALUE_DEFINITION",
        "OPERATOR",
        "REQUIREMENT_TYPE"
    ]
].isna().sum()

# Requirement counts per request
requirements_per_request = (
    all_ground_truth.groupby("REQUEST_ID").size()
)

# MUST coverage
must_per_request = (
    all_ground_truth[
        all_ground_truth["REQUIREMENT_TYPE"] == "MUST"
    ]
    .groupby("REQUEST_ID")
    .size()
)

print("=== FINAL DATASET VALIDATION ===")

print("Total requests:",
      all_ground_truth["REQUEST_ID"].nunique())

print("Total requirements:",
      len(all_ground_truth))

print("Duplicate requirements:",
      duplicate_requirements)

print("Requests without MUST:",
      all_ground_truth["REQUEST_ID"].nunique()
      - len(must_per_request))

print("Minimum requirements:",
      requirements_per_request.min())

print("Maximum requirements:",
      requirements_per_request.max())

print("\nMissing values:")
print(missing_values)

print("\nOperator distribution:")
print(all_ground_truth["OPERATOR"].value_counts())

print("\nRequirement type distribution:")
print(all_ground_truth["REQUIREMENT_TYPE"].value_counts())

assert all_ground_truth["REQUEST_ID"].nunique() == 3000
assert len(all_ground_truth) == 12021
assert duplicate_requirements == 0
assert missing_values.sum() == 0
assert requirements_per_request.min() >= 1
assert all_ground_truth["REQUIREMENT_TYPE"].isin(
    ["MUST", "PREFERRED"]
).all()

print("\nFINAL VALIDATION: PASSED")

=== FINAL DATASET VALIDATION ===
Total requests: 3000
Total requirements: 12021
Duplicate requirements: 0
Requests without MUST: 18
Minimum requirements: 2
Maximum requirements: 6

Missing values:
REQUEST_ID          0
CHARACTERISTIC      0
VALUE               0
VALUE_DEFINITION    0
OPERATOR            0
REQUIREMENT_TYPE    0
dtype: int64

Operator distribution:
OPERATOR
EQUAL       7998
CONTAINS    4023
Name: count, dtype: int64

Requirement type distribution:
REQUIREMENT_TYPE
MUST         8362
PREFERRED    3659
Name: count, dtype: int64

FINAL VALIDATION: PASSED


In [92]:
# Step 90 - Analyze requests without MUST requirements

all_request_ids = set(all_ground_truth["REQUEST_ID"])

must_request_ids = set(
    all_ground_truth.loc[
        all_ground_truth["REQUIREMENT_TYPE"] == "MUST",
        "REQUEST_ID"
    ]
)

requests_without_must = sorted(
    all_request_ids - must_request_ids
)

print("Requests without MUST:", len(requests_without_must))

print("\nRequest IDs:")
print(requests_without_must)

print("\nSource distribution:")

original_ids = set(final_ground_truth["REQUEST_ID"])
new_ids = set(new_ground_truth["REQUEST_ID"])

print(
    "Original 1,010:",
    len(set(requests_without_must) & original_ids)
)

print(
    "New 1,990:",
    len(set(requests_without_must) & new_ids)
)

print("\nSample requirements:")

display(
    all_ground_truth[
        all_ground_truth["REQUEST_ID"].isin(
            requests_without_must[:5]
        )
    ]
)

Requests without MUST: 18

Request IDs:
['R0067', 'R0130', 'R0145', 'R0165', 'R0216', 'R0335', 'R0339', 'R0354', 'R0374', 'R0379', 'R0513', 'R0518', 'R0606', 'R0665', 'R0666', 'R0791', 'R0859', 'R0911']

Source distribution:
Original 1,010: 18
New 1,990: 0

Sample requirements:


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
259,R0067,CLEANING,V001,ETC,EQUAL,PREFERRED
260,R0067,PANEL_COLOR,V003,SILVER,EQUAL,PREFERRED
505,R0130,NUMBER_OF_FUNCTIONS,V004,10,EQUAL,PREFERRED
506,R0130,HANDLE_TYPE,V005,H5,EQUAL,PREFERRED
507,R0130,HANDLE_COLOR,V001,BLACK,EQUAL,PREFERRED
565,R0145,OVEN_TYPE,V003,COMBINATION,EQUAL,PREFERRED
566,R0145,HANDLE_TYPE,V001,H1,EQUAL,PREFERRED
649,R0165,TELESCOPIC_RAILS,V001,0,EQUAL,PREFERRED
650,R0165,WIFI,V002,NO,EQUAL,PREFERRED
842,R0216,WIDTH_CM,V001,45,EQUAL,PREFERRED


In [93]:
# Step 91 - Save validated Ground Truth dataset

from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "request_ground_truth_3000.csv"

all_ground_truth.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_file)
print("Requests:", all_ground_truth["REQUEST_ID"].nunique())
print("Requirements:", len(all_ground_truth))

Saved: ../data/processed/request_ground_truth_3000.csv
Requests: 3000
Requirements: 12021


In [94]:
# Step 93 - Test OpenAI API connection

import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from project root
env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)

api_key = os.getenv("OPENAI_API_KEY")

assert api_key, "OPENAI_API_KEY not found in .env"

print("API key loaded: YES")

# Initialize API client
client = OpenAI(api_key=api_key)

# Small test request
response = client.responses.create(
    model="gpt-5-mini",
    input="Reply with exactly: API connection successful",
    max_output_tokens=100
)

print("API response:", response.output_text)
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)

API key loaded: YES
API response: API connection successful
Input tokens: 13
Output tokens: 95


In [95]:
# Step 94 - Generate natural customer requests using OpenAI API

import json
import pandas as pd
from pathlib import Path

# Load validated Ground Truth
gt_path = Path("../data/processed/request_ground_truth_3000.csv")
ground_truth_3000 = pd.read_csv(gt_path, dtype=str)

# Select 10 requests for the API pilot
# Include both EQUAL and CONTAINS cases
contains_ids = (
    ground_truth_3000.loc[
        ground_truth_3000["OPERATOR"] == "CONTAINS",
        "REQUEST_ID"
    ]
    .drop_duplicates()
    .head(7)
    .tolist()
)

equal_only_ids = (
    ground_truth_3000.groupby("REQUEST_ID")["OPERATOR"]
    .apply(lambda x: (x == "EQUAL").all())
)

equal_only_ids = (
    equal_only_ids[equal_only_ids]
    .index[:3]
    .tolist()
)

pilot_ids = list(dict.fromkeys(equal_only_ids + contains_ids))

print("Pilot requests:", len(pilot_ids))
print("Request IDs:", pilot_ids)

pilot_results = []

total_input_tokens = 0
total_output_tokens = 0

system_prompt = """
You are generating realistic English customer requests
for a household oven product recommendation system.

Convert structured product requirements into a natural,
realistic customer request.

STRICT RULES:
1. Preserve every requirement.
2. Do not add product features or requirements.
3. MUST requirements must remain mandatory.
4. PREFERRED requirements must remain optional.
5. CONTAINS means the oven must include the specified
   function, not that it must have only that function.
6. EQUAL means the specified characteristic must match
   the requested value.
7. Do not mention technical characteristic names or codes.
8. Use varied sentence structures and natural vocabulary.
9. Write as a real customer, not as a product specification.
10. Return only the customer request text.
"""

for request_id in pilot_ids:

    requirements = (
        ground_truth_3000.loc[
            ground_truth_3000["REQUEST_ID"] == request_id,
            [
                "CHARACTERISTIC",
                "VALUE_DEFINITION",
                "OPERATOR",
                "REQUIREMENT_TYPE"
            ]
        ]
        .to_dict(orient="records")
    )

    user_prompt = (
        "Write one natural English customer request "
        "using these exact requirements:\n\n"
        + json.dumps(requirements, ensure_ascii=False, indent=2)
    )

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=system_prompt,
        input=user_prompt,
        max_output_tokens=500
    )

    generated_text = response.output_text.strip()

    total_input_tokens += response.usage.input_tokens
    total_output_tokens += response.usage.output_tokens

    pilot_results.append({
        "REQUEST_ID": request_id,
        "LLM_TEXT": generated_text,
        "INPUT_TOKENS": response.usage.input_tokens,
        "OUTPUT_TOKENS": response.usage.output_tokens
    })

    print(f"\n{request_id}")
    print(generated_text)

pilot_df = pd.DataFrame(pilot_results)

output_path = Path("../data/interim/llm_pilot_10.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

pilot_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n=== PILOT SUMMARY ===")
print("Generated requests:", len(pilot_df))
print("Input tokens:", total_input_tokens)
print("Output tokens:", total_output_tokens)
print("Saved:", output_path)

Pilot requests: 10
Request IDs: ['R0001', 'R0002', 'R0003', 'R1011', 'R1012', 'R1013', 'R1014', 'R1015', 'R1016', 'R1017']

R0001
I need a steam oven that is exactly 45 cm wide.

R0002


R0003


R1011


R1012


R1013


R1014


R1015


R1016


R1017


=== PILOT SUMMARY ===
Generated requests: 10
Input tokens: 3447
Output tokens: 4408
Saved: ../data/interim/llm_pilot_10.csv


In [96]:
# Step 95 - Diagnose empty LLM responses

print("=== LAST API RESPONSE ===")

print("Response status:", response.status)
print("Output text length:", len(response.output_text))
print("Output items:", len(response.output))

print("\nToken usage:")
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)

if response.usage.output_tokens_details:
    print(
        "Reasoning tokens:",
        response.usage.output_tokens_details.reasoning_tokens
    )

print("\n=== PILOT TEXT VALIDATION ===")

pilot_df["TEXT_LENGTH"] = (
    pilot_df["LLM_TEXT"]
    .fillna("")
    .astype(str)
    .str.len()
)

print("Total requests:", len(pilot_df))
print("Empty texts:", (pilot_df["TEXT_LENGTH"] == 0).sum())

display(
    pilot_df[
        ["REQUEST_ID", "TEXT_LENGTH", "INPUT_TOKENS", "OUTPUT_TOKENS"]
    ]
)

=== LAST API RESPONSE ===
Response status: incomplete
Output text length: 0
Output items: 1

Token usage:
Input tokens: 311
Output tokens: 448
Reasoning tokens: 448

=== PILOT TEXT VALIDATION ===
Total requests: 10
Empty texts: 9


,REQUEST_ID,TEXT_LENGTH,INPUT_TOKENS,OUTPUT_TOKENS
0,R0001,47,263,376
1,R0002,0,310,448
2,R0003,0,445,448
3,R1011,0,312,448
4,R1012,0,369,448
5,R1013,0,452,448
6,R1014,0,360,448
7,R1015,0,313,448
8,R1016,0,312,448
9,R1017,0,311,448


In [97]:
# Step 96 - Test one request with reduced reasoning

import json

test_id = "R0001"

requirements = (
    ground_truth_3000.loc[
        ground_truth_3000["REQUEST_ID"] == test_id,
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ]
    .to_dict(orient="records")
)

user_prompt = (
    "Write one natural English customer request "
    "using these exact requirements:\n\n"
    + json.dumps(requirements, ensure_ascii=False)
)

test_response = client.responses.create(
    model="gpt-5-mini",
    instructions=system_prompt,
    input=user_prompt,
    reasoning={"effort": "minimal"},
    max_output_tokens=1000
)

print("Request:", test_id)
print("Status:", test_response.status)
print("Generated text:", test_response.output_text)

print("\nToken usage:")
print("Input:", test_response.usage.input_tokens)
print("Output:", test_response.usage.output_tokens)
print(
    "Reasoning:",
    test_response.usage.output_tokens_details.reasoning_tokens
)

assert test_response.output_text.strip(), \
    "No customer text generated"

Request: R0001
Status: completed
Generated text: I'm looking for a steam oven that's exactly 45 cm wide.

Token usage:
Input: 247
Output: 31
Reasoning: 0


In [98]:
# Step 97 - Complete the 10-request LLM pilot

import json
import pandas as pd
from pathlib import Path

pilot_results = [{
    "REQUEST_ID": "R0001",
    "LLM_TEXT": test_response.output_text.strip(),
    "INPUT_TOKENS": test_response.usage.input_tokens,
    "OUTPUT_TOKENS": test_response.usage.output_tokens
}]

for request_id in pilot_ids:

    if request_id == "R0001":
        continue

    requirements = (
        ground_truth_3000.loc[
            ground_truth_3000["REQUEST_ID"] == request_id,
            [
                "CHARACTERISTIC",
                "VALUE_DEFINITION",
                "OPERATOR",
                "REQUIREMENT_TYPE"
            ]
        ]
        .to_dict(orient="records")
    )

    user_prompt = (
        "Write one natural English customer request "
        "using these exact requirements:\n\n"
        + json.dumps(requirements, ensure_ascii=False)
    )

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=system_prompt,
        input=user_prompt,
        reasoning={"effort": "minimal"},
        max_output_tokens=1000
    )

    generated_text = response.output_text.strip()

    if response.status != "completed" or not generated_text:
        print(f"FAILED: {request_id} - {response.status}")
        continue

    pilot_results.append({
        "REQUEST_ID": request_id,
        "LLM_TEXT": generated_text,
        "INPUT_TOKENS": response.usage.input_tokens,
        "OUTPUT_TOKENS": response.usage.output_tokens
    })

    print(f"Completed: {request_id}")

pilot_df = pd.DataFrame(pilot_results)

output_path = Path("../data/interim/llm_pilot_10.csv")

pilot_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n=== PILOT SUMMARY ===")
print("Generated requests:", len(pilot_df))
print("Empty texts:", pilot_df["LLM_TEXT"].eq("").sum())
print("Input tokens:", pilot_df["INPUT_TOKENS"].sum())
print("Output tokens:", pilot_df["OUTPUT_TOKENS"].sum())

display(
    pilot_df[["REQUEST_ID", "LLM_TEXT"]]
)

Completed: R0002
Completed: R0003
Completed: R1011
Completed: R1012
Completed: R1013
Completed: R1014
Completed: R1015
Completed: R1016
Completed: R1017

=== PILOT SUMMARY ===
Generated requests: 10
Empty texts: 0
Input tokens: 3151
Output tokens: 564


,REQUEST_ID,LLM_TEXT
0,R0001,I'm looking for a steam oven that's exactly 45 cm wide.
1,R0002,"I'm looking for an oven with a white control panel, packaged with shrink wrap, and with an internal capacity of 50 liters."
2,R0003,"I'm looking for a built-in combination oven about 60 cm wide with two baking trays and a glass front panel, and it must have the K2-style knobs; I’d prefer the control panel to be silver if possible."
3,R1011,"I'm looking for an oven that definitely has the K2 style knobs, and if possible it should include a bottom-plus-fan cooking mode and preferably come with an H2 style handle."
4,R1012,"I'm looking for an oven that definitely has a defrost setting, and I'd prefer one that also offers a double grill with fan and a turnspit, or alternatively a fan with a ring heating element. If possible, I'd like it to come with K5-style knobs."
5,R1013,"I'm looking for an oven that definitely has both a double grill with fan and a 3D cooking mode, and it must offer exactly 10 functions in total. I also need the oven to come without any baking tray and to have black control knobs. If possible, I'd prefer it to also include a pizza setting."
6,R1014,"I'm looking for an oven that definitely has both a double grill function and a regular grill, and it must not have a rotisserie/turnspit feature. Ideally the oven's capacity would be 65 liters, but that's just a preference."
7,R1015,"I'm looking for an oven that definitely has both top and bottom heating elements, and if possible it should be energy class A. I'd also prefer one that doesn't come with a roasting tray."
8,R1016,"I'm looking for an oven that has a top heating function and also a bottom heating with fan option, and it must not have Wi‑Fi."
9,R1017,"I'm looking for an oven with exactly 5 functions and a glass control panel, and if possible I'd prefer it to include a top heating function."


In [99]:
# Step 115 - Verify saved LLM pilot results

from pathlib import Path
import pandas as pd

pilot_path = Path("../data/interim/llm_pilot_10.csv")

assert pilot_path.exists(), "Pilot CSV file not found"

pilot_saved = pd.read_csv(pilot_path)

print("Saved requests:", len(pilot_saved))
print("Columns:", pilot_saved.columns.tolist())
print("Unique request IDs:", pilot_saved["REQUEST_ID"].nunique())

if "LLM_TEXT" in pilot_saved.columns:
    print(
        "Empty texts:",
        pilot_saved["LLM_TEXT"].fillna("").str.strip().eq("").sum()
    )

display(pilot_saved.head(3))

Saved requests: 10
Columns: ['REQUEST_ID', 'LLM_TEXT', 'INPUT_TOKENS', 'OUTPUT_TOKENS']
Unique request IDs: 10
Empty texts: 0


,REQUEST_ID,LLM_TEXT,INPUT_TOKENS,OUTPUT_TOKENS
0,R0001,I'm looking for a steam oven that's exactly 45 cm wide.,247,31
1,R0002,"I'm looking for an oven with a white control panel, packaged with shrink wrap, and with an internal capacity of 50 liters.",286,44
2,R0003,"I'm looking for a built-in combination oven about 60 cm wide with two baking trays and a glass front panel, and it must have the K2-style knobs; I’d prefer the control panel to be silver if possible.",397,63


In [100]:
# Step 116 - Prepare resumable LLM generation

from pathlib import Path
import pandas as pd

# Input files
ground_truth_path = Path("../data/processed/request_ground_truth_3000.csv")
pilot_path = Path("../data/interim/llm_pilot_10.csv")

# Output file
output_path = Path("../data/interim/llm_customer_requests_3000.csv")

# Load Ground Truth
ground_truth = pd.read_csv(ground_truth_path)

# Load existing pilot
pilot = pd.read_csv(pilot_path)

# Validate Ground Truth
assert ground_truth["REQUEST_ID"].nunique() == 3000
assert len(ground_truth) == 12021

# Validate pilot
assert pilot["REQUEST_ID"].nunique() == 10
assert pilot["LLM_TEXT"].fillna("").str.strip().ne("").all()

# Save pilot as initial output
if not output_path.exists():
    pilot.to_csv(output_path, index=False, encoding="utf-8-sig")

# Load existing generated texts
generated = pd.read_csv(output_path)

completed_ids = set(
    generated.loc[
        generated["LLM_TEXT"].fillna("").str.strip().ne(""),
        "REQUEST_ID"
    ]
)

all_ids = set(ground_truth["REQUEST_ID"].unique())

remaining_ids = sorted(all_ids - completed_ids)

print("Total requests:", len(all_ids))
print("Already generated:", len(completed_ids))
print("Remaining:", len(remaining_ids))
print("Output file:", output_path)

assert completed_ids.issubset(all_ids)

print("\nPREPARATION: PASSED")

Total requests: 3000
Already generated: 10
Remaining: 2990
Output file: ../data/interim/llm_customer_requests_3000.csv

PREPARATION: PASSED


In [103]:
# Step 117 - Resumable API generation with checkpoints

import os
import time
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Configuration
MODEL = "gpt-5-mini"
BATCH_LIMIT = 3000
MAX_RETRIES = 3

ground_truth_path = Path("../data/processed/request_ground_truth_3000.csv")
output_path = Path("../data/interim/llm_customer_requests_3000.csv")
env_path = Path("../.env")

load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "OPENAI_API_KEY not found"

client = OpenAI(api_key=api_key, timeout=45.0, max_retries=0)

ground_truth = pd.read_csv(ground_truth_path)

# Load existing results
generated = pd.read_csv(output_path)

# Keep only valid, non-empty results
generated = generated[
    generated["LLM_TEXT"].fillna("").str.strip().ne("")
].copy()

generated = generated.drop_duplicates(
    subset=["REQUEST_ID"],
    keep="first"
)

completed_ids = set(generated["REQUEST_ID"])

all_ids = sorted(ground_truth["REQUEST_ID"].unique())
remaining_ids = [
    request_id
    for request_id in all_ids
    if request_id not in completed_ids
]

# Process only the next group
batch_ids = remaining_ids[:BATCH_LIMIT]

print("Total requests:", len(all_ids))
print("Already completed:", len(completed_ids))
print("Processing now:", len(batch_ids))
print("Remaining before batch:", len(remaining_ids))

total_input_tokens = 0
total_output_tokens = 0
new_completed = 0

for request_id in batch_ids:

    requirements = ground_truth[
        ground_truth["REQUEST_ID"] == request_id
    ]

    requirement_lines = []

    for row in requirements.itertuples(index=False):

        requirement_lines.append(
            f"- {row.CHARACTERISTIC}: "
            f"{row.VALUE_DEFINITION} "
            f"(operator={row.OPERATOR}, "
            f"priority={row.REQUIREMENT_TYPE})"
        )

    requirement_text = "\n".join(requirement_lines)

    prompt = f"""
Write ONE natural English customer request for an oven.

The customer requirements are:

{requirement_text}

Rules:
- Preserve every requirement.
- Do not add requirements.
- MUST requirements must remain mandatory.
- PREFERRED requirements must remain optional.
- EQUAL means the specified characteristic value.
- CONTAINS means the oven must include the specified function.
- Multiple CONTAINS requirements are cumulative, not alternatives.
- Do not expose technical characteristic names or operators.
- Use varied, realistic customer language.
- Avoid repetitive templates.
- Return only the customer request text.
"""

    success = False

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = client.responses.create(
                model=MODEL,
                input=prompt,
                reasoning={"effort": "minimal"},
                max_output_tokens=300
            )

            text = response.output_text.strip()

            if response.status != "completed" or not text:
                raise ValueError(
                    f"Invalid response: {response.status}"
                )

            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens

            new_row = pd.DataFrame([{
                "REQUEST_ID": request_id,
                "LLM_TEXT": text,
                "INPUT_TOKENS": input_tokens,
                "OUTPUT_TOKENS": output_tokens
            }])

            generated = pd.concat(
                [generated, new_row],
                ignore_index=True
            )

            # Atomic checkpoint
            temp_path = output_path.with_suffix(".tmp")

            generated.to_csv(
                temp_path,
                index=False,
                encoding="utf-8-sig"
            )

            os.replace(temp_path, output_path)

            total_input_tokens += input_tokens
            total_output_tokens += output_tokens

            new_completed += 1
            success = True

            print(f"Completed: {request_id}")

            break

        except Exception as error:

            print(
                f"Error: {request_id}, "
                f"attempt {attempt}/{MAX_RETRIES}: {error}"
            )

            if attempt < MAX_RETRIES:
                time.sleep(3 * attempt)

    if not success:
        print("Stopped after repeated errors:", request_id)
        break

print("\n=== BATCH SUMMARY ===")
print("New requests:", new_completed)
print("Total saved:", len(generated))
print("Remaining:", len(all_ids) - len(generated))
print("Input tokens:", total_input_tokens)
print("Output tokens:", total_output_tokens)
print("Saved:", output_path)

Total requests: 3000
Already completed: 130
Processing now: 2870
Remaining before batch: 2870
Completed: R0124
Completed: R0125
Completed: R0126
Completed: R0127
Completed: R0128
Completed: R0129
Completed: R0130
Completed: R0131
Completed: R0132
Completed: R0133
Completed: R0134
Completed: R0135
Completed: R0136
Completed: R0137
Completed: R0138
Completed: R0139
Completed: R0140
Completed: R0141
Completed: R0142
Completed: R0143
Completed: R0144
Completed: R0145
Completed: R0146
Completed: R0147
Completed: R0148
Completed: R0149
Completed: R0150
Completed: R0151
Completed: R0152
Completed: R0153
Completed: R0154
Completed: R0155
Completed: R0156
Completed: R0157
Completed: R0158
Completed: R0159
Completed: R0160
Completed: R0161
Completed: R0162
Completed: R0163
Completed: R0164
Completed: R0165
Completed: R0166
Completed: R0167
Completed: R0168
Completed: R0169
Completed: R0170
Completed: R0171
Completed: R0172
Completed: R0173
Completed: R0174
Completed: R0175
Completed: R0176
Compl